# Notebook 22 — Jockey and Trainer Identity

## Bounded question

Can raw `jockey` and `trainer` labels be mapped safely to stable source-internal and, where evidence permits, real-world participant identities without merging different people or splitting the same person unnecessarily?

## Purpose

This notebook investigates the source semantics, physical behaviour and identity risks of the runner-level `jockey` and `trainer` fields.

It is an identity study, not a participant-performance ranking exercise. Raw-label aggregates remain source-label analysis unless and until an identity mapping is supported by evidence.

The investigation will begin with source-wide profiling. It will not assume that punctuation removal, case folding, initial expansion, title removal, whitespace normalisation or fuzzy matching is safe.

The study will distinguish between:

* immutable raw source labels;
* exploratory candidate-match text;
* source-internal participant occurrences;
* governed equivalence decisions;
* governed split decisions;
* unresolved relationships;
* authority- or evidence-backed real-world identities.

No broad name normalisation or cross-role merge is authorised in advance.

## Source and grain

* **Source database:** `data/raw/form_2015-present/form_2015-present/raceform.db`
* **Source table:** `data`
* **Governed data-row predicate:** `rowid <> 1`
* **Fields under investigation:** `jockey`, `trainer`
* **Declared source type:** to be confirmed from SQLite
* **Provisional grain:** runner-level source assertions about race participants
* **Raw preservation:** required
* **Current analytical status:** identity semantics pending
* **Notebook 20 relationship:** existing blank supplementation remains governed and must not be silently replaced or reinterpreted here
* **Notebook 19 relationship:** the pending horse/pedigree authority gate is separate and must not be modified by this notebook

## Scope

The investigation will test:

* SQL null, empty-string, whitespace-only and placeholder behaviour;
* populated and distinct raw-label counts;
* punctuation, spacing, case, diacritic, bracket, title and suffix conventions;
* initials versus expanded names;
* spelling variants and probable source defects;
* same-name collision risk;
* jurisdiction and active-period boundaries;
* one person appearing under several labels;
* one label potentially referring to several people;
* jockey and trainer role distinctions;
* race-level and runner-level occurrence consistency;
* safe source-internal identity keys;
* occurrence splitting where required;
* evidence thresholds for equivalence, correction, splitting and preservation;
* cases that must remain unresolved.

## Closure obligations

If the evidence supports governed identity mappings, closure will require persisted and reloaded outputs, reusable implementation, focused tests, an independent source-wide validator, integration documentation, an explicit manual-verification decision, a reader-facing Minto report, lessons learned, audit and project-status updates, and recorded local validation.

The notebook will proceed one evidence-led stage at a time. Stable implementation and permanent reference outputs will be created only after the observed source behaviour justifies them.


## 1. Establish the source population and physical field behaviour

The first stage confirms the governed runner population, the declared SQLite storage for both fields, and the basic physical behaviour of each raw label.

This stage measures:

* governed runner rows;
* SQL nulls;
* empty strings;
* whitespace-only values;
* populated rows;
* distinct populated raw labels;
* leading and trailing whitespace;
* minimum and maximum populated lengths.

These checks precede any identity interpretation. No label will be trimmed, case-folded, de-punctuated, tokenised or fuzzy-matched in this stage.


In [1]:
from pathlib import Path
import sqlite3

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

DATA_ROW_PREDICATE = "rowid <> 1"
PARTICIPANT_FIELDS = ("jockey", "trainer")

assert SOURCE_DB_PATH.exists(), f"Source database not found: {SOURCE_DB_PATH}"

connection = sqlite3.connect(f"file:{SOURCE_DB_PATH}?mode=ro", uri=True)

schema = pd.read_sql_query("PRAGMA table_info(data)", connection)
participant_schema = schema.loc[schema["name"].isin(PARTICIPANT_FIELDS)].copy()

assert set(participant_schema["name"]) == set(PARTICIPANT_FIELDS), (
    "Expected jockey and trainer fields were not both present in the source schema."
)

profile_rows = []

for field_name in PARTICIPANT_FIELDS:
    field_profile = pd.read_sql_query(
        f'''
        SELECT
            '{field_name}' AS field_name,
            COUNT(*) AS governed_runner_rows,
            SUM({field_name} IS NULL) AS null_rows,
            SUM({field_name} = '') AS empty_string_rows,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> ''
                AND TRIM({field_name}) = ''
            ) AS whitespace_only_rows,
            SUM(
                {field_name} IS NOT NULL
                AND TRIM({field_name}) <> ''
            ) AS populated_rows,
            COUNT(
                DISTINCT CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN {field_name}
                END
            ) AS distinct_populated_labels,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> LTRIM({field_name})
            ) AS leading_whitespace_rows,
            SUM(
                {field_name} IS NOT NULL
                AND {field_name} <> RTRIM({field_name})
            ) AS trailing_whitespace_rows,
            MIN(
                CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN LENGTH({field_name})
                END
            ) AS minimum_populated_length,
            MAX(
                CASE
                    WHEN {field_name} IS NOT NULL
                         AND TRIM({field_name}) <> ''
                    THEN LENGTH({field_name})
                END
            ) AS maximum_populated_length
        FROM data
        WHERE {DATA_ROW_PREDICATE}
        ''',
        connection,
    )
    profile_rows.append(field_profile)

participant_physical_profile = pd.concat(profile_rows, ignore_index=True)

display(participant_schema)
display(participant_physical_profile)


,cid,name,type,notnull,dflt_value,pk
26,26,jockey,TEXT,0,None,0
27,27,trainer,TEXT,0,None,0


,field_name,governed_runner_rows,null_rows,empty_string_rows,whitespace_only_rows,populated_rows,distinct_populated_labels,leading_whitespace_rows,trailing_whitespace_rows,minimum_populated_length,maximum_populated_length
0,jockey,1851285,0,2,0,1851283,7917,0,0,5,33
1,trainer,1851285,0,9,0,1851276,10708,0,0,4,39


## 2. Profile the raw jockey-label population

The jockey investigation begins with exact source labels only.

This stage measures:

* how frequently each raw jockey label occurs;
* how many provisional races each label appears in;
* the first and last source dates attached to each label;
* the size of the low-frequency label tail;
* the most common exact labels.

These are source-label statistics, not confirmed career records for real people. No punctuation change, abbreviation expansion, fuzzy match, merge or identity correction is applied.

In [2]:
jockey_label_frequency = pd.read_sql_query(
    f"""
    SELECT
        jockey AS raw_jockey_label,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT date || '|' || course || '|' || off
        ) AS provisional_races,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND jockey IS NOT NULL
      AND jockey <> ''
    GROUP BY jockey
    ORDER BY
        runner_rows DESC,
        raw_jockey_label
    """,
    connection,
)

jockey_frequency_summary = pd.DataFrame(
    [
        {
            "distinct_raw_jockey_labels": len(jockey_label_frequency),
            "single_row_labels": int(
                jockey_label_frequency["runner_rows"].eq(1).sum()
            ),
            "labels_with_fewer_than_5_rows": int(
                jockey_label_frequency["runner_rows"].lt(5).sum()
            ),
            "labels_with_at_least_100_rows": int(
                jockey_label_frequency["runner_rows"].ge(100).sum()
            ),
            "labels_with_at_least_1000_rows": int(
                jockey_label_frequency["runner_rows"].ge(1000).sum()
            ),
            "median_runner_rows": jockey_label_frequency[
                "runner_rows"
            ].median(),
            "maximum_runner_rows": jockey_label_frequency[
                "runner_rows"
            ].max(),
            "maximum_provisional_races": jockey_label_frequency[
                "provisional_races"
            ].max(),
        }
    ]
)

display(jockey_frequency_summary)
display(jockey_label_frequency.head(25))

,distinct_raw_jockey_labels,single_row_labels,labels_with_fewer_than_5_rows,labels_with_at_least_100_rows,labels_with_at_least_1000_rows,median_runner_rows,maximum_runner_rows,maximum_provisional_races
0,7917,1331,2816,1676,457,10.0,14241,14241


,raw_jockey_label,runner_rows,provisional_races,first_date,last_date
0,Luke Morris,14241,14241,2015-01-03,2026-05-27
1,David Probert,10128,10128,2015-01-03,2026-05-26
2,Oisin Murphy,10017,10017,2015-01-04,2026-05-27
3,Tom Marquand,9918,9918,2015-01-03,2026-05-27
4,Maxime Guyon,9036,9036,2015-01-25,2026-05-24
5,Brian Hughes,8922,8922,2015-01-01,2026-03-14
6,Hollie Doyle,8594,8594,2015-01-02,2026-05-22
7,Silvestre De Sousa,8382,8382,2015-01-04,2026-05-27
8,Colin Keane,8350,8350,2015-01-09,2026-05-26
9,Mickael Barzalona,8024,8024,2015-01-08,2026-05-24


### Initial jockey-frequency findings

The source contains **7,917 distinct populated raw jockey labels**.

The population is highly uneven:

* **1,331** labels occur on only one runner row;
* **2,816** labels occur fewer than five times;
* **1,676** labels occur at least 100 times;
* **457** labels occur at least 1,000 times;
* the median raw label occurs **10** times;
* the most frequent label, `Luke Morris`, occurs **14,241** times.

The high-frequency end contains plausible full-name jockey labels with long source periods. However, frequency does not prove that each label represents exactly one real person, nor that different labels represent different people.

For the displayed high-frequency labels, `runner_rows` equals `provisional_races`. The source therefore shows no displayed case where the same exact jockey label is attached to more than one runner in the same provisional race.

This consistency is useful, but it is not yet an identity rule. The low-frequency tail may contain:

* genuinely infrequent jockeys;
* jurisdiction-specific participants;
* spelling defects;
* abbreviated or incomplete names;
* one-off source labels;
* aliases or presentation variants of more frequent labels.

The next step is to measure the full frequency distribution before inspecting particular candidate variants.

In [3]:
# Group exact raw jockey labels into frequency bands.
#
# This is descriptive profiling only:
# - raw labels are not altered;
# - no labels are merged;
# - the bands do not represent confirmed individual jockeys;
# - low-frequency labels are not assumed to be errors or aliases.
jockey_frequency_bands = pd.cut(
    jockey_label_frequency["runner_rows"],
    bins=[0, 1, 4, 9, 24, 49, 99, 249, 499, 999, float("inf")],
    labels=[
        "1",
        "2–4",
        "5–9",
        "10–24",
        "25–49",
        "50–99",
        "100–249",
        "250–499",
        "500–999",
        "1,000+",
    ],
    include_lowest=True,
)

# Summarise both:
# 1. how many distinct raw labels fall into each band; and
# 2. how many source runner rows those labels collectively represent.
#
# Keeping these measures separate shows whether a large tail of rare labels
# accounts for a material share of the source population.
jockey_frequency_distribution = (
    jockey_label_frequency.assign(frequency_band=jockey_frequency_bands)
    .groupby("frequency_band", observed=False)
    .agg(
        distinct_raw_labels=("raw_jockey_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .reset_index()
)

# Express each band's label count as a share of all exact raw jockey labels.
jockey_frequency_distribution["label_share_pct"] = (
    jockey_frequency_distribution["distinct_raw_labels"]
    / jockey_frequency_distribution["distinct_raw_labels"].sum()
    * 100
).round(2)

# Express each band's row count as a share of all populated jockey rows.
# This remains source-label coverage, not confirmed person-level activity.
jockey_frequency_distribution["runner_row_share_pct"] = (
    jockey_frequency_distribution["runner_rows"]
    / jockey_frequency_distribution["runner_rows"].sum()
    * 100
).round(2)

display(jockey_frequency_distribution)

,frequency_band,distinct_raw_labels,runner_rows,label_share_pct,runner_row_share_pct
0,1,1331,1331,16.81,0.07
1,2–4,1485,4115,18.76,0.22
2,5–9,1011,6813,12.77,0.37
3,10–24,1160,18046,14.65,0.97
4,25–49,670,23588,8.46,1.27
5,50–99,584,41085,7.38,2.22
6,100–249,582,93593,7.35,5.06
7,250–499,350,123964,4.42,6.70
8,500–999,287,204111,3.63,11.03
9,"1,000+",457,1334637,5.77,72.09


### Jockey-label concentration findings

The raw jockey-label population has a pronounced long tail.

Labels occurring fewer than ten times account for:

* **3,827** distinct raw labels;
* **48.34%** of all distinct raw jockey labels;
* only **12,259** runner rows;
* **0.66%** of populated jockey rows.

At the opposite end, the **457** labels occurring at least 1,000 times represent:

* only **5.77%** of distinct raw labels;
* **1,334,637** runner rows;
* **72.09%** of populated jockey rows.

The source is therefore dominated operationally by a relatively small set of high-frequency exact labels, while almost half of the label vocabulary lies in a very low-frequency tail.

This does not justify treating rare labels as errors. The tail may contain genuine occasional or international jockeys as well as spelling defects, abbreviated forms and aliases. It does, however, show that identity-resolution effort and source-row impact are distributed very differently.

The next step is to profile visible label structure without changing the raw text. This will test how often labels contain initials, punctuation, brackets, apostrophes, hyphens, digits or other conventions that could affect later candidate matching.

In [4]:
# Profile visible structural features of each exact raw jockey label.
#
# This cell does not normalise, edit or merge any label. The flags describe
# only what characters and token patterns are present in the immutable source
# string. They are candidate-investigation signals, not identity decisions.
jockey_label_structure = jockey_label_frequency.copy()

# Count whitespace-separated tokens as a simple description of label shape.
# Token counts do not imply that every token is a forename, surname or title.
jockey_label_structure["token_count"] = (
    jockey_label_structure["raw_jockey_label"]
    .str.split()
    .str.len()
)

# Record punctuation and character classes that may reflect legitimate naming
# conventions, abbreviations, source defects or jurisdiction-specific formats.
jockey_label_structure["contains_period"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"\.",
        regex=True,
    )
)
jockey_label_structure["contains_apostrophe"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"['’]",
        regex=True,
    )
)
jockey_label_structure["contains_hyphen"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        "-",
        regex=False,
    )
)
jockey_label_structure["contains_parenthesis"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"[()]",
        regex=True,
    )
)
jockey_label_structure["contains_bracket"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"[\[\]]",
        regex=True,
    )
)
jockey_label_structure["contains_digit"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"\d",
        regex=True,
    )
)
jockey_label_structure["contains_non_ascii"] = (
    jockey_label_structure["raw_jockey_label"].map(
        lambda value: not value.isascii()
    )
)

# Identify labels containing at least one single-letter token. This captures
# initial-style presentation such as "P J McDonald", but does not assert that
# differently expanded forms refer to the same person.
jockey_label_structure["contains_single_letter_token"] = (
    jockey_label_structure["raw_jockey_label"].str.contains(
        r"(?:^|\s)[A-Za-z](?:\s|$)",
        regex=True,
    )
)

# Summarise each structural feature by both distinct-label count and source-row
# coverage. Row coverage helps distinguish widespread conventions from rare
# label-level curiosities.
structure_flags = [
    "contains_period",
    "contains_apostrophe",
    "contains_hyphen",
    "contains_parenthesis",
    "contains_bracket",
    "contains_digit",
    "contains_non_ascii",
    "contains_single_letter_token",
]

jockey_structure_summary_rows = []

for flag in structure_flags:
    matching_labels = jockey_label_structure.loc[
        jockey_label_structure[flag]
    ]

    jockey_structure_summary_rows.append(
        {
            "structural_feature": flag,
            "distinct_raw_labels": len(matching_labels),
            "runner_rows": int(matching_labels["runner_rows"].sum()),
        }
    )

jockey_structure_summary = pd.DataFrame(
    jockey_structure_summary_rows
)

# Add percentages against the complete populated raw-label and runner-row
# populations. These percentages remain descriptive source statistics.
jockey_structure_summary["label_share_pct"] = (
    jockey_structure_summary["distinct_raw_labels"]
    / len(jockey_label_structure)
    * 100
).round(2)

jockey_structure_summary["runner_row_share_pct"] = (
    jockey_structure_summary["runner_rows"]
    / jockey_label_structure["runner_rows"].sum()
    * 100
).round(2)

# Show the token-count distribution separately because token count is
# categorical shape information rather than a Boolean structural feature.
jockey_token_count_summary = (
    jockey_label_structure.groupby("token_count", as_index=False)
    .agg(
        distinct_raw_labels=("raw_jockey_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("token_count")
)

display(jockey_structure_summary)
display(jockey_token_count_summary)

,structural_feature,distinct_raw_labels,runner_rows,label_share_pct,runner_row_share_pct
0,contains_period,0,0,0.00,0.00
1,contains_apostrophe,0,0,0.00,0.00
2,contains_hyphen,186,24259,2.35,1.31
3,contains_parenthesis,0,0,0.00,0.00
4,contains_bracket,0,0,0.00,0.00
5,contains_digit,0,0,0.00,0.00
6,contains_non_ascii,0,0,0.00,0.00
7,contains_single_letter_token,1757,140694,22.19,7.60


,token_count,distinct_raw_labels,runner_rows
0,1,1,2
1,2,4629,1628365
2,3,2707,195179
3,4,549,26946
4,5,28,719
5,6,2,71
6,7,1,1


### Jockey-label structural findings

The populated jockey labels follow a strongly standardised source format.

Across **7,917** distinct raw labels:

* **0** contain periods;
* **0** contain apostrophes;
* **0** contain parentheses or square brackets;
* **0** contain digits;
* **0** contain non-ASCII characters;
* **186** contain hyphens;
* **1,757** contain at least one single-letter token.

The absence of periods does not mean initials are absent. Instead, the source commonly represents initials as separate unpunctuated tokens, as illustrated by labels such as `P J McDonald`.

Token structure is concentrated:

* **4,629** labels contain two tokens;
* **2,707** contain three tokens;
* **549** contain four tokens;
* only **32** contain five or more tokens;
* one populated label contains a single token and occurs on two rows.

The source therefore appears to apply substantial character-level standardisation. Accents, apostrophes and punctuation may have been removed or transliterated before the data reached this database. Their absence must not be interpreted as proof that the corresponding real-world names lack those characters.

Initial-based labels are material at label level but represent a smaller share of source rows:

* **22.19%** of distinct labels contain a single-letter token;
* those labels account for **7.60%** of populated jockey rows.

The next step is to inspect the structurally unusual residue directly. This includes the single-token label, labels with five or more tokens, and representative high-frequency initial and hyphenated forms. Inspection remains descriptive and will not establish identity equivalence.

In [5]:
# Inspect the small structural residue and representative common conventions.
#
# This cell preserves and displays exact raw labels only. Selection is based
# on visible label structure and frequency; it does not imply that any label
# is malformed, duplicated or equivalent to another label.

# The single-token population is sufficiently small to inspect in full.
single_token_jockey_labels = (
    jockey_label_structure.loc[
        jockey_label_structure["token_count"].eq(1),
        [
            "raw_jockey_label",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_jockey_label"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

# Labels with five or more tokens are rare enough to inspect completely.
# Long labels may reflect multiple initials, compound names, titles, source
# defects or jurisdiction-specific presentation, so no interpretation is
# assigned before reviewing the exact strings.
long_jockey_labels = (
    jockey_label_structure.loc[
        jockey_label_structure["token_count"].ge(5),
        [
            "raw_jockey_label",
            "token_count",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["token_count", "runner_rows", "raw_jockey_label"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

# Show the most frequent labels containing single-letter tokens. These provide
# examples of the source's unpunctuated initial convention without assuming
# that an expanded form exists elsewhere or refers to the same person.
common_initial_jockey_labels = (
    jockey_label_structure.loc[
        jockey_label_structure["contains_single_letter_token"],
        [
            "raw_jockey_label",
            "token_count",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_jockey_label"],
        ascending=[False, True],
    )
    .head(30)
    .reset_index(drop=True)
)

# Show the most frequent hyphenated labels. Hyphens can be legitimate parts
# of names and are not treated as separators or removable punctuation.
common_hyphenated_jockey_labels = (
    jockey_label_structure.loc[
        jockey_label_structure["contains_hyphen"],
        [
            "raw_jockey_label",
            "token_count",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_jockey_label"],
        ascending=[False, True],
    )
    .head(30)
    .reset_index(drop=True)
)

display(single_token_jockey_labels)
display(long_jockey_labels)
display(common_initial_jockey_labels)
display(common_hyphenated_jockey_labels)

,raw_jockey_label,runner_rows,provisional_races,first_date,last_date
0,Reserve,2,2,2026-01-10,2026-02-13


,raw_jockey_label,token_count,runner_rows,provisional_races,first_date,last_date
0,Mr Cesar Alonso Vidal De La Pena,7,1,1,2018-04-13,2018-04-13
1,Al Moatasem bin Said Al Balushi,6,64,64,2015-11-17,2026-04-11
2,Victoria Alonso Vidal De La Pena,6,7,7,2021-12-16,2025-12-17
3,Mr M J M OSullivan,5,237,237,2016-05-19,2022-07-29
4,Qais Bin Saif Al Busaidi,5,198,198,2022-12-18,2026-04-11
5,Mlle Anna Van Den Troost,5,116,116,2016-06-16,2024-04-14
6,Anas bin Salim Al Siyabi,5,49,49,2016-12-15,2026-04-24
7,Mr S J P Baragry,5,36,36,2017-04-17,2024-04-01
8,Mr Gabriel Roth Le Vaillant,5,22,22,2018-07-04,2023-06-04
9,Mr R M P McNally,5,16,16,2015-11-29,2018-04-12


,raw_jockey_label,token_count,runner_rows,provisional_races,first_date,last_date
0,P J McDonald,3,7693,7693,2015-01-01,2026-05-27
1,W J Lee,3,6317,6317,2015-01-02,2026-05-26
2,K C Leung,3,4715,4715,2015-04-26,2026-05-27
3,C Y Ho,3,4572,4572,2015-05-31,2026-05-27
4,J F Egan,3,4249,4249,2015-01-01,2026-05-26
5,J J Slevin,3,3755,3755,2015-01-10,2026-05-27
6,M L Yeung,3,3672,3672,2015-05-31,2026-05-27
7,N G McCullagh,3,2862,2862,2015-01-02,2026-05-19
8,J M Sheridan,3,2617,2617,2019-04-06,2026-05-26
9,L P Dempsey,3,2466,2466,2015-01-01,2026-01-03


,raw_jockey_label,token_count,runner_rows,provisional_races,first_date,last_date
0,Sam Twiston-Davies,2,7420,7420,2015-01-01,2026-05-27
1,Pierre-Charles Boudot,2,4544,4544,2015-01-10,2026-02-14
2,Pierre-Louis Jamin,2,1503,1503,2015-09-21,2026-05-27
3,Conor Stone-Walsh,2,1032,1032,2022-05-24,2026-05-27
4,William Twiston-Davies,2,913,913,2015-01-02,2018-09-12
5,Christophe-Patrice Lemaire,2,832,832,2015-04-05,2026-05-24
6,Emma Smith-Chaston,2,599,599,2017-05-31,2025-04-22
7,Tom Kiely-Marshall,2,486,486,2023-05-03,2026-05-27
8,Elle-May Croot,2,439,439,2021-01-02,2026-04-05
9,Leo-Paul Brechet,2,402,402,2023-09-03,2026-05-26


### Structural residue findings

Direct inspection confirms that unusual jockey-label structures are not one homogeneous problem.

The only single-token populated label is `Reserve`, occurring twice in 2026. This appears structurally unlike a personal name and must be investigated as a possible source placeholder rather than assigned a participant identity.

Long labels include several legitimate naming structures:

* multi-part surnames;
* Arabic patronymic or family-name conventions;
* embedded initials;
* compound surnames;
* amateur or social titles such as `Mr`, `Ms`, `Mlle`, `Mme` and `Frau`;
* suffixes such as `Jr`.

The source therefore mixes personal-name content with presentation metadata. A title cannot safely remain part of the eventual identity key without first testing whether the same person appears under another title or without one.

The displayed residue also contains possible variation candidates, including:

* `Mr S J P Baragry` and `Mr S J P Bargary`;
* `Mlle Anna Van Den Troost` and `Mme Anna Van Den Troost`.

These examples do not yet justify correction or equivalence. Similar strings may represent source defects, title changes, or different people. They require chronology, jurisdiction, race context and external evidence before any governed merge.

The next step measures title and prefix conventions source-wide. No title is removed from the raw label and no person-level identity is inferred.

In [7]:
# Profile the first whitespace-separated token of each exact raw jockey label.
#
# The first token may be a forename, initial, honorific, amateur designation
# or another part of a legitimate personal name. This cell describes the
# source convention only; it does not remove prefixes or create identity keys.
jockey_label_prefix_profile = jockey_label_frequency.copy()

jockey_label_prefix_profile["first_token"] = (
    jockey_label_prefix_profile["raw_jockey_label"]
    .str.split()
    .str[0]
)

# These title-like tokens were observed directly in the inspected residue.
# They are treated as candidate presentation prefixes, not as a complete or
# authoritative list of racing titles.
observed_title_like_tokens = {
    "Mr",
    "Mrs",
    "Miss",
    "Ms",
    "Mlle",
    "Mme",
    "Frau",
}

jockey_label_prefix_profile["has_observed_title_prefix"] = (
    jockey_label_prefix_profile["first_token"].isin(
        observed_title_like_tokens
    )
)

# Summarise every first token so that common source conventions and unexpected
# prefixes remain visible. Counts refer to exact raw labels and source rows,
# not confirmed people or careers.
jockey_first_token_summary = (
    jockey_label_prefix_profile.groupby(
        "first_token",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_jockey_label", "size"),
        runner_rows=("runner_rows", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
    )
    .sort_values(
        ["distinct_raw_labels", "runner_rows", "first_token"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

# Summarise only the observed title-like prefix population.
# Raw labels remain unchanged; this isolates the scale of title-bearing labels
# before testing whether title variants overlap with untitled labels.
jockey_title_prefix_summary = (
    jockey_label_prefix_profile.loc[
        jockey_label_prefix_profile["has_observed_title_prefix"]
    ]
    .groupby(
        "first_token",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_jockey_label", "size"),
        runner_rows=("runner_rows", "sum"),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
    )
    .sort_values(
        ["distinct_raw_labels", "runner_rows", "first_token"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

# Show the most frequent exact labels for each observed title-like prefix.
# This supports inspection of how titles are used without assuming that two
# differently titled labels refer to the same person.
title_label_examples = (
    jockey_label_prefix_profile.loc[
        jockey_label_prefix_profile["has_observed_title_prefix"],
        [
            "first_token",
            "raw_jockey_label",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ],
    ]
    .sort_values(
        ["first_token", "runner_rows", "raw_jockey_label"],
        ascending=[True, False, True],
    )
    .groupby(
        "first_token",
        group_keys=False,
    )
    .head(15)
    .reset_index(drop=True)
)

display(jockey_first_token_summary.head(30))
display(jockey_title_prefix_summary)
display(title_label_examples)

,first_token,distinct_raw_labels,runner_rows,first_date,last_date
0,Mr,1238,44623,2015-01-01,2026-05-27
1,Miss,565,15386,2015-01-01,2026-05-27
2,Mlle,222,14836,2015-01-03,2025-09-10
3,Mme,150,5353,2015-01-10,2026-05-24
4,Ms,91,2209,2015-01-01,2026-05-27
5,Jose,52,3706,2015-01-01,2026-05-17
6,A,48,5807,2015-01-01,2026-05-03
7,David,44,38404,2015-01-01,2026-05-27
8,J,44,11585,2015-01-01,2026-05-27
9,M,44,5557,2015-01-01,2026-05-27


,first_token,distinct_raw_labels,runner_rows,first_date,last_date
0,Mr,1238,44623,2015-01-01,2026-05-27
1,Miss,565,15386,2015-01-01,2026-05-27
2,Mlle,222,14836,2015-01-03,2025-09-10
3,Mme,150,5353,2015-01-10,2026-05-24
4,Ms,91,2209,2015-01-01,2026-05-27
5,Mrs,31,483,2015-01-02,2026-05-16
6,Frau,27,313,2015-04-14,2026-05-15


,first_token,raw_jockey_label,runner_rows,provisional_races,first_date,last_date
0,Frau,Frau Berit Weber,56,56,2015-08-03,2025-11-02
1,Frau,Frau Stefanie Koyuncu,56,56,2018-05-11,2025-11-08
2,Frau,Frau Lilli-Marie Engels,44,44,2016-08-01,2025-11-08
3,Frau,Frau Nina Baltromei,30,30,2023-07-06,2026-05-14
4,Frau,Frau M Riehl,29,29,2019-12-02,2025-08-17
...,...,...,...,...,...,...
100,Ms,Ms L A Byrne,23,23,2024-09-06,2026-04-04
101,Ms,Ms L Ward,21,21,2021-04-13,2023-04-05
102,Ms,Ms C A Walsh,20,20,2022-12-21,2024-07-18
103,Ms,Ms Hannah Smullen,20,20,2023-07-20,2025-10-04


## 3. Candidate duplicate and variant jockey labels

The source contains 7,917 distinct raw jockey labels. A distinct label is not yet assumed to represent either:

* one unique real-world individual; or
* a unique presentation of that individual.

The first candidate-generation stage tests only a narrow presentation difference already observed in the source: a leading title such as `Mr`, `Miss`, `Mlle`, `Mme`, `Ms`, `Mrs` or `Frau`.

For comparison only, the leading title is removed and whitespace is standardised. Raw labels remain unchanged.

A shared comparison key identifies labels requiring investigation; it does not establish that the labels belong to the same person. Every resulting candidate group must later be checked against source context and authoritative external evidence before any merge or split decision is recorded.

No fuzzy matching, spelling correction, initial expansion or identity resolution is performed in this stage.

In [10]:
# Generate the narrowest defensible set of possible jockey-label variants.
#
# The comparison removes only the leading title forms already demonstrated by
# the source profile. It does not remove initials, punctuation within names,
# hyphens or any other potentially identity-bearing information.
observed_jockey_title_prefixes = {
    "Mr",
    "Mrs",
    "Miss",
    "Ms",
    "Mlle",
    "Mme",
    "Frau",
}

jockey_strict_comparison = jockey_label_frequency.copy()

# Preserve the first token separately so each candidate retains evidence of
# whether its raw source label contained one of the observed title prefixes.
jockey_strict_comparison["first_token"] = (
    jockey_strict_comparison["raw_jockey_label"]
    .str.split()
    .str[0]
)

jockey_strict_comparison["has_observed_title_prefix"] = (
    jockey_strict_comparison["first_token"].isin(
        observed_jockey_title_prefixes
    )
)

# Construct a candidate-generation key only.
#
# Leading titles are removed where present, repeated internal whitespace is
# collapsed, and case is folded. This key must never replace the raw label or
# be treated as a resolved person identifier.
jockey_strict_comparison["strict_comparison_key"] = (
    jockey_strict_comparison["raw_jockey_label"]
    .where(
        ~jockey_strict_comparison["has_observed_title_prefix"],
        jockey_strict_comparison["raw_jockey_label"]
        .str.split(n=1)
        .str[1],
    )
    .str.split()
    .str.join(" ")
    .str.casefold()
)

# Count the number of distinct raw labels represented by each comparison key.
# Only keys containing more than one raw label are possible variant groups.
strict_comparison_group_counts = (
    jockey_strict_comparison.groupby(
        "strict_comparison_key",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_jockey_label", "nunique"),
        combined_runner_rows=("runner_rows", "sum"),
        earliest_date=("first_date", "min"),
        latest_date=("last_date", "max"),
    )
)

strict_candidate_keys = strict_comparison_group_counts.loc[
    strict_comparison_group_counts["distinct_raw_labels"].gt(1)
].copy()

# Attach the individual raw labels back to each candidate group so that no
# variation is hidden inside an aggregate.
jockey_strict_candidate_labels = (
    jockey_strict_comparison.merge(
        strict_candidate_keys,
        how="inner",
        on="strict_comparison_key",
        validate="many_to_one",
        suffixes=("", "_group"),
    )
    .sort_values(
        [
            "combined_runner_rows",
            "strict_comparison_key",
            "raw_jockey_label",
        ],
        ascending=[False, True, True],
    )
    .reset_index(drop=True)
)

# Summarise the scale of the candidate residue. These counts describe possible
# label variants only and make no claim about real-world identity.
strict_candidate_summary = pd.DataFrame(
    [
        {
            "candidate_groups": len(strict_candidate_keys),
            "raw_labels_in_candidate_groups": (
                jockey_strict_candidate_labels[
                    "raw_jockey_label"
                ].nunique()
            ),
            "title_bearing_candidate_labels": int(
                jockey_strict_candidate_labels[
                    "has_observed_title_prefix"
                ].sum()
            ),
            "candidate_group_runner_rows": int(
                strict_candidate_keys[
                    "combined_runner_rows"
                ].sum()
            ),
        }
    ]
)

display(strict_candidate_summary)

display(
    jockey_strict_candidate_labels[
        [
            "strict_comparison_key",
            "raw_jockey_label",
            "has_observed_title_prefix",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
            "distinct_raw_labels",
            "combined_runner_rows",
        ]
    ]
)

,candidate_groups,raw_labels_in_candidate_groups,title_bearing_candidate_labels,candidate_group_runner_rows
0,212,426,284,42389


,strict_comparison_key,raw_jockey_label,has_observed_title_prefix,runner_rows,provisional_races,first_date,last_date,distinct_raw_labels,combined_runner_rows
0,marie velon,Mlle Marie Velon,True,1428,1428,2017-10-17,2023-12-29,2,2449
1,marie velon,Mme Marie Velon,True,1021,1021,2024-01-19,2026-05-24,2,2449
2,coralie pacaut,Mlle Coralie Pacaut,True,1571,1571,2015-10-29,2023-12-30,2,1897
3,coralie pacaut,Mme Coralie Pacaut,True,326,326,2024-03-08,2026-05-22,2,1897
4,frida valle skar,Mlle Frida Valle Skar,True,797,797,2017-11-27,2024-01-13,2,1470
...,...,...,...,...,...,...,...,...,...
421,juliette benech,Mme Juliette Benech,True,1,1,2025-06-28,2025-06-28,2,2
422,lucas gueracague,Lucas Gueracague,False,1,1,2025-08-07,2025-08-07,2,2
423,lucas gueracague,Mr Lucas Gueracague,True,1,1,2026-05-17,2026-05-17,2,2
424,thomas briand,Mr Thomas Briand,True,1,1,2025-12-12,2025-12-12,2,2


### Strict candidate-pair chronology

The strict comparison produced 212 candidate groups containing 426 distinct raw labels.

These groups may represent different phenomena:

* transition between two title forms;
* titled and untitled presentation of the same person;
* simultaneous source conventions;
* genuinely different people sharing the same title-stripped name;
* source errors or other unresolved cases.

The next stage expands each group into individual raw-label pairs and compares their observed active periods.

Temporal separation can support a possible presentation transition, while overlapping activity increases collision risk. Neither result proves identity:

* non-overlapping labels may still represent different people;
* overlapping labels may still represent one person recorded under competing source conventions.

The output remains a candidate-review list. No pair is merged or resolved here.

In [11]:
from itertools import combinations

# Expand each strict comparison group into every unordered pair of raw labels.
#
# Pair-level evidence is easier to review than a group aggregate, especially
# if a comparison key contains more than two source labels.
strict_candidate_pair_rows = []

for (
    strict_comparison_key,
    candidate_group,
) in jockey_strict_candidate_labels.groupby(
    "strict_comparison_key",
    sort=True,
):
    candidate_records = candidate_group.to_dict("records")

    for left_record, right_record in combinations(
        candidate_records,
        2,
    ):
        # Convert the source date boundaries to timestamps solely for interval
        # comparison. The original date strings remain available in the output.
        left_first_date = pd.Timestamp(left_record["first_date"])
        left_last_date = pd.Timestamp(left_record["last_date"])
        right_first_date = pd.Timestamp(right_record["first_date"])
        right_last_date = pd.Timestamp(right_record["last_date"])

        # Two observed periods overlap when neither interval ends before the
        # other begins. Same-day boundaries therefore count as overlap.
        overlap_start = max(left_first_date, right_first_date)
        overlap_end = min(left_last_date, right_last_date)
        periods_overlap = overlap_start <= overlap_end

        if periods_overlap:
            temporal_relationship = "observed_periods_overlap"
            overlap_days = (
                overlap_end - overlap_start
            ).days + 1
            gap_days = 0
        else:
            temporal_relationship = "observed_periods_separate"
            overlap_days = 0

            # Calculate the number of clear calendar days between the two
            # observed periods. This describes source chronology only and does
            # not establish that one label succeeded the other.
            if left_last_date < right_first_date:
                gap_days = (
                    right_first_date - left_last_date
                ).days - 1
            else:
                gap_days = (
                    left_first_date - right_last_date
                ).days - 1

        left_has_title = bool(
            left_record["has_observed_title_prefix"]
        )
        right_has_title = bool(
            right_record["has_observed_title_prefix"]
        )

        # Classify only the visible source-label structure. The classification
        # does not interpret a title as a licence, marital or identity status.
        if left_has_title and right_has_title:
            pair_label_structure = "two_title_bearing_labels"
        elif left_has_title or right_has_title:
            pair_label_structure = "title_bearing_and_untitled"
        else:
            pair_label_structure = "two_untitled_labels"

        strict_candidate_pair_rows.append(
            {
                "strict_comparison_key": strict_comparison_key,
                "left_raw_jockey_label": (
                    left_record["raw_jockey_label"]
                ),
                "right_raw_jockey_label": (
                    right_record["raw_jockey_label"]
                ),
                "left_first_token": left_record["first_token"],
                "right_first_token": right_record["first_token"],
                "pair_label_structure": pair_label_structure,
                "left_runner_rows": left_record["runner_rows"],
                "right_runner_rows": right_record["runner_rows"],
                "left_first_date": left_record["first_date"],
                "left_last_date": left_record["last_date"],
                "right_first_date": right_record["first_date"],
                "right_last_date": right_record["last_date"],
                "temporal_relationship": temporal_relationship,
                "overlap_days": overlap_days,
                "gap_days": gap_days,
                "combined_runner_rows": (
                    left_record["runner_rows"]
                    + right_record["runner_rows"]
                ),
            }
        )

jockey_strict_candidate_pairs = pd.DataFrame(
    strict_candidate_pair_rows
)

# Summarise how much of the strict residue consists of title transitions,
# title-versus-untitled forms and temporally overlapping candidates.
strict_pair_summary = (
    jockey_strict_candidate_pairs.groupby(
        [
            "pair_label_structure",
            "temporal_relationship",
        ],
        as_index=False,
    )
    .agg(
        candidate_pairs=(
            "strict_comparison_key",
            "size",
        ),
        comparison_keys=(
            "strict_comparison_key",
            "nunique",
        ),
        combined_runner_rows=(
            "combined_runner_rows",
            "sum",
        ),
    )
    .sort_values(
        [
            "pair_label_structure",
            "temporal_relationship",
        ]
    )
    .reset_index(drop=True)
)

# Rank overlapping candidates first because simultaneous use of two labels is
# especially important when testing whether a comparison key hides more than
# one real person. High-volume pairs are shown before sparse pairs.
jockey_strict_candidate_pairs = (
    jockey_strict_candidate_pairs.sort_values(
        [
            "temporal_relationship",
            "combined_runner_rows",
            "strict_comparison_key",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

display(strict_pair_summary)
display(jockey_strict_candidate_pairs)

,pair_label_structure,temporal_relationship,candidate_pairs,comparison_keys,combined_runner_rows
0,title_bearing_and_untitled,observed_periods_overlap,28,28,4830
1,title_bearing_and_untitled,observed_periods_separate,114,114,19323
2,two_title_bearing_labels,observed_periods_overlap,8,8,3219
3,two_title_bearing_labels,observed_periods_separate,65,65,15171
4,two_untitled_labels,observed_periods_separate,1,1,14


,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,left_first_token,right_first_token,pair_label_structure,left_runner_rows,right_runner_rows,left_first_date,left_last_date,right_first_date,right_last_date,temporal_relationship,overlap_days,gap_days,combined_runner_rows
0,frida valle skar,Mlle Frida Valle Skar,Mme Frida Valle Skar,Mlle,Mme,two_title_bearing_labels,797,673,2017-11-27,2024-01-13,2024-01-05,2026-05-24,observed_periods_overlap,9,0,1470
1,tristan durrell,Mr Tristan Durrell,Tristan Durrell,Mr,Tristan,title_bearing_and_untitled,114,748,2019-03-02,2022-04-29,2018-11-19,2026-05-25,observed_periods_overlap,1155,0,862
2,toby wynne,Mr Toby Wynne,Toby Wynne,Mr,Toby,title_bearing_and_untitled,66,708,2020-02-19,2022-06-03,2022-05-02,2026-05-25,observed_periods_overlap,33,0,774
3,b oneill,Miss B ONeill,Mr B ONeill,Miss,Mr,two_title_bearing_labels,10,700,2015-05-02,2021-05-31,2015-02-10,2026-05-22,observed_periods_overlap,2222,0,710
4,jack andrews,Jack Andrews,Mr Jack Andrews,Jack,Mr,title_bearing_and_untitled,523,161,2015-05-14,2026-05-26,2016-04-16,2025-04-24,observed_periods_overlap,3296,0,684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,charlotta ericsson,Charlotta Ericsson,Miss Charlotta Ericsson,Charlotta,Miss,title_bearing_and_untitled,1,1,2023-09-17,2023-09-17,2024-03-21,2024-03-21,observed_periods_separate,0,185,2
212,j osullivan,J OSullivan,Mr J OSullivan,J,Mr,title_bearing_and_untitled,1,1,2019-11-01,2019-11-01,2016-05-23,2016-05-23,observed_periods_separate,0,1256,2
213,juliette benech,Mlle Juliette Benech,Mme Juliette Benech,Mlle,Mme,two_title_bearing_labels,1,1,2021-08-07,2021-08-07,2025-06-28,2025-06-28,observed_periods_separate,0,1420,2
214,lucas gueracague,Lucas Gueracague,Mr Lucas Gueracague,Lucas,Mr,title_bearing_and_untitled,1,1,2025-08-07,2025-08-07,2026-05-17,2026-05-17,observed_periods_separate,0,282,2


### Same-race collision test

Observed date overlap does not establish that two labels represent different people. One jockey may be recorded under competing presentation conventions during the same period.

A stronger test is whether both labels occur within the same provisional race.

The provisional race key is:

`date + course + off`

Where two candidate labels occur on separate runner rows within the same provisional race, they cannot ordinarily represent one jockey riding both runners. Such a result is therefore evidence against merging the labels automatically.

However, same-race occurrence is not by itself proof of two correctly recorded people. The underlying rows may still contain a source error, duplicated record or incorrect jockey attribution. Every detected collision remains subject to source-row inspection and external verification.

Labels without a same-race collision also remain unresolved. Absence of collision does not prove that they represent the same person.

In [12]:
# Test whether any strict candidate pair occurs on separate runner rows within
# the same provisional race.
#
# This is stronger negative identity evidence than simple active-date overlap:
# one real jockey cannot ordinarily ride two different horses in one race.
# The result still remains a review candidate because the source itself may be
# duplicated or incorrect.

strict_candidate_raw_labels = sorted(
    jockey_strict_candidate_labels[
        "raw_jockey_label"
    ].unique()
)

# Use a parameterised IN clause so only the 426 candidate-label populations
# are loaded from the immutable source. The raw labels are not modified.
candidate_label_placeholders = ", ".join(
    ["?"] * len(strict_candidate_raw_labels)
)

jockey_strict_candidate_occurrences = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        course,
        off,
        race_name,
        horse,
        trainer,
        jockey AS raw_jockey_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND jockey IN ({candidate_label_placeholders})
    """,
    connection,
    params=strict_candidate_raw_labels,
)

# Attach the exploratory comparison key to each immutable source occurrence.
# This mapping is used only to restrict race-level comparisons to labels that
# were already placed in the same strict candidate group.
raw_label_to_strict_key = (
    jockey_strict_candidate_labels[
        [
            "raw_jockey_label",
            "strict_comparison_key",
        ]
    ]
    .drop_duplicates()
    .set_index("raw_jockey_label")[
        "strict_comparison_key"
    ]
)

jockey_strict_candidate_occurrences[
    "strict_comparison_key"
] = jockey_strict_candidate_occurrences[
    "raw_jockey_label"
].map(raw_label_to_strict_key)

# Retain only provisional races containing more than one distinct raw label
# from the same strict comparison group.
same_race_candidate_groups = (
    jockey_strict_candidate_occurrences.groupby(
        [
            "strict_comparison_key",
            "date",
            "course",
            "off",
        ],
        as_index=False,
    )
    .agg(
        distinct_candidate_labels=(
            "raw_jockey_label",
            "nunique",
        ),
        candidate_runner_rows=(
            "source_rowid",
            "size",
        ),
    )
)

same_race_candidate_groups = same_race_candidate_groups.loc[
    same_race_candidate_groups[
        "distinct_candidate_labels"
    ].gt(1)
].copy()

# Recover the underlying runner rows for every detected candidate-group race.
# These rows preserve source lineage and allow each collision to be inspected
# through its horses, trainers and source row identifiers.
same_race_candidate_occurrences = (
    jockey_strict_candidate_occurrences.merge(
        same_race_candidate_groups[
            [
                "strict_comparison_key",
                "date",
                "course",
                "off",
            ]
        ],
        how="inner",
        on=[
            "strict_comparison_key",
            "date",
            "course",
            "off",
        ],
        validate="many_to_one",
    )
    .sort_values(
        [
            "strict_comparison_key",
            "date",
            "course",
            "off",
            "raw_jockey_label",
            "horse",
        ]
    )
    .reset_index(drop=True)
)

# Expand each same-race candidate group into its individual unordered label
# pairs. Sorting the two labels creates a stable pair key regardless of their
# order in the source rows or earlier candidate-generation output.
same_race_pair_rows = []

for (
    race_group_key,
    race_group,
) in same_race_candidate_occurrences.groupby(
    [
        "strict_comparison_key",
        "date",
        "course",
        "off",
    ],
    sort=True,
):
    (
        strict_comparison_key,
        race_date,
        race_course,
        race_off,
    ) = race_group_key

    race_labels = sorted(
        race_group["raw_jockey_label"].unique()
    )

    for left_label, right_label in combinations(
        race_labels,
        2,
    ):
        left_occurrences = race_group.loc[
            race_group["raw_jockey_label"].eq(left_label)
        ]
        right_occurrences = race_group.loc[
            race_group["raw_jockey_label"].eq(right_label)
        ]

        same_race_pair_rows.append(
            {
                "strict_comparison_key": strict_comparison_key,
                "pair_label_a": left_label,
                "pair_label_b": right_label,
                "date": race_date,
                "course": race_course,
                "off": race_off,
                "left_horses": " | ".join(
                    sorted(
                        left_occurrences["horse"]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
                "right_horses": " | ".join(
                    sorted(
                        right_occurrences["horse"]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
                "left_trainers": " | ".join(
                    sorted(
                        left_occurrences["trainer"]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
                "right_trainers": " | ".join(
                    sorted(
                        right_occurrences["trainer"]
                        .dropna()
                        .astype(str)
                        .unique()
                    )
                ),
                "source_rowids": " | ".join(
                    sorted(
                        race_group["source_rowid"]
                        .astype(str)
                        .unique()
                    )
                ),
            }
        )

same_race_candidate_pair_occurrences = pd.DataFrame(
    same_race_pair_rows
)

# Aggregate the race-level evidence to one row per candidate pair while
# retaining the date range and number of separate same-race collisions.
if same_race_candidate_pair_occurrences.empty:
    same_race_candidate_pair_summary = pd.DataFrame(
        columns=[
            "strict_comparison_key",
            "pair_label_a",
            "pair_label_b",
            "same_race_collisions",
            "first_collision_date",
            "last_collision_date",
        ]
    )
else:
    same_race_candidate_pair_summary = (
        same_race_candidate_pair_occurrences.groupby(
            [
                "strict_comparison_key",
                "pair_label_a",
                "pair_label_b",
            ],
            as_index=False,
        )
        .agg(
            same_race_collisions=("date", "size"),
            first_collision_date=("date", "min"),
            last_collision_date=("date", "max"),
        )
        .sort_values(
            [
                "same_race_collisions",
                "strict_comparison_key",
            ],
            ascending=[False, True],
        )
        .reset_index(drop=True)
    )

# Create the same stable unordered pair key on the complete strict pair table,
# then attach the collision count without changing any prior candidate result.
jockey_strict_candidate_pairs[
    "pair_label_a"
] = jockey_strict_candidate_pairs[
    [
        "left_raw_jockey_label",
        "right_raw_jockey_label",
    ]
].min(axis=1)

jockey_strict_candidate_pairs[
    "pair_label_b"
] = jockey_strict_candidate_pairs[
    [
        "left_raw_jockey_label",
        "right_raw_jockey_label",
    ]
].max(axis=1)

jockey_strict_candidate_pairs = (
    jockey_strict_candidate_pairs.merge(
        same_race_candidate_pair_summary,
        how="left",
        on=[
            "strict_comparison_key",
            "pair_label_a",
            "pair_label_b",
        ],
        validate="one_to_one",
    )
)

jockey_strict_candidate_pairs[
    "same_race_collisions"
] = (
    jockey_strict_candidate_pairs[
        "same_race_collisions"
    ]
    .fillna(0)
    .astype(int)
)

jockey_strict_candidate_pairs[
    "same_race_collision_status"
] = jockey_strict_candidate_pairs[
    "same_race_collisions"
].gt(0).map(
    {
        True: "collision_detected",
        False: "no_collision_observed",
    }
)

# Summarise the decisive negative evidence without interpreting collision-free
# pairs as confirmed aliases.
same_race_collision_summary = pd.DataFrame(
    [
        {
            "strict_candidate_pairs": len(
                jockey_strict_candidate_pairs
            ),
            "pairs_with_same_race_collision": int(
                jockey_strict_candidate_pairs[
                    "same_race_collisions"
                ].gt(0).sum()
            ),
            "pairs_without_same_race_collision": int(
                jockey_strict_candidate_pairs[
                    "same_race_collisions"
                ].eq(0).sum()
            ),
            "distinct_collision_races": len(
                same_race_candidate_groups
            ),
            "collision_source_rows": len(
                same_race_candidate_occurrences
            ),
        }
    ]
)

display(same_race_collision_summary)
display(same_race_candidate_pair_summary)
display(same_race_candidate_pair_occurrences)

,strict_candidate_pairs,pairs_with_same_race_collision,pairs_without_same_race_collision,distinct_collision_races,collision_source_rows
0,216,1,215,1,2


,strict_comparison_key,pair_label_a,pair_label_b,same_race_collisions,first_collision_date,last_collision_date
0,b oneill,Miss B ONeill,Mr B ONeill,1,2017-11-11,2017-11-11


,strict_comparison_key,pair_label_a,pair_label_b,date,course,off,left_horses,right_horses,left_trainers,right_trainers,source_rowids
0,b oneill,Miss B ONeill,Mr B ONeill,2017-11-11,Naas (IRE),4:05,Hawthorn Echo (IRE),Chisholm Trail (IRE),Peter McCreery,Paul Nolan,463339 | 463450


### Same-race collision finding

The same-race test found one collision among the 216 strict candidate pairs.

On 11 November 2017 at Naas, the 4:05 race contains:

* `Miss B ONeill` riding `Hawthorn Echo (IRE)` for Peter McCreery;
* `Mr B ONeill` riding `Chisholm Trail (IRE)` for Paul Nolan.

The two labels occur on separate source rows and separate runners within the same provisional race.

Published result evidence independently confirms both riders. Horse Racing Ireland identifies `Mr B ONeill` as Barry O'Neill. The complete identity of `Miss B ONeill` remains unresolved from the evidence inspected so far, but the simultaneous race occurrence proves that she is a different person.

Decision:

* `Miss B ONeill` and `Mr B ONeill` must not be merged;
* the shared title-stripped comparison key `b oneill` is a genuine real-world identity collision;
* title removal is suitable only for candidate generation and cannot be used as an automatic identity key;
* the immutable raw labels and both source-row lineages must remain preserved.

This decision depends on external evidence and must be captured with reusable provenance before Notebook 22 closes.

In [13]:
# Compare the source context of the 215 strict candidate pairs that never
# occur together in one provisional race.
#
# Shared horses can provide strong candidate evidence that two raw labels may
# describe the same rider under different presentation conventions. Shared
# trainers are weaker evidence because many jockeys ride for the same trainer.
#
# Neither measure resolves identity. The output only ranks the finite set that
# must later be checked against authoritative external evidence.

collision_free_strict_pairs = (
    jockey_strict_candidate_pairs.loc[
        jockey_strict_candidate_pairs[
            "same_race_collisions"
        ].eq(0)
    ]
    .copy()
    .reset_index(drop=True)
)

# Reduce the already-loaded candidate occurrences to distinct label/context
# combinations. Raw horse and trainer labels remain unchanged.
jockey_label_horse_context = (
    jockey_strict_candidate_occurrences[
        [
            "raw_jockey_label",
            "horse",
        ]
    ]
    .dropna(subset=["horse"])
    .drop_duplicates()
)

jockey_label_trainer_context = (
    jockey_strict_candidate_occurrences[
        [
            "raw_jockey_label",
            "trainer",
        ]
    ]
    .dropna(subset=["trainer"])
    .drop_duplicates()
)

pair_context_rows = []

for pair_record in collision_free_strict_pairs.to_dict("records"):
    left_label = pair_record["left_raw_jockey_label"]
    right_label = pair_record["right_raw_jockey_label"]

    # Build exact raw-label sets separately for each candidate label. No horse
    # or trainer identity normalisation is introduced at this stage.
    left_horses = set(
        jockey_label_horse_context.loc[
            jockey_label_horse_context[
                "raw_jockey_label"
            ].eq(left_label),
            "horse",
        ]
    )
    right_horses = set(
        jockey_label_horse_context.loc[
            jockey_label_horse_context[
                "raw_jockey_label"
            ].eq(right_label),
            "horse",
        ]
    )

    left_trainers = set(
        jockey_label_trainer_context.loc[
            jockey_label_trainer_context[
                "raw_jockey_label"
            ].eq(left_label),
            "trainer",
        ]
    )
    right_trainers = set(
        jockey_label_trainer_context.loc[
            jockey_label_trainer_context[
                "raw_jockey_label"
            ].eq(right_label),
            "trainer",
        ]
    )

    shared_horses = sorted(left_horses & right_horses)
    shared_trainers = sorted(left_trainers & right_trainers)

    pair_context_rows.append(
        {
            "strict_comparison_key": (
                pair_record["strict_comparison_key"]
            ),
            "left_raw_jockey_label": left_label,
            "right_raw_jockey_label": right_label,
            "pair_label_structure": (
                pair_record["pair_label_structure"]
            ),
            "temporal_relationship": (
                pair_record["temporal_relationship"]
            ),
            "overlap_days": pair_record["overlap_days"],
            "gap_days": pair_record["gap_days"],
            "left_runner_rows": pair_record["left_runner_rows"],
            "right_runner_rows": pair_record["right_runner_rows"],
            "shared_horse_count": len(shared_horses),
            "shared_horses": " | ".join(shared_horses),
            "shared_trainer_count": len(shared_trainers),
            "shared_trainers": " | ".join(shared_trainers),
            "combined_runner_rows": (
                pair_record["combined_runner_rows"]
            ),
        }
    )

jockey_strict_pair_context = pd.DataFrame(
    pair_context_rows
)

# Classify the source-internal support conservatively.
#
# Shared horses are treated as stronger candidate evidence than shared
# trainers. A lack of shared context does not establish different identities.
jockey_strict_pair_context["context_candidate_status"] = (
    "no_shared_source_context"
)

jockey_strict_pair_context.loc[
    jockey_strict_pair_context[
        "shared_trainer_count"
    ].gt(0),
    "context_candidate_status",
] = "shared_trainer_only"

jockey_strict_pair_context.loc[
    jockey_strict_pair_context[
        "shared_horse_count"
    ].gt(0),
    "context_candidate_status",
] = "shared_horse"

strict_pair_context_summary = (
    jockey_strict_pair_context.groupby(
        [
            "context_candidate_status",
            "temporal_relationship",
        ],
        as_index=False,
    )
    .agg(
        candidate_pairs=(
            "strict_comparison_key",
            "size",
        ),
        combined_runner_rows=(
            "combined_runner_rows",
            "sum",
        ),
    )
    .sort_values(
        [
            "context_candidate_status",
            "temporal_relationship",
        ]
    )
    .reset_index(drop=True)
)

# Rank shared-horse candidates first, followed by shared-trainer candidates.
# Higher-context and higher-volume pairs are shown first for review.
context_status_rank = {
    "shared_horse": 0,
    "shared_trainer_only": 1,
    "no_shared_source_context": 2,
}

jockey_strict_pair_context["context_status_rank"] = (
    jockey_strict_pair_context[
        "context_candidate_status"
    ].map(context_status_rank)
)

jockey_strict_pair_context = (
    jockey_strict_pair_context.sort_values(
        [
            "context_status_rank",
            "shared_horse_count",
            "shared_trainer_count",
            "combined_runner_rows",
            "strict_comparison_key",
        ],
        ascending=[True, False, False, False, True],
    )
    .drop(columns="context_status_rank")
    .reset_index(drop=True)
)

display(strict_pair_context_summary)
display(jockey_strict_pair_context)

,context_candidate_status,temporal_relationship,candidate_pairs,combined_runner_rows
0,no_shared_source_context,observed_periods_overlap,5,66
1,no_shared_source_context,observed_periods_separate,50,1502
2,shared_horse,observed_periods_overlap,23,6152
3,shared_horse,observed_periods_separate,91,30779
4,shared_trainer_only,observed_periods_overlap,7,1121
5,shared_trainer_only,observed_periods_separate,39,2227


,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,pair_label_structure,temporal_relationship,overlap_days,gap_days,left_runner_rows,right_runner_rows,shared_horse_count,shared_horses,shared_trainer_count,shared_trainers,combined_runner_rows,context_candidate_status
0,marie velon,Mlle Marie Velon,Mme Marie Velon,two_title_bearing_labels,observed_periods_separate,0,20,1428,1021,80,Abstract (FR) | Acclam (FR) | Adhamiy (FR) | A...,116,A & G Botti | A S & D Allard | A Schepens | A ...,2449,shared_horse
1,frida valle skar,Mlle Frida Valle Skar,Mme Frida Valle Skar,two_title_bearing_labels,observed_periods_overlap,9,0,797,673,43,Almazora (FR) | Amoureuse (FR) | Anonyme (FR) ...,35,A Lopez | A S & D Allard | A Sagot | B Goudot ...,1470,shared_horse
2,perrine cheyer,Mlle Perrine Cheyer,Mme Perrine Cheyer,two_title_bearing_labels,observed_periods_separate,0,5,416,242,35,A Stolen Kiss (FR) | Albert Bridge (FR) | Bonn...,37,A Brenckle | A S & D Allard | A Sagot | A Sche...,658,shared_horse
3,jack andrews,Jack Andrews,Mr Jack Andrews,title_bearing_and_untitled,observed_periods_overlap,3296,0,523,161,30,Anariza (FR) | Begin The Luck (IRE) | Boleyn B...,14,Ben Case | Caroline Bailey | Caroline Fryer | ...,684,shared_horse
4,ambre molins,Mlle Ambre Molins,Mme Ambre Molins,two_title_bearing_labels,observed_periods_separate,0,14,490,420,26,Americano (FR) | Auen Adventure (GER) | Dulini...,30,B Goudot | B Legros | Brian Beaunez | C Boutin...,910,shared_horse
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210,charlotta ericsson,Charlotta Ericsson,Miss Charlotta Ericsson,title_bearing_and_untitled,observed_periods_separate,0,185,1,1,0,,0,,2,no_shared_source_context
211,j osullivan,J OSullivan,Mr J OSullivan,title_bearing_and_untitled,observed_periods_separate,0,1256,1,1,0,,0,,2,no_shared_source_context
212,juliette benech,Mlle Juliette Benech,Mme Juliette Benech,two_title_bearing_labels,observed_periods_separate,0,1420,1,1,0,,0,,2,no_shared_source_context
213,lucas gueracague,Lucas Gueracague,Mr Lucas Gueracague,title_bearing_and_untitled,observed_periods_separate,0,282,1,1,0,,0,,2,no_shared_source_context


### Strict candidate manual-review queue

The 215 collision-free strict pairs show materially different levels of source-internal support.

Pairs sharing exact horse labels are the strongest alias candidates because the same horses appear under both jockey labels. Shared trainers provide weaker support, while pairs with no shared source context rely only on title-stripped name equivalence.

This evidence is used only to order manual research. It does not resolve identity.

The review queue therefore records:

* the known same-race collision as a confirmed split candidate;
* shared-horse pairs as the highest-priority alias candidates;
* shared-trainer-only pairs as secondary candidates;
* pairs with no shared context as lower-evidence candidates that still require review.

Every pair remains unresolved until authoritative external evidence supports a merge, split or unresolved decision.

In [14]:
# Build one complete strict-candidate review queue containing the confirmed
# same-race collision and all 215 collision-free candidate pairs.
#
# Review priority reflects the strength of source-internal evidence only. It is
# not an identity confidence score and does not authorise any automatic merge.

strict_candidate_review_queue = (
    jockey_strict_candidate_pairs[
        [
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "pair_label_structure",
            "temporal_relationship",
            "overlap_days",
            "gap_days",
            "left_runner_rows",
            "right_runner_rows",
            "combined_runner_rows",
            "same_race_collisions",
        ]
    ]
    .merge(
        jockey_strict_pair_context[
            [
                "strict_comparison_key",
                "left_raw_jockey_label",
                "right_raw_jockey_label",
                "shared_horse_count",
                "shared_horses",
                "shared_trainer_count",
                "shared_trainers",
                "context_candidate_status",
            ]
        ],
        how="left",
        on=[
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
        ],
        validate="one_to_one",
    )
)

# The same-race collision was excluded from the earlier context table, so its
# context fields remain blank. This is deliberate: its collision evidence
# already places it in the strongest do-not-merge review category.
strict_candidate_review_queue[
    "context_candidate_status"
] = strict_candidate_review_queue[
    "context_candidate_status"
].fillna("same_race_collision")

strict_candidate_review_queue[
    [
        "shared_horse_count",
        "shared_trainer_count",
    ]
] = strict_candidate_review_queue[
    [
        "shared_horse_count",
        "shared_trainer_count",
    ]
].fillna(0).astype(int)

strict_candidate_review_queue[
    [
        "shared_horses",
        "shared_trainers",
    ]
] = strict_candidate_review_queue[
    [
        "shared_horses",
        "shared_trainers",
    ]
].fillna("")

# Assign review categories without making an identity decision.
#
# A same-race collision is reviewed first as possible evidence of two people or
# a source attribution defect. Shared-horse pairs follow because they provide
# the strongest positive alias evidence available internally.
review_priority_by_status = {
    "same_race_collision": 1,
    "shared_horse": 2,
    "shared_trainer_only": 3,
    "no_shared_source_context": 4,
}

review_reason_by_status = {
    "same_race_collision": (
        "labels occur on separate runners in the same provisional race"
    ),
    "shared_horse": (
        "labels share one or more exact raw horse labels"
    ),
    "shared_trainer_only": (
        "labels share trainers but no exact raw horse labels"
    ),
    "no_shared_source_context": (
        "candidate supported only by strict title-stripped name equivalence"
    ),
}

strict_candidate_review_queue["review_priority"] = (
    strict_candidate_review_queue[
        "context_candidate_status"
    ].map(review_priority_by_status)
)

strict_candidate_review_queue["review_reason"] = (
    strict_candidate_review_queue[
        "context_candidate_status"
    ].map(review_reason_by_status)
)

# All pairs remain explicitly unresolved. These fields will later be replaced
# only through governed manual or authoritative verification.
strict_candidate_review_queue["identity_decision"] = "unresolved"
strict_candidate_review_queue["verification_status"] = "not_started"
strict_candidate_review_queue["verification_id"] = ""

# Add a stable notebook-local candidate identifier so each pair can be tracked
# through external research and later reference construction.
strict_candidate_review_queue = (
    strict_candidate_review_queue.sort_values(
        [
            "review_priority",
            "shared_horse_count",
            "shared_trainer_count",
            "combined_runner_rows",
            "strict_comparison_key",
        ],
        ascending=[True, False, False, False, True],
    )
    .reset_index(drop=True)
)

strict_candidate_review_queue.insert(
    0,
    "candidate_pair_id",
    [
        f"JOCKEY-STRICT-{candidate_number:04d}"
        for candidate_number in range(
            1,
            len(strict_candidate_review_queue) + 1,
        )
    ],
)

# Confirm that every original strict pair is represented exactly once.
assert len(strict_candidate_review_queue) == 216
assert (
    strict_candidate_review_queue[
        "candidate_pair_id"
    ].is_unique
)
assert (
    strict_candidate_review_queue[
        [
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
        ]
    ].duplicated().sum()
    == 0
)

strict_candidate_review_summary = (
    strict_candidate_review_queue.groupby(
        [
            "review_priority",
            "context_candidate_status",
            "review_reason",
        ],
        as_index=False,
    )
    .agg(
        candidate_pairs=("candidate_pair_id", "size"),
        pairs_with_period_overlap=(
            "temporal_relationship",
            lambda values: int(
                values.eq(
                    "observed_periods_overlap"
                ).sum()
            ),
        ),
        combined_runner_rows=(
            "combined_runner_rows",
            "sum",
        ),
        shared_horse_links=(
            "shared_horse_count",
            "sum",
        ),
    )
    .sort_values("review_priority")
    .reset_index(drop=True)
)

display(strict_candidate_review_summary)

display(
    strict_candidate_review_queue[
        [
            "candidate_pair_id",
            "review_priority",
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "pair_label_structure",
            "temporal_relationship",
            "gap_days",
            "overlap_days",
            "same_race_collisions",
            "shared_horse_count",
            "shared_trainer_count",
            "combined_runner_rows",
            "context_candidate_status",
            "identity_decision",
            "verification_status",
        ]
    ]
)

,review_priority,context_candidate_status,review_reason,candidate_pairs,pairs_with_period_overlap,combined_runner_rows,shared_horse_links
0,1,same_race_collision,labels occur on separate runners in the same p...,1,1,710,0
1,2,shared_horse,labels share one or more exact raw horse labels,114,23,36931,681
2,3,shared_trainer_only,labels share trainers but no exact raw horse l...,46,7,3348,0
3,4,no_shared_source_context,candidate supported only by strict title-strip...,55,5,1568,0


,candidate_pair_id,review_priority,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,pair_label_structure,temporal_relationship,gap_days,overlap_days,same_race_collisions,shared_horse_count,shared_trainer_count,combined_runner_rows,context_candidate_status,identity_decision,verification_status
0,JOCKEY-STRICT-0001,1,b oneill,Miss B ONeill,Mr B ONeill,two_title_bearing_labels,observed_periods_overlap,0,2222,1,0,0,710,same_race_collision,unresolved,not_started
1,JOCKEY-STRICT-0002,2,marie velon,Mlle Marie Velon,Mme Marie Velon,two_title_bearing_labels,observed_periods_separate,20,0,0,80,116,2449,shared_horse,unresolved,not_started
2,JOCKEY-STRICT-0003,2,frida valle skar,Mlle Frida Valle Skar,Mme Frida Valle Skar,two_title_bearing_labels,observed_periods_overlap,0,9,0,43,35,1470,shared_horse,unresolved,not_started
3,JOCKEY-STRICT-0004,2,perrine cheyer,Mlle Perrine Cheyer,Mme Perrine Cheyer,two_title_bearing_labels,observed_periods_separate,5,0,0,35,37,658,shared_horse,unresolved,not_started
4,JOCKEY-STRICT-0005,2,jack andrews,Jack Andrews,Mr Jack Andrews,title_bearing_and_untitled,observed_periods_overlap,0,3296,0,30,14,684,shared_horse,unresolved,not_started
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,JOCKEY-STRICT-0212,4,charlotta ericsson,Charlotta Ericsson,Miss Charlotta Ericsson,title_bearing_and_untitled,observed_periods_separate,185,0,0,0,0,2,no_shared_source_context,unresolved,not_started
212,JOCKEY-STRICT-0213,4,j osullivan,J OSullivan,Mr J OSullivan,title_bearing_and_untitled,observed_periods_separate,1256,0,0,0,0,2,no_shared_source_context,unresolved,not_started
213,JOCKEY-STRICT-0214,4,juliette benech,Mlle Juliette Benech,Mme Juliette Benech,two_title_bearing_labels,observed_periods_separate,1420,0,0,0,0,2,no_shared_source_context,unresolved,not_started
214,JOCKEY-STRICT-0215,4,lucas gueracague,Lucas Gueracague,Mr Lucas Gueracague,title_bearing_and_untitled,observed_periods_separate,282,0,0,0,0,2,no_shared_source_context,unresolved,not_started


### Persist the strict jockey-identity review queue

The strict candidate analysis has produced a finite review population of 216 jockey-label pairs:

* 1 same-race collision;
* 114 pairs sharing exact horse labels;
* 46 pairs sharing trainers but no exact horse labels;
* 55 pairs supported only by title-stripped label equivalence.

This population should not remain only in notebook memory. It will be persisted as a review queue at:

`data/processed/jockey_identity/jockey_strict_candidate_review_queue.csv`

The file is an analytical working output, not yet the final governed identity reference.

Each row represents one candidate relationship between two immutable raw jockey labels. It preserves:

* the candidate-generation method;
* source-internal supporting or contradictory evidence;
* chronology;
* review priority;
* unresolved identity and verification states;
* fields for later external evidence and decisions.

The raw labels remain unchanged. A comparison key or shared-horse pattern does not authorise a merge.

After external verification, confirmed relationships will be transferred into a separately validated participant-identity reference with stable entity identifiers and permanent provenance. Unresolved cases will remain explicitly unresolved rather than being guessed.

In [15]:
from pathlib import Path

# Persist the complete strict candidate population as a working review queue.
#
# This is written under data/processed rather than data/reference because the
# 216 relationships have not yet been externally verified. The file records
# the review workload and its source-internal evidence; it is not a completed
# participant-identity authority.
JOCKEY_IDENTITY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "jockey_identity"
)

JOCKEY_STRICT_REVIEW_QUEUE_PATH = (
    JOCKEY_IDENTITY_OUTPUT_DIR
    / "jockey_strict_candidate_review_queue.csv"
)

JOCKEY_IDENTITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Create the persisted schema explicitly rather than writing every temporary
# notebook column. The review fields begin unresolved and will be populated
# only when external evidence has actually been inspected.
jockey_strict_candidate_review_output = (
    strict_candidate_review_queue[
        [
            "candidate_pair_id",
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "pair_label_structure",
            "temporal_relationship",
            "gap_days",
            "overlap_days",
            "left_runner_rows",
            "right_runner_rows",
            "combined_runner_rows",
            "same_race_collisions",
            "shared_horse_count",
            "shared_horses",
            "shared_trainer_count",
            "shared_trainers",
            "context_candidate_status",
            "review_priority",
            "review_reason",
        ]
    ]
    .copy()
)

# Record how each pair entered the queue. At this stage all 216 rows were
# generated through exact case-folded equivalence after removal of only the
# observed leading title vocabulary.
jockey_strict_candidate_review_output[
    "candidate_generation_method"
] = "observed_leading_title_removed_exact_match"

# These fields form the manual-review state. They deliberately remain blank or
# unresolved until an external source has been opened and its provenance has
# been captured.
jockey_strict_candidate_review_output[
    "identity_relationship"
] = "unresolved"

jockey_strict_candidate_review_output[
    "verified_person_name"
] = ""

jockey_strict_candidate_review_output[
    "verification_status"
] = "not_started"

jockey_strict_candidate_review_output[
    "verification_id"
] = ""

jockey_strict_candidate_review_output[
    "evidence_type"
] = ""

jockey_strict_candidate_review_output[
    "evidence_locator"
] = ""

jockey_strict_candidate_review_output[
    "evidence_accessed_date"
] = ""

jockey_strict_candidate_review_output[
    "confidence"
] = ""

jockey_strict_candidate_review_output[
    "review_notes"
] = ""

jockey_strict_candidate_review_output[
    "database_action"
] = "preserve_raw_unresolved"

# Keep the schema order deliberate so evidence and final decisions sit beside
# the candidate evidence that produced the review requirement.
jockey_strict_candidate_review_output = (
    jockey_strict_candidate_review_output[
        [
            "candidate_pair_id",
            "candidate_generation_method",
            "strict_comparison_key",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "pair_label_structure",
            "temporal_relationship",
            "gap_days",
            "overlap_days",
            "left_runner_rows",
            "right_runner_rows",
            "combined_runner_rows",
            "same_race_collisions",
            "shared_horse_count",
            "shared_horses",
            "shared_trainer_count",
            "shared_trainers",
            "context_candidate_status",
            "review_priority",
            "review_reason",
            "identity_relationship",
            "verified_person_name",
            "verification_status",
            "verification_id",
            "evidence_type",
            "evidence_locator",
            "evidence_accessed_date",
            "confidence",
            "review_notes",
            "database_action",
        ]
    ]
)

# Validate the in-memory review population before persistence.
assert len(jockey_strict_candidate_review_output) == 216
assert jockey_strict_candidate_review_output[
    "candidate_pair_id"
].is_unique

assert (
    jockey_strict_candidate_review_output[
        "identity_relationship"
    ].eq("unresolved").all()
)

assert (
    jockey_strict_candidate_review_output[
        "verification_status"
    ].eq("not_started").all()
)

assert (
    jockey_strict_candidate_review_output[
        "database_action"
    ].eq("preserve_raw_unresolved").all()
)

# Write once, then reload from disk. Subsequent analysis should be able to
# reproduce the review population from the persisted artifact rather than
# relying only on the current notebook session.
jockey_strict_candidate_review_output.to_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    index=False,
)

reloaded_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

# Confirm that persistence preserved row count, schema, candidate identifiers
# and the intentionally unresolved decision state.
assert len(reloaded_jockey_strict_review_queue) == 216

assert (
    reloaded_jockey_strict_review_queue.columns.tolist()
    == jockey_strict_candidate_review_output.columns.tolist()
)

assert reloaded_jockey_strict_review_queue[
    "candidate_pair_id"
].is_unique

assert (
    reloaded_jockey_strict_review_queue[
        "identity_relationship"
    ].eq("unresolved").all()
)

assert (
    reloaded_jockey_strict_review_queue[
        "verification_status"
    ].eq("not_started").all()
)

persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                JOCKEY_STRICT_REVIEW_QUEUE_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_candidate_pairs": len(
                reloaded_jockey_strict_review_queue
            ),
            "unique_candidate_pair_ids": (
                reloaded_jockey_strict_review_queue[
                    "candidate_pair_id"
                ].nunique()
            ),
            "same_race_collision_pairs": int(
                reloaded_jockey_strict_review_queue[
                    "same_race_collisions"
                ].gt(0).sum()
            ),
            "shared_horse_pairs": int(
                reloaded_jockey_strict_review_queue[
                    "context_candidate_status"
                ].eq("shared_horse").sum()
            ),
            "unresolved_relationships": int(
                reloaded_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "verification_not_started": int(
                reloaded_jockey_strict_review_queue[
                    "verification_status"
                ].eq("not_started").sum()
            ),
        }
    ]
)

display(persistence_summary)

display(
    reloaded_jockey_strict_review_queue.head(10)
)

,output_path,persisted_candidate_pairs,unique_candidate_pair_ids,same_race_collision_pairs,shared_horse_pairs,unresolved_relationships,verification_not_started
0,data/processed/jockey_identity/jockey_strict_c...,216,216,1,114,216,216


,candidate_pair_id,candidate_generation_method,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,pair_label_structure,temporal_relationship,gap_days,overlap_days,left_runner_rows,...,identity_relationship,verified_person_name,verification_status,verification_id,evidence_type,evidence_locator,evidence_accessed_date,confidence,review_notes,database_action
0,JOCKEY-STRICT-0001,observed_leading_title_removed_exact_match,b oneill,Miss B ONeill,Mr B ONeill,two_title_bearing_labels,observed_periods_overlap,0,2222,10,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
1,JOCKEY-STRICT-0002,observed_leading_title_removed_exact_match,marie velon,Mlle Marie Velon,Mme Marie Velon,two_title_bearing_labels,observed_periods_separate,20,0,1428,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
2,JOCKEY-STRICT-0003,observed_leading_title_removed_exact_match,frida valle skar,Mlle Frida Valle Skar,Mme Frida Valle Skar,two_title_bearing_labels,observed_periods_overlap,0,9,797,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
3,JOCKEY-STRICT-0004,observed_leading_title_removed_exact_match,perrine cheyer,Mlle Perrine Cheyer,Mme Perrine Cheyer,two_title_bearing_labels,observed_periods_separate,5,0,416,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
4,JOCKEY-STRICT-0005,observed_leading_title_removed_exact_match,jack andrews,Jack Andrews,Mr Jack Andrews,title_bearing_and_untitled,observed_periods_overlap,0,3296,523,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
5,JOCKEY-STRICT-0006,observed_leading_title_removed_exact_match,ambre molins,Mlle Ambre Molins,Mme Ambre Molins,two_title_bearing_labels,observed_periods_separate,14,0,490,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
6,JOCKEY-STRICT-0007,observed_leading_title_removed_exact_match,paddy hanlon,Mr Paddy Hanlon,Paddy Hanlon,title_bearing_and_untitled,observed_periods_separate,13,0,49,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
7,JOCKEY-STRICT-0008,observed_leading_title_removed_exact_match,coralie pacaut,Mlle Coralie Pacaut,Mme Coralie Pacaut,two_title_bearing_labels,observed_periods_separate,68,0,1571,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
8,JOCKEY-STRICT-0009,observed_leading_title_removed_exact_match,tristan durrell,Mr Tristan Durrell,Tristan Durrell,title_bearing_and_untitled,observed_periods_overlap,0,1155,114,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved
9,JOCKEY-STRICT-0010,observed_leading_title_removed_exact_match,alice stevens,Alice Stevens,Miss Alice Stevens,title_bearing_and_untitled,observed_periods_separate,14,0,338,...,unresolved,,not_started,,,,,,,preserve_raw_unresolved


### Refine the review schema before recording decisions

The persisted review queue requires one schema correction before external decisions are entered.

A single `verified_person_name` field is insufficient for relationships classified as `different_people`, because each raw label may resolve to a different named individual. The review artifact will therefore store:

* `left_verified_person_name`;
* `right_verified_person_name`.

Either field may remain blank where the relationship is established but the person's complete name has not been verified.

The relationship vocabulary is:

* `same_person`;
* `different_people`;
* `unresolved`.

A `different_people` decision does not require both complete names where separate identity is independently established, such as two labels occurring on different runners in the same race. It does require the evidence and reasoning to be preserved.

The first candidate, `Miss B ONeill` versus `Mr B ONeill`, is therefore resolvable as `different_people`. Published results independently list both labels on separate horses in the same Naas race on 11 November 2017. The complete name represented by `Miss B ONeill` remains unresolved and will not be invented.

In [16]:
# Refine the persisted review schema before recording external decisions.
#
# A single verified_person_name field cannot safely represent a split
# relationship. Replace it with one verified-name field for each raw label.
reloaded_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

verified_person_name_position = (
    reloaded_jockey_strict_review_queue.columns.get_loc(
        "verified_person_name"
    )
)

reloaded_jockey_strict_review_queue = (
    reloaded_jockey_strict_review_queue.drop(
        columns="verified_person_name"
    )
)

reloaded_jockey_strict_review_queue.insert(
    verified_person_name_position,
    "left_verified_person_name",
    "",
)

reloaded_jockey_strict_review_queue.insert(
    verified_person_name_position + 1,
    "right_verified_person_name",
    "",
)

# Record the first externally supported relationship.
#
# The published result confirms that both labels occur on different runners
# in the same race. This establishes separate people even though the complete
# name represented by the female rider's source label remains unresolved.
b_oneill_mask = reloaded_jockey_strict_review_queue[
    "candidate_pair_id"
].eq("JOCKEY-STRICT-0001")

assert b_oneill_mask.sum() == 1

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "identity_relationship",
] = "different_people"

# Barry O'Neill is retained only on the right-hand label. The left-hand full
# name remains blank because it has not yet been established sufficiently.
reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "left_verified_person_name",
] = ""

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "right_verified_person_name",
] = "Barry O'Neill"

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "verification_status",
] = "confirmed"

# Use a Notebook 22-specific permanent verification identifier. This identifier
# will later be added to the governed manual-verification register or replaced
# by an equivalent specialist participant-identity reference.
reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "verification_id",
] = "NB22-JOCKEY-0001"

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "evidence_type",
] = "published_result"

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "evidence_locator",
] = (
    "https://www.racingpost.com/results/192/naas/"
    "2017-11-11/688901"
)

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "evidence_accessed_date",
] = "2026-08-04"

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "confidence",
] = "high"

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "review_notes",
] = (
    "The published 4:05 Naas result on 2017-11-11 lists "
    "Miss B O'Neill riding Hawthorn Echo and Mr B O'Neill "
    "riding Chisholm Trail in the same 13-runner race. The "
    "labels therefore represent different people. The complete "
    "name of Miss B O'Neill remains unresolved and must not be "
    "inferred from the initial alone."
)

reloaded_jockey_strict_review_queue.loc[
    b_oneill_mask,
    "database_action",
] = "preserve_separate_participant_identities"

# Validate the revised state before replacing the persisted working artifact.
assert (
    reloaded_jockey_strict_review_queue.loc[
        b_oneill_mask,
        "identity_relationship",
    ].iloc[0]
    == "different_people"
)

assert (
    reloaded_jockey_strict_review_queue.loc[
        b_oneill_mask,
        "verification_status",
    ].iloc[0]
    == "confirmed"
)

assert (
    reloaded_jockey_strict_review_queue.loc[
        b_oneill_mask,
        "left_verified_person_name",
    ].iloc[0]
    == ""
)

assert (
    reloaded_jockey_strict_review_queue.loc[
        b_oneill_mask,
        "right_verified_person_name",
    ].iloc[0]
    == "Barry O'Neill"
)

assert (
    reloaded_jockey_strict_review_queue[
        "candidate_pair_id"
    ].is_unique
)

assert len(reloaded_jockey_strict_review_queue) == 216

# Replace the working review artifact, reload it and verify that the first
# decision and all 215 untouched unresolved candidates survive persistence.
reloaded_jockey_strict_review_queue.to_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    index=False,
)

verified_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

verification_progress_summary = pd.DataFrame(
    [
        {
            "candidate_pairs": len(
                verified_jockey_strict_review_queue
            ),
            "confirmed_same_person": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("same_person").sum()
            ),
            "confirmed_different_people": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("different_people").sum()
            ),
            "unresolved_relationships": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "confirmed_verifications": int(
                verified_jockey_strict_review_queue[
                    "verification_status"
                ].eq("confirmed").sum()
            ),
        }
    ]
)

display(verification_progress_summary)

display(
    verified_jockey_strict_review_queue.loc[
        verified_jockey_strict_review_queue[
            "candidate_pair_id"
        ].eq("JOCKEY-STRICT-0001")
    ]
)

,candidate_pairs,confirmed_same_person,confirmed_different_people,unresolved_relationships,confirmed_verifications
0,216,0,1,215,1


,candidate_pair_id,candidate_generation_method,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,pair_label_structure,temporal_relationship,gap_days,overlap_days,left_runner_rows,...,left_verified_person_name,right_verified_person_name,verification_status,verification_id,evidence_type,evidence_locator,evidence_accessed_date,confidence,review_notes,database_action
0,JOCKEY-STRICT-0001,observed_leading_title_removed_exact_match,b oneill,Miss B ONeill,Mr B ONeill,two_title_bearing_labels,observed_periods_overlap,0,2222,10,...,,Barry O'Neill,confirmed,NB22-JOCKEY-0001,published_result,https://www.racingpost.com/results/192/naas/20...,2026-08-04,high,The published 4:05 Naas result on 2017-11-11 l...,preserve_separate_participant_identities


### Verified alias: Marie Velon

`Mlle Marie Velon` and `Mme Marie Velon` are confirmed as two source-label presentations of the same jockey, Marie Vélon.

Source-internal evidence includes:

* 80 shared exact horse labels;
* 116 shared trainer labels;
* 2,449 combined runner rows;
* no same-race collision;
* only 20 days between the source-observed label periods.

External evidence provides a stable real-world identity:

* France Galop records one jockey, Marie Vélon, with a continuous professional career;
* Racing Post assigns Marie Velon one jockey profile, identifier `95747`;
* historical and current results associated with that profile use the presentation `Mme Marie Velon`.

The external result presentation does not support treating `Mlle` and `Mme` as reliable chronological status markers. Historical races may currently be displayed with a later or standardised title.

Decision:

* relationship: `same_person`;
* verified name: `Marie Vélon`;
* both immutable raw labels remain preserved;
* both labels may map to one future jockey participant identity;
* the title itself must not be used as an effective-dated identity attribute without separate evidence.

In [17]:
# Record the second externally verified strict relationship.
#
# France Galop records one continuous jockey identity for Marie Vélon, while
# Racing Post uses the single jockey profile identifier 95747. Together with
# the extensive shared-horse evidence, this supports a same-person decision.
#
# The source titles remain preserved exactly. The decision does not interpret
# Mlle-to-Mme as a verified marital or chronological transition.

verified_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

marie_velon_mask = verified_jockey_strict_review_queue[
    "candidate_pair_id"
].eq("JOCKEY-STRICT-0002")

assert marie_velon_mask.sum() == 1

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "identity_relationship",
] = "same_person"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "left_verified_person_name",
] = "Marie Vélon"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "right_verified_person_name",
] = "Marie Vélon"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "verification_status",
] = "confirmed"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "verification_id",
] = "NB22-JOCKEY-0002"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "evidence_type",
] = "governing_body_profile; published_jockey_profile"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "evidence_locator",
] = (
    "https://www.france-galop.com/fr/content/"
    "la-jockey-marie-velon-signe-une-annee-2022-exceptionnelle"
    "; "
    "https://www.racingpost.com/profile/jockey/95747/"
    "mme-marie-velon"
)

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "evidence_accessed_date",
] = "2026-08-04"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "confidence",
] = "high"

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "review_notes",
] = (
    "France Galop records one jockey, Marie Vélon, with a "
    "continuous professional career. Racing Post assigns Marie "
    "Velon the single jockey profile identifier 95747. The two "
    "source labels also share 80 exact horse labels and 116 "
    "trainer labels and never occur together in one provisional "
    "race. Treat both raw labels as presentations of the same "
    "person. Do not interpret Mlle and Mme as reliable "
    "effective-dated status values because published historical "
    "results may display a standardised or later title."
)

verified_jockey_strict_review_queue.loc[
    marie_velon_mask,
    "database_action",
] = "map_both_labels_to_same_participant_identity"

# Validate the completed relationship before replacing the persisted file.
verified_marie_velon_row = (
    verified_jockey_strict_review_queue.loc[
        marie_velon_mask
    ].iloc[0]
)

assert (
    verified_marie_velon_row["identity_relationship"]
    == "same_person"
)

assert (
    verified_marie_velon_row["left_verified_person_name"]
    == "Marie Vélon"
)

assert (
    verified_marie_velon_row["right_verified_person_name"]
    == "Marie Vélon"
)

assert (
    verified_marie_velon_row["verification_status"]
    == "confirmed"
)

assert (
    verified_marie_velon_row["same_race_collisions"]
    == 0
)

assert (
    verified_marie_velon_row["shared_horse_count"]
    == 80
)

# Persist and reload the working review artifact after the decision.
verified_jockey_strict_review_queue.to_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    index=False,
)

verified_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

verification_progress_summary = pd.DataFrame(
    [
        {
            "candidate_pairs": len(
                verified_jockey_strict_review_queue
            ),
            "confirmed_same_person": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("same_person").sum()
            ),
            "confirmed_different_people": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("different_people").sum()
            ),
            "unresolved_relationships": int(
                verified_jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "confirmed_verifications": int(
                verified_jockey_strict_review_queue[
                    "verification_status"
                ].eq("confirmed").sum()
            ),
        }
    ]
)

display(verification_progress_summary)

display(
    verified_jockey_strict_review_queue.loc[
        verified_jockey_strict_review_queue[
            "candidate_pair_id"
        ].isin(
            [
                "JOCKEY-STRICT-0001",
                "JOCKEY-STRICT-0002",
            ]
        )
    ]
)

,candidate_pairs,confirmed_same_person,confirmed_different_people,unresolved_relationships,confirmed_verifications
0,216,1,1,214,2


,candidate_pair_id,candidate_generation_method,strict_comparison_key,left_raw_jockey_label,right_raw_jockey_label,pair_label_structure,temporal_relationship,gap_days,overlap_days,left_runner_rows,...,left_verified_person_name,right_verified_person_name,verification_status,verification_id,evidence_type,evidence_locator,evidence_accessed_date,confidence,review_notes,database_action
0,JOCKEY-STRICT-0001,observed_leading_title_removed_exact_match,b oneill,Miss B ONeill,Mr B ONeill,two_title_bearing_labels,observed_periods_overlap,0,2222,10,...,,Barry O'Neill,confirmed,NB22-JOCKEY-0001,published_result,https://www.racingpost.com/results/192/naas/20...,2026-08-04,high,The published 4:05 Naas result on 2017-11-11 l...,preserve_separate_participant_identities
1,JOCKEY-STRICT-0002,observed_leading_title_removed_exact_match,marie velon,Mlle Marie Velon,Mme Marie Velon,two_title_bearing_labels,observed_periods_separate,20,0,1428,...,Marie Vélon,Marie Vélon,confirmed,NB22-JOCKEY-0002,governing_body_profile; published_jockey_profile,https://www.france-galop.com/fr/content/la-joc...,2026-08-04,high,"France Galop records one jockey, Marie Vélon, ...",map_both_labels_to_same_participant_identity


### Prepare the first batch of unresolved jockey relationships

Two strict candidate relationships have now been verified:

* `JOCKEY-STRICT-0001` — different people;
* `JOCKEY-STRICT-0002` — same person.

The remaining 214 relationships will not be handled through one bespoke notebook cell per pair.

Instead, external verification will use bounded review batches. The first batch contains the 25 highest-ranked unresolved pairs supported by shared exact horse labels.

The batch file is a temporary research worksheet. It preserves the permanent candidate identifiers from the complete review queue and provides empty fields for evidence and decisions.

After the batch has been researched:

1. completed decisions will be validated;
2. the matching rows in the complete 216-row review queue will be updated;
3. the batch file will remain as review provenance;
4. unresolved members will remain unresolved rather than being forced into a decision.

In [19]:
# Prepare the first reusable external-verification batch rather than creating
# one bespoke notebook cell for every candidate pair.

JOCKEY_VERIFICATION_BATCH_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "jockey_identity"
    / "verification_batches"
)

JOCKEY_VERIFICATION_BATCH_01_PATH = (
    JOCKEY_VERIFICATION_BATCH_DIR
    / "jockey_strict_verification_batch_01.csv"
)

JOCKEY_VERIFICATION_BATCH_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

verified_jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

# Select the 25 strongest unresolved shared-horse candidates.
#
# Existing review order already ranks candidates by:
# - source-evidence category;
# - shared-horse count;
# - shared-trainer count;
# - combined runner volume.
jockey_verification_batch_01 = (
    verified_jockey_strict_review_queue.loc[
        verified_jockey_strict_review_queue[
            "verification_status"
        ].eq("not_started")
        & verified_jockey_strict_review_queue[
            "context_candidate_status"
        ].eq("shared_horse")
    ]
    .head(25)
    .copy()
    .reset_index(drop=True)
)

# Keep only the evidence needed for efficient external review.
jockey_verification_batch_01 = (
    jockey_verification_batch_01[
        [
            "candidate_pair_id",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "temporal_relationship",
            "gap_days",
            "overlap_days",
            "left_runner_rows",
            "right_runner_rows",
            "combined_runner_rows",
            "shared_horse_count",
            "shared_horses",
            "shared_trainer_count",
            "shared_trainers",
        ]
    ]
)

# Add explicit review-result fields. These should be populated only from
# inspected external evidence, not inferred automatically from source overlap.
jockey_verification_batch_01[
    "identity_relationship"
] = "unresolved"

jockey_verification_batch_01[
    "left_verified_person_name"
] = ""

jockey_verification_batch_01[
    "right_verified_person_name"
] = ""

jockey_verification_batch_01[
    "verification_status"
] = "not_started"

jockey_verification_batch_01[
    "evidence_type"
] = ""

jockey_verification_batch_01[
    "evidence_locator"
] = ""

jockey_verification_batch_01[
    "evidence_accessed_date"
] = ""

jockey_verification_batch_01[
    "confidence"
] = ""

jockey_verification_batch_01[
    "review_notes"
] = ""

jockey_verification_batch_01[
    "database_action"
] = "preserve_raw_unresolved"

# Validate the bounded review population before persistence.
assert len(jockey_verification_batch_01) == 25
assert jockey_verification_batch_01[
    "candidate_pair_id"
].is_unique

assert jockey_verification_batch_01[
    "shared_horse_count"
].gt(0).all()

assert jockey_verification_batch_01[
    "verification_status"
].eq("not_started").all()

# Persist and reload the batch worksheet.
jockey_verification_batch_01.to_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    index=False,
)

reloaded_jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

assert len(reloaded_jockey_verification_batch_01) == 25

assert (
    reloaded_jockey_verification_batch_01.columns.tolist()
    == jockey_verification_batch_01.columns.tolist()
)

assert reloaded_jockey_verification_batch_01[
    "candidate_pair_id"
].is_unique

batch_01_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                JOCKEY_VERIFICATION_BATCH_01_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "candidate_pairs": len(
                reloaded_jockey_verification_batch_01
            ),
            "total_shared_horse_links": int(
                reloaded_jockey_verification_batch_01[
                    "shared_horse_count"
                ].sum()
            ),
            "combined_runner_rows": int(
                reloaded_jockey_verification_batch_01[
                    "combined_runner_rows"
                ].sum()
            ),
            "verification_not_started": int(
                reloaded_jockey_verification_batch_01[
                    "verification_status"
                ].eq("not_started").sum()
            ),
        }
    ]
)

display(batch_01_summary)

display(
    reloaded_jockey_verification_batch_01[
        [
            "candidate_pair_id",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "temporal_relationship",
            "shared_horse_count",
            "shared_trainer_count",
            "combined_runner_rows",
            "identity_relationship",
            "verification_status",
        ]
    ]
)

,output_path,candidate_pairs,total_shared_horse_links,combined_runner_rows,verification_not_started
0,data/processed/jockey_identity/verification_ba...,25,387,16221,25


,candidate_pair_id,left_raw_jockey_label,right_raw_jockey_label,temporal_relationship,shared_horse_count,shared_trainer_count,combined_runner_rows,identity_relationship,verification_status
0,JOCKEY-STRICT-0003,Mlle Frida Valle Skar,Mme Frida Valle Skar,observed_periods_overlap,43,35,1470,unresolved,not_started
1,JOCKEY-STRICT-0004,Mlle Perrine Cheyer,Mme Perrine Cheyer,observed_periods_separate,35,37,658,unresolved,not_started
2,JOCKEY-STRICT-0005,Jack Andrews,Mr Jack Andrews,observed_periods_overlap,30,14,684,unresolved,not_started
3,JOCKEY-STRICT-0006,Mlle Ambre Molins,Mme Ambre Molins,observed_periods_separate,26,30,910,unresolved,not_started
4,JOCKEY-STRICT-0007,Mr Paddy Hanlon,Paddy Hanlon,observed_periods_separate,22,3,593,unresolved,not_started
5,JOCKEY-STRICT-0008,Mlle Coralie Pacaut,Mme Coralie Pacaut,observed_periods_separate,18,42,1897,unresolved,not_started
6,JOCKEY-STRICT-0009,Mr Tristan Durrell,Tristan Durrell,observed_periods_overlap,17,8,862,unresolved,not_started
7,JOCKEY-STRICT-0010,Alice Stevens,Miss Alice Stevens,observed_periods_separate,16,16,506,unresolved,not_started
8,JOCKEY-STRICT-0011,Mlle Maryline Eon,Mme Maryline Eon,observed_periods_separate,15,31,800,unresolved,not_started
9,JOCKEY-STRICT-0012,Freddie Gordon,Mr Freddie Gordon,observed_periods_separate,15,1,489,unresolved,not_started


### Authority-first external verification schema

The 25-pair verification batch will use the relevant racing authority as its primary identity source wherever public evidence is available.

Authority evidence differs by jurisdiction:

* the BHA and HRI may provide a public participant profile or stable identifier;
* France Galop may publish an official name in results, rankings, articles or licensing material without exposing a public participant identifier;
* some historical or lower-profile riders may not have a sufficiently complete public authority record.

The review schema must therefore permit an authority-confirmed name without requiring an authority participant ID.

For each candidate pair, the review will record separately:

* the governing authority;
* the authority-published name;
* any public authority participant identifier;
* the authority evidence type and locator;
* any secondary provider identifier used as corroboration;
* the final relationship decision.

The evidence hierarchy is:

1. public governing-authority identity evidence;
2. authority-published rides, results or licensing records;
3. stable secondary participant identifiers;
4. source-internal shared horses, trainers and chronology.

Secondary providers may corroborate an identity but do not replace authority evidence where authority evidence is publicly available.

No candidate is automatically merged because the two labels differ only by a title. Where public evidence remains insufficient, the relationship remains unresolved.

In [20]:
# Refine the first verification batch around authority-first identity evidence.
#
# Public authority systems are not uniform. An authority may expose:
# - a registered name and stable participant identifier;
# - a published name without a public identifier;
# - only result, ranking or licensing evidence;
# - no sufficient public evidence.
#
# The schema therefore keeps authority name, identifier and evidence method
# separate and allows identifiers to remain blank.

jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

authority_review_columns = {
    "governing_jurisdiction": "",
    "governing_authority": "",
    "authority_registered_name": "",
    "authority_participant_id": "",
    "authority_evidence_type": "",
    "authority_evidence_locator": "",
    "authority_evidence_accessed_date": "",
    "authority_lookup_status": "not_checked",
    "secondary_provider": "",
    "secondary_participant_id": "",
    "secondary_display_name": "",
    "secondary_evidence_locator": "",
    "secondary_lookup_status": "not_checked",
}

for column_name, default_value in authority_review_columns.items():
    if column_name not in jockey_verification_batch_01.columns:
        jockey_verification_batch_01[column_name] = default_value

# Reorder the worksheet so authority evidence sits immediately before the
# final decision fields. Existing candidate evidence and prior decisions remain
# unchanged.
decision_columns = [
    "identity_relationship",
    "left_verified_person_name",
    "right_verified_person_name",
    "verification_status",
    "evidence_type",
    "evidence_locator",
    "evidence_accessed_date",
    "confidence",
    "review_notes",
    "database_action",
]

authority_columns = list(authority_review_columns)

candidate_columns = [
    column_name
    for column_name in jockey_verification_batch_01.columns
    if column_name not in authority_columns + decision_columns
]

jockey_verification_batch_01 = jockey_verification_batch_01[
    candidate_columns
    + authority_columns
    + decision_columns
]

# Controlled vocabularies describe the state of the external lookup, not the
# final participant relationship.
allowed_authority_lookup_statuses = {
    "not_checked",
    "profile_confirmed",
    "published_name_confirmed",
    "result_or_ride_confirmed",
    "no_public_record_found",
    "insufficient_or_conflicting",
}

allowed_secondary_lookup_statuses = {
    "not_checked",
    "profile_confirmed",
    "identifier_confirmed",
    "no_public_record_found",
    "insufficient_or_conflicting",
}

assert len(jockey_verification_batch_01) == 25
assert jockey_verification_batch_01[
    "candidate_pair_id"
].is_unique

assert set(
    jockey_verification_batch_01[
        "authority_lookup_status"
    ].unique()
).issubset(allowed_authority_lookup_statuses)

assert set(
    jockey_verification_batch_01[
        "secondary_lookup_status"
    ].unique()
).issubset(allowed_secondary_lookup_statuses)

# No authority lookup has yet been performed for this batch, so the identity
# decisions must remain unchanged.
assert jockey_verification_batch_01[
    "authority_lookup_status"
].eq("not_checked").all()

assert jockey_verification_batch_01[
    "identity_relationship"
].eq("unresolved").all()

# Persist and reload the refined worksheet.
jockey_verification_batch_01.to_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    index=False,
)

reloaded_jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

assert len(reloaded_jockey_verification_batch_01) == 25

assert (
    reloaded_jockey_verification_batch_01.columns.tolist()
    == jockey_verification_batch_01.columns.tolist()
)

authority_schema_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                JOCKEY_VERIFICATION_BATCH_01_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "candidate_pairs": len(
                reloaded_jockey_verification_batch_01
            ),
            "authority_not_checked": int(
                reloaded_jockey_verification_batch_01[
                    "authority_lookup_status"
                ].eq("not_checked").sum()
            ),
            "secondary_not_checked": int(
                reloaded_jockey_verification_batch_01[
                    "secondary_lookup_status"
                ].eq("not_checked").sum()
            ),
            "unresolved_relationships": int(
                reloaded_jockey_verification_batch_01[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "authority_schema_columns": len(
                authority_columns
            ),
        }
    ]
)

display(authority_schema_summary)

display(
    reloaded_jockey_verification_batch_01[
        [
            "candidate_pair_id",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "governing_jurisdiction",
            "governing_authority",
            "authority_registered_name",
            "authority_participant_id",
            "authority_lookup_status",
            "secondary_provider",
            "secondary_participant_id",
            "secondary_lookup_status",
            "identity_relationship",
        ]
    ]
)

,output_path,candidate_pairs,authority_not_checked,secondary_not_checked,unresolved_relationships,authority_schema_columns
0,data/processed/jockey_identity/verification_ba...,25,25,25,25,13


,candidate_pair_id,left_raw_jockey_label,right_raw_jockey_label,governing_jurisdiction,governing_authority,authority_registered_name,authority_participant_id,authority_lookup_status,secondary_provider,secondary_participant_id,secondary_lookup_status,identity_relationship
0,JOCKEY-STRICT-0003,Mlle Frida Valle Skar,Mme Frida Valle Skar,,,,,not_checked,,,not_checked,unresolved
1,JOCKEY-STRICT-0004,Mlle Perrine Cheyer,Mme Perrine Cheyer,,,,,not_checked,,,not_checked,unresolved
2,JOCKEY-STRICT-0005,Jack Andrews,Mr Jack Andrews,,,,,not_checked,,,not_checked,unresolved
3,JOCKEY-STRICT-0006,Mlle Ambre Molins,Mme Ambre Molins,,,,,not_checked,,,not_checked,unresolved
4,JOCKEY-STRICT-0007,Mr Paddy Hanlon,Paddy Hanlon,,,,,not_checked,,,not_checked,unresolved
5,JOCKEY-STRICT-0008,Mlle Coralie Pacaut,Mme Coralie Pacaut,,,,,not_checked,,,not_checked,unresolved
6,JOCKEY-STRICT-0009,Mr Tristan Durrell,Tristan Durrell,,,,,not_checked,,,not_checked,unresolved
7,JOCKEY-STRICT-0010,Alice Stevens,Miss Alice Stevens,,,,,not_checked,,,not_checked,unresolved
8,JOCKEY-STRICT-0011,Mlle Maryline Eon,Mme Maryline Eon,,,,,not_checked,,,not_checked,unresolved
9,JOCKEY-STRICT-0012,Freddie Gordon,Mr Freddie Gordon,,,,,not_checked,,,not_checked,unresolved


### Establish jurisdiction from source-race context

The governing authority must not be assigned from a jockey’s name, title or apparent nationality.

A rider may compete in several jurisdictions, while the two raw labels may come from different feeds or countries. The first verification step is therefore to profile the courses attached to each label in the 25-pair batch.

For every candidate pair, this stage records:

* the distinct courses associated with the left label;
* the distinct courses associated with the right label;
* courses shared by both labels;
* the most frequent courses for each label;
* whether the two labels appear to occupy the same source-racing environment.

This remains source-internal evidence. It does not yet determine the licensing authority or resolve identity.

The course evidence will be inspected before assigning `governing_jurisdiction` and `governing_authority`.

In [21]:
# Profile the source-course context for every raw label in verification batch 01.
#
# Jurisdiction and authority must follow the rider's actual source-race context,
# not assumptions based on names or honorifics.

jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

batch_01_raw_labels = sorted(
    set(
        jockey_verification_batch_01[
            "left_raw_jockey_label"
        ]
    )
    | set(
        jockey_verification_batch_01[
            "right_raw_jockey_label"
        ]
    )
)

# Use the source occurrences already loaded for strict candidate analysis.
# Preserve raw course labels exactly as presented by the source.
batch_01_jockey_course_occurrences = (
    jockey_strict_candidate_occurrences.loc[
        jockey_strict_candidate_occurrences[
            "raw_jockey_label"
        ].isin(batch_01_raw_labels),
        [
            "raw_jockey_label",
            "course",
        ],
    ]
    .dropna(subset=["course"])
    .copy()
)

batch_01_jockey_course_frequency = (
    batch_01_jockey_course_occurrences.groupby(
        [
            "raw_jockey_label",
            "course",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "runner_rows"})
    .sort_values(
        [
            "raw_jockey_label",
            "runner_rows",
            "course",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

def summarise_label_courses(
    raw_jockey_label,
    *,
    top_n=8,
):
    """Return exact course-set and compact frequency evidence for one label."""

    label_course_rows = (
        batch_01_jockey_course_frequency.loc[
            batch_01_jockey_course_frequency[
                "raw_jockey_label"
            ].eq(raw_jockey_label)
        ]
        .copy()
    )

    course_set = set(label_course_rows["course"])

    top_course_text = " | ".join(
        (
            label_course_rows.head(top_n)["course"]
            + " ("
            + label_course_rows.head(top_n)[
                "runner_rows"
            ].astype(str)
            + ")"
        )
    )

    return {
        "course_set": course_set,
        "distinct_course_count": len(course_set),
        "top_courses": top_course_text,
    }


pair_course_context_rows = []

for pair_record in jockey_verification_batch_01.to_dict(
    "records"
):
    left_label = pair_record["left_raw_jockey_label"]
    right_label = pair_record["right_raw_jockey_label"]

    left_context = summarise_label_courses(left_label)
    right_context = summarise_label_courses(right_label)

    shared_courses = sorted(
        left_context["course_set"]
        & right_context["course_set"]
    )

    all_courses = sorted(
        left_context["course_set"]
        | right_context["course_set"]
    )

    if (
        left_context["course_set"]
        == right_context["course_set"]
    ):
        source_course_relationship = "identical_course_sets"
    elif shared_courses:
        source_course_relationship = "partially_shared_courses"
    else:
        source_course_relationship = "no_shared_courses"

    pair_course_context_rows.append(
        {
            "candidate_pair_id": pair_record[
                "candidate_pair_id"
            ],
            "left_raw_jockey_label": left_label,
            "right_raw_jockey_label": right_label,
            "left_distinct_courses": left_context[
                "distinct_course_count"
            ],
            "right_distinct_courses": right_context[
                "distinct_course_count"
            ],
            "shared_course_count": len(shared_courses),
            "combined_distinct_courses": len(all_courses),
            "source_course_relationship": (
                source_course_relationship
            ),
            "left_top_courses": left_context[
                "top_courses"
            ],
            "right_top_courses": right_context[
                "top_courses"
            ],
            "shared_courses": " | ".join(
                shared_courses
            ),
        }
    )

batch_01_pair_course_context = pd.DataFrame(
    pair_course_context_rows
)

assert len(batch_01_pair_course_context) == 25
assert batch_01_pair_course_context[
    "candidate_pair_id"
].is_unique

# Merge the source-course evidence into the persisted verification worksheet.
course_context_columns = [
    "left_distinct_courses",
    "right_distinct_courses",
    "shared_course_count",
    "combined_distinct_courses",
    "source_course_relationship",
    "left_top_courses",
    "right_top_courses",
    "shared_courses",
]

jockey_verification_batch_01 = (
    jockey_verification_batch_01.drop(
        columns=[
            column_name
            for column_name in course_context_columns
            if column_name
            in jockey_verification_batch_01.columns
        ],
        errors="ignore",
    )
    .merge(
        batch_01_pair_course_context[
            ["candidate_pair_id"]
            + course_context_columns
        ],
        how="left",
        on="candidate_pair_id",
        validate="one_to_one",
    )
)

assert jockey_verification_batch_01[
    "source_course_relationship"
].notna().all()

# Persist and reload the enriched batch worksheet.
jockey_verification_batch_01.to_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    index=False,
)

reloaded_jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

course_context_summary = (
    reloaded_jockey_verification_batch_01.groupby(
        "source_course_relationship",
        as_index=False,
    )
    .agg(
        candidate_pairs=("candidate_pair_id", "size"),
        shared_course_links=(
            "shared_course_count",
            "sum",
        ),
    )
    .sort_values(
        [
            "candidate_pairs",
            "source_course_relationship",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

display(course_context_summary)

display(
    reloaded_jockey_verification_batch_01[
        [
            "candidate_pair_id",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "left_distinct_courses",
            "right_distinct_courses",
            "shared_course_count",
            "source_course_relationship",
            "left_top_courses",
            "right_top_courses",
        ]
    ]
)

,source_course_relationship,candidate_pairs,shared_course_links
0,partially_shared_courses,25,391


,candidate_pair_id,left_raw_jockey_label,right_raw_jockey_label,left_distinct_courses,right_distinct_courses,shared_course_count,source_course_relationship,left_top_courses,right_top_courses
0,JOCKEY-STRICT-0003,Mlle Frida Valle Skar,Mme Frida Valle Skar,16,15,7,partially_shared_courses,Chantilly (FR) (291) | Deauville (FR) (222) | ...,Chantilly (FR) (175) | Deauville (FR) (120) | ...
1,JOCKEY-STRICT-0004,Mlle Perrine Cheyer,Mme Perrine Cheyer,12,11,5,partially_shared_courses,Chantilly (FR) (167) | Deauville (FR) (121) | ...,Chantilly (FR) (74) | Deauville (FR) (66) | Ch...
2,JOCKEY-STRICT-0005,Jack Andrews,Mr Jack Andrews,42,33,32,partially_shared_courses,Huntingdon (51) | Southwell (44) | Fakenham (4...,Cheltenham (20) | Warwick (15) | Market Rasen ...
3,JOCKEY-STRICT-0006,Mlle Ambre Molins,Mme Ambre Molins,16,13,5,partially_shared_courses,Deauville (FR) (157) | Chantilly (FR) (155) | ...,Deauville (FR) (101) | Chantilly (FR) (79) | D...
4,JOCKEY-STRICT-0007,Mr Paddy Hanlon,Paddy Hanlon,32,58,28,partially_shared_courses,Galway (IRE) (6) | Ayr (3) | Listowel (IRE) (3...,Downpatrick (IRE) (41) | Kilbeggan (IRE) (35) ...
5,JOCKEY-STRICT-0008,Mlle Coralie Pacaut,Mme Coralie Pacaut,26,15,7,partially_shared_courses,Deauville (FR) (477) | Chantilly (FR) (450) | ...,Deauville (FR) (106) | Chantilly (FR) (63) | S...
6,JOCKEY-STRICT-0009,Mr Tristan Durrell,Tristan Durrell,33,41,31,partially_shared_courses,Fakenham (9) | Uttoxeter (9) | Cheltenham (7) ...,Warwick (53) | Uttoxeter (42) | Doncaster (32)...
7,JOCKEY-STRICT-0010,Alice Stevens,Miss Alice Stevens,42,51,32,partially_shared_courses,Market Rasen (31) | Wetherby (22) | Doncaster ...,Ludlow (17) | Cheltenham (14) | Stratford (14)...
8,JOCKEY-STRICT-0011,Mlle Maryline Eon,Mme Maryline Eon,25,5,4,partially_shared_courses,Deauville (FR) (218) | Chantilly (FR) (117) | ...,Deauville (FR) (115) | Chantilly (FR) (85) | S...
9,JOCKEY-STRICT-0012,Freddie Gordon,Mr Freddie Gordon,34,33,23,partially_shared_courses,Plumpton (75) | Fontwell (52) | Newbury (34) |...,Fontwell (15) | Plumpton (13) | Cheltenham (6)...


### Proportionate stopping rule for strict jockey candidates

Strict title removal identified 216 candidate relationships between raw jockey labels. It did not establish 216 real-world identity relationships.

The investigation confirmed:

* one same-race collision that must remain split;
* one externally supported same-person relationship;
* 214 relationships that remain unresolved;
* shared horses, trainers, courses and active periods can strengthen a candidate but cannot prove identity;
* comprehensive historical verification would require disproportionate manual reconstruction across several racing authorities and publication systems.

The database exists to support analysis and writing. It is not intended to become a complete international jockey registry.

The stopping rule is therefore:

1. preserve every immutable raw jockey label and its source-row lineage;
2. preserve every generated candidate relationship;
3. apply only externally supported merge or split decisions;
4. leave all other candidate relationships unresolved;
5. revisit an unresolved relationship only when a specific analysis, article or materially important participant requires it;
6. never aggregate unresolved raw labels as one definitive real-world jockey.

The 25-pair authority-review batch is retained as evidence of the proposed verification method, but its remaining research is deferred until an analytical need justifies it.

This is an intentional governed unresolved state, not incomplete automatic cleaning.

In [23]:
# Apply the proportionate stopping rule without changing any unresolved
# relationship into a merge or split.
#
# The complete candidate queue remains available for future targeted review.
# Confirmed decisions remain active. All other relationships are explicitly
# deferred until a specific analytical use makes their resolution material.

jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

assert len(jockey_strict_review_queue) == 216
assert jockey_strict_review_queue[
    "candidate_pair_id"
].is_unique

assert len(jockey_verification_batch_01) == 25
assert jockey_verification_batch_01[
    "candidate_pair_id"
].is_unique

# Remove the unsupported full-name enrichment from the B O'Neill split.
#
# The published result proves that Miss B O'Neill and Mr B O'Neill rode in the
# same race and therefore represent different people. That result does not by
# itself establish that Mr B O'Neill's full registered name was Barry O'Neill.
b_oneill_mask = jockey_strict_review_queue[
    "candidate_pair_id"
].eq("JOCKEY-STRICT-0001")

assert b_oneill_mask.sum() == 1

jockey_strict_review_queue.loc[
    b_oneill_mask,
    "left_verified_person_name",
] = ""

jockey_strict_review_queue.loc[
    b_oneill_mask,
    "right_verified_person_name",
] = ""

jockey_strict_review_queue.loc[
    b_oneill_mask,
    "review_notes",
] = (
    "The published 4:05 Naas result on 2017-11-11 lists "
    "Miss B O'Neill and Mr B O'Neill as separate riders in the "
    "same race. This confirms that the two raw labels represent "
    "different people. The evidence does not establish either "
    "rider's full registered name, so both verified-name fields "
    "remain blank."
)

# Add explicit review-scope fields to both working artifacts.
for review_frame in (
    jockey_strict_review_queue,
    jockey_verification_batch_01,
):
    if "review_scope_status" not in review_frame.columns:
        review_frame["review_scope_status"] = ""

    if "review_trigger" not in review_frame.columns:
        review_frame["review_trigger"] = ""

    if "review_scope_reason" not in review_frame.columns:
        review_frame["review_scope_reason"] = ""

    confirmed_mask = review_frame[
        "identity_relationship"
    ].isin(
        [
            "same_person",
            "different_people",
        ]
    )

    unresolved_mask = review_frame[
        "identity_relationship"
    ].eq("unresolved")

    review_frame.loc[
        confirmed_mask,
        "review_scope_status",
    ] = "completed"

    review_frame.loc[
        confirmed_mask,
        "review_trigger",
    ] = ""

    review_frame.loc[
        confirmed_mask,
        "review_scope_reason",
    ] = (
        "Relationship supported by inspected external evidence."
    )

    review_frame.loc[
        unresolved_mask,
        "review_scope_status",
    ] = "deferred_until_material_use"

    review_frame.loc[
        unresolved_mask,
        "review_trigger",
    ] = "specific_analysis_or_material_identity_risk"

    review_frame.loc[
        unresolved_mask,
        "review_scope_reason",
    ] = (
        "Strict normalisation generated an identity candidate only. "
        "Comprehensive external resolution is disproportionate without "
        "a specific analytical need. Preserve both raw labels and leave "
        "the relationship unresolved."
    )

# Preserve unresolved database behaviour.
jockey_strict_review_queue.loc[
    jockey_strict_review_queue[
        "identity_relationship"
    ].eq("unresolved"),
    "database_action",
] = "preserve_raw_unresolved"

jockey_verification_batch_01.loc[
    jockey_verification_batch_01[
        "identity_relationship"
    ].eq("unresolved"),
    "database_action",
] = "preserve_raw_unresolved"

# Validate the stopping state before persistence.
assert jockey_strict_review_queue[
    "identity_relationship"
].eq("same_person").sum() == 1

assert jockey_strict_review_queue[
    "identity_relationship"
].eq("different_people").sum() == 1

assert jockey_strict_review_queue[
    "identity_relationship"
].eq("unresolved").sum() == 214

assert jockey_strict_review_queue[
    "review_scope_status"
].eq("deferred_until_material_use").sum() == 214

assert jockey_verification_batch_01[
    "review_scope_status"
].eq("deferred_until_material_use").sum() == 25

verified_b_oneill_row = jockey_strict_review_queue.loc[
    b_oneill_mask
].iloc[0]

assert (
    verified_b_oneill_row["identity_relationship"]
    == "different_people"
)

assert verified_b_oneill_row[
    "left_verified_person_name"
] == ""

assert verified_b_oneill_row[
    "right_verified_person_name"
] == ""

# Persist and reload both working artifacts.
jockey_strict_review_queue.to_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    index=False,
)

jockey_verification_batch_01.to_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    index=False,
)

jockey_strict_review_queue = pd.read_csv(
    JOCKEY_STRICT_REVIEW_QUEUE_PATH,
    keep_default_na=False,
)

jockey_verification_batch_01 = pd.read_csv(
    JOCKEY_VERIFICATION_BATCH_01_PATH,
    keep_default_na=False,
)

stopping_rule_summary = pd.DataFrame(
    [
        {
            "candidate_pairs": len(
                jockey_strict_review_queue
            ),
            "confirmed_same_person": int(
                jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("same_person").sum()
            ),
            "confirmed_different_people": int(
                jockey_strict_review_queue[
                    "identity_relationship"
                ].eq("different_people").sum()
            ),
            "deferred_unresolved": int(
                jockey_strict_review_queue[
                    "review_scope_status"
                ].eq(
                    "deferred_until_material_use"
                ).sum()
            ),
            "batch_pairs_retained": len(
                jockey_verification_batch_01
            ),
            "batch_pairs_deferred": int(
                jockey_verification_batch_01[
                    "review_scope_status"
                ].eq(
                    "deferred_until_material_use"
                ).sum()
            ),
        }
    ]
)

display(stopping_rule_summary)

display(
    jockey_strict_review_queue.loc[
        jockey_strict_review_queue[
            "candidate_pair_id"
        ].isin(
            [
                "JOCKEY-STRICT-0001",
                "JOCKEY-STRICT-0002",
            ]
        ),
        [
            "candidate_pair_id",
            "left_raw_jockey_label",
            "right_raw_jockey_label",
            "identity_relationship",
            "left_verified_person_name",
            "right_verified_person_name",
            "verification_status",
            "review_scope_status",
            "review_trigger",
            "database_action",
        ],
    ]
)

,candidate_pairs,confirmed_same_person,confirmed_different_people,deferred_unresolved,batch_pairs_retained,batch_pairs_deferred
0,216,1,1,214,25,25


,candidate_pair_id,left_raw_jockey_label,right_raw_jockey_label,identity_relationship,left_verified_person_name,right_verified_person_name,verification_status,review_scope_status,review_trigger,database_action
0,JOCKEY-STRICT-0001,Miss B ONeill,Mr B ONeill,different_people,,,confirmed,completed,,preserve_separate_participant_identities
1,JOCKEY-STRICT-0002,Mlle Marie Velon,Mme Marie Velon,same_person,Marie Vélon,Marie Vélon,confirmed,completed,,map_both_labels_to_same_participant_identity


## Trainer identity

### Establish the immutable raw-label baseline

The jockey investigation demonstrated that normalised text can generate useful identity candidates but cannot safely manufacture participant identities.

The trainer investigation therefore begins with the same conservative separation:

* the raw trainer label is immutable source evidence;
* a distinct raw label is not automatically a distinct real-world person;
* similar labels are not automatically aliases;
* normalisation may later generate review candidates only;
* unresolved relationships must remain unresolved unless their resolution becomes analytically material.

This stage profiles the complete governed trainer population before defining any comparison rules.

It records:

* total governed runner rows;
* SQL-null and empty-string trainer values;
* distinct populated raw trainer labels;
* runner-row and provisional-race frequency by exact label;
* first and last observed dates.

No trainer labels are altered or merged.

In [24]:
# Establish the immutable raw trainer-label baseline.
#
# This stage performs exact source profiling only. It does not normalise,
# compare, merge or resolve any trainer identities.

trainer_source_profile_summary = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS governed_runner_rows,
        SUM(
            CASE
                WHEN trainer IS NULL THEN 1
                ELSE 0
            END
        ) AS sql_null_trainer_rows,
        SUM(
            CASE
                WHEN trainer = '' THEN 1
                ELSE 0
            END
        ) AS empty_string_trainer_rows,
        SUM(
            CASE
                WHEN trainer IS NOT NULL
                 AND trainer <> ''
                THEN 1
                ELSE 0
            END
        ) AS populated_trainer_rows,
        COUNT(
            DISTINCT CASE
                WHEN trainer IS NOT NULL
                 AND trainer <> ''
                THEN trainer
            END
        ) AS distinct_populated_trainer_labels
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

trainer_label_frequency = pd.read_sql_query(
    f"""
    SELECT
        trainer AS raw_trainer_label,
        COUNT(*) AS runner_rows,
        COUNT(
            DISTINCT
                COALESCE(date, '')
                || CHAR(31)
                || COALESCE(course, '')
                || CHAR(31)
                || COALESCE(off, '')
        ) AS provisional_races,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND trainer IS NOT NULL
      AND trainer <> ''
    GROUP BY trainer
    ORDER BY
        runner_rows DESC,
        raw_trainer_label
    """,
    connection,
)

assert len(trainer_source_profile_summary) == 1

assert (
    trainer_source_profile_summary.loc[
        0,
        "governed_runner_rows",
    ]
    == 1_851_285
)

assert (
    trainer_source_profile_summary.loc[
        0,
        "populated_trainer_rows",
    ]
    + trainer_source_profile_summary.loc[
        0,
        "sql_null_trainer_rows",
    ]
    + trainer_source_profile_summary.loc[
        0,
        "empty_string_trainer_rows",
    ]
    == trainer_source_profile_summary.loc[
        0,
        "governed_runner_rows",
    ]
)

assert trainer_label_frequency[
    "raw_trainer_label"
].is_unique

assert (
    len(trainer_label_frequency)
    == trainer_source_profile_summary.loc[
        0,
        "distinct_populated_trainer_labels",
    ]
)

assert trainer_label_frequency[
    "runner_rows"
].sum() == trainer_source_profile_summary.loc[
    0,
    "populated_trainer_rows",
]

display(trainer_source_profile_summary)

display(
    trainer_label_frequency.head(25)
)

,governed_runner_rows,sql_null_trainer_rows,empty_string_trainer_rows,populated_trainer_rows,distinct_populated_trainer_labels
0,1851285,0,9,1851276,10708


,raw_trainer_label,runner_rows,provisional_races,first_date,last_date
0,Gordon Elliott,15140,11344,2015-01-01,2026-05-27
1,Richard Fahey,14643,12496,2015-01-01,2026-03-23
2,Tim Easterby,13367,11121,2015-01-01,2026-05-27
3,Richard Hannon,13165,11191,2015-01-03,2026-05-27
4,Joseph Patrick OBrien,11068,8349,2016-06-06,2026-05-26
5,David OMeara,10847,9522,2015-01-02,2026-05-27
6,W P Mullins,10807,6958,2015-01-01,2026-05-27
7,Andrew Balding,9804,9468,2015-01-01,2026-05-27
8,Michael Appleby,9584,9133,2015-01-01,2026-05-27
9,Tony Carroll,9521,8255,2015-01-01,2026-05-27


### Reconcile governed blanks and test strict trainer-title candidates

The nine empty trainer labels are not a new unresolved population. They reconcile with Notebook 20:

* nine immutable raw trainer blanks;
* four confirmed downstream supplementations;
* five blanks preserved as unresolved.

Notebook 22 does not reopen those decisions.

The next bounded question is whether populated trainer labels offer a safer identity-resolution opportunity than jockey labels.

The test uses only the same conservative transformation applied to jockey candidates:

* collapse whitespace;
* remove one recognised leading title;
* apply case-folding for comparison;
* preserve every raw label unchanged.

Recognised titles are limited to `Mr`, `Mrs`, `Miss`, `Ms`, `Mlle`, `Mme` and `Frau`.

A resulting group is only a candidate relationship. Trainer context is especially difficult to interpret:

* the same trainer may have several runners in one race;
* the same horse may legitimately move between trainers;
* overlapping active periods do not prove identity;
* separated periods may represent a label transition, succession or two different people;
* conventions such as `Mrs John Harrington` cannot safely be interpreted merely by deleting the title.

This stage therefore measures the candidate population and its source context without merging any trainer labels.

In [26]:
# Reconcile the nine immutable raw trainer blanks with Notebook 20.
#
# Notebook 20 stores the blank marker together with source locators in
# raw_source_value, for example:
# "blank; source_rowid=...; race_id=...; repair_record_id=..."

MANUAL_VERIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "manual_verifications.csv"
)

CONNECTION_IDENTITY_REPAIRS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "connection_identity_repairs.csv"
)

manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    keep_default_na=False,
)

connection_identity_repairs = pd.read_csv(
    CONNECTION_IDENTITY_REPAIRS_PATH,
    keep_default_na=False,
)

nb20_trainer_blank_verifications = (
    manual_verifications.loc[
        manual_verifications[
            "verification_id"
        ].str.startswith("NB20-CONNECTION-")
        & manual_verifications[
            "source_field"
        ].eq("trainer")
        & manual_verifications[
            "raw_source_value"
        ].str.startswith("blank;")
    ]
    .copy()
    .sort_values("verification_id")
    .reset_index(drop=True)
)

nb20_trainer_repairs = (
    connection_identity_repairs.loc[
        connection_identity_repairs[
            "source_field"
        ].eq("trainer")
    ]
    .copy()
    .sort_values("verification_id")
    .reset_index(drop=True)
)

raw_empty_trainer_rows = int(
    trainer_source_profile_summary.loc[
        0,
        "empty_string_trainer_rows",
    ]
)

confirmed_trainer_supplementations = int(
    nb20_trainer_blank_verifications[
        "database_action"
    ].eq("source_supplementation").sum()
)

preserved_unresolved_trainer_blanks = int(
    nb20_trainer_blank_verifications[
        "database_action"
    ].eq("preserve_raw_unresolved").sum()
)

assert raw_empty_trainer_rows == 9
assert len(nb20_trainer_blank_verifications) == 9
assert confirmed_trainer_supplementations == 4
assert preserved_unresolved_trainer_blanks == 5
assert len(nb20_trainer_repairs) == 4

assert nb20_trainer_blank_verifications[
    "verification_id"
].is_unique

assert nb20_trainer_repairs[
    "verification_id"
].is_unique

assert (
    confirmed_trainer_supplementations
    + preserved_unresolved_trainer_blanks
    == raw_empty_trainer_rows
)

assert set(
    nb20_trainer_repairs["verification_id"]
) == set(
    nb20_trainer_blank_verifications.loc[
        nb20_trainer_blank_verifications[
            "database_action"
        ].eq("source_supplementation"),
        "verification_id",
    ]
)

trainer_blank_reconciliation = pd.DataFrame(
    [
        {
            "raw_empty_trainer_rows": raw_empty_trainer_rows,
            "notebook_20_verifications": len(
                nb20_trainer_blank_verifications
            ),
            "confirmed_supplementations": (
                confirmed_trainer_supplementations
            ),
            "preserved_unresolved": (
                preserved_unresolved_trainer_blanks
            ),
            "promoted_repair_records": len(
                nb20_trainer_repairs
            ),
            "reconciliation_status": "exact_match",
        }
    ]
)

display(trainer_blank_reconciliation)

display(
    nb20_trainer_blank_verifications[
        [
            "verification_id",
            "source_date",
            "source_course",
            "source_horse",
            "verified_value",
            "verification_status",
            "database_action",
        ]
    ]
)

,raw_empty_trainer_rows,notebook_20_verifications,confirmed_supplementations,preserved_unresolved,promoted_repair_records,reconciliation_status
0,9,9,4,5,4,exact_match


,verification_id,source_date,source_course,source_horse,verified_value,verification_status,database_action
0,NB20-CONNECTION-0038,2015-05-06,Sonoda (JPN),Maximum Kaiser (JPN),Masaya Komura,confirmed,source_supplementation
1,NB20-CONNECTION-0039,2016-05-07,Maisons-Laffitte (FR),Star White (FR),K Borgel,confirmed,source_supplementation
2,NB20-CONNECTION-0040,2016-05-07,Maisons-Laffitte (FR),Colombia DEmra (FR),C Plisson,confirmed,source_supplementation
3,NB20-CONNECTION-0041,2018-09-10,Chantilly (FR),Numbers Talk (IRE),,unresolved,preserve_raw_unresolved
4,NB20-CONNECTION-0042,2018-09-14,Saint-Cloud (FR),Valley Kid (FR),Stal't Neerhof,confirmed,source_supplementation
5,NB20-CONNECTION-0043,2018-09-17,Maisons-Laffitte (FR),Unital (FR),,unresolved,preserve_raw_unresolved
6,NB20-CONNECTION-0044,2018-09-17,Maisons-Laffitte (FR),Talento (IRE),,unresolved,preserve_raw_unresolved
7,NB20-CONNECTION-0045,2018-09-28,Saint-Cloud (FR),Totem (FR),,unresolved,preserve_raw_unresolved
8,NB20-CONNECTION-0046,2018-09-28,Saint-Cloud (FR),Jasmine A La Plage (FR),,unresolved,preserve_raw_unresolved


In [31]:
import re

# Generate strict trainer identity candidates from populated raw labels.
#
# This stage only removes one recognised leading title for comparison.
# It does not alter raw labels, create trainer identities or authorise merges.

TRAINER_LEADING_TITLES = (
    "Mr",
    "Mrs",
    "Miss",
    "Ms",
    "Mlle",
    "Mme",
    "Frau",
)

trainer_title_pattern = re.compile(
    r"^(?P<title>"
    + "|".join(
        re.escape(title)
        for title in TRAINER_LEADING_TITLES
    )
    + r")\s+",
    flags=re.IGNORECASE,
)


def derive_strict_trainer_candidate_fields(raw_label):
    """Return conservative comparison fields for one raw trainer label."""

    collapsed_label = " ".join(str(raw_label).split())
    title_match = trainer_title_pattern.match(
        collapsed_label
    )

    if title_match:
        leading_title = title_match.group("title")
        comparison_label = trainer_title_pattern.sub(
            "",
            collapsed_label,
            count=1,
        )
        title_removed = True
    else:
        leading_title = ""
        comparison_label = collapsed_label
        title_removed = False

    return pd.Series(
        {
            "collapsed_trainer_label": collapsed_label,
            "leading_title": leading_title,
            "title_removed": title_removed,
            "strict_comparison_key": (
                comparison_label.casefold()
            ),
        }
    )


trainer_strict_label_profile = (
    trainer_label_frequency.copy()
)

trainer_strict_fields = (
    trainer_strict_label_profile[
        "raw_trainer_label"
    ].apply(
        derive_strict_trainer_candidate_fields
    )
)

trainer_strict_label_profile = pd.concat(
    [
        trainer_strict_label_profile,
        trainer_strict_fields,
    ],
    axis=1,
)

assert trainer_strict_label_profile[
    "raw_trainer_label"
].is_unique

trainer_strict_group_profile = (
    trainer_strict_label_profile.groupby(
        "strict_comparison_key",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=(
            "raw_trainer_label",
            "nunique",
        ),
        titled_labels=(
            "title_removed",
            "sum",
        ),
        combined_runner_rows=(
            "runner_rows",
            "sum",
        ),
        earliest_date=(
            "first_date",
            "min",
        ),
        latest_date=(
            "last_date",
            "max",
        ),
    )
)

trainer_strict_candidate_groups = (
    trainer_strict_group_profile.loc[
        trainer_strict_group_profile[
            "distinct_raw_labels"
        ].gt(1)
        & trainer_strict_group_profile[
            "titled_labels"
        ].gt(0)
    ]
    .copy()
    .sort_values(
        [
            "combined_runner_rows",
            "strict_comparison_key",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

trainer_strict_candidate_labels = (
    trainer_strict_label_profile.loc[
        trainer_strict_label_profile[
            "strict_comparison_key"
        ].isin(
            trainer_strict_candidate_groups[
                "strict_comparison_key"
            ]
        )
    ]
    .copy()
    .sort_values(
        [
            "strict_comparison_key",
            "title_removed",
            "raw_trainer_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

trainer_strict_candidate_summary = pd.DataFrame(
    [
        {
            "distinct_populated_trainer_labels": len(
                trainer_strict_label_profile
            ),
            "labels_with_recognised_title": int(
                trainer_strict_label_profile[
                    "title_removed"
                ].sum()
            ),
            "strict_candidate_groups": len(
                trainer_strict_candidate_groups
            ),
            "candidate_raw_labels": len(
                trainer_strict_candidate_labels
            ),
            "candidate_runner_rows": int(
                trainer_strict_candidate_labels[
                    "runner_rows"
                ].sum()
            ),
        }
    ]
)

display(trainer_strict_candidate_summary)

display(
    trainer_strict_candidate_labels[
        [
            "strict_comparison_key",
            "raw_trainer_label",
            "leading_title",
            "title_removed",
            "runner_rows",
            "provisional_races",
            "first_date",
            "last_date",
        ]
    ]
)

,distinct_populated_trainer_labels,labels_with_recognised_title,strict_candidate_groups,candidate_raw_labels,candidate_runner_rows
0,10708,767,53,106,18769


,strict_comparison_key,raw_trainer_label,leading_title,title_removed,runner_rows,provisional_races,first_date,last_date
0,a budka,Mlle A Budka,Mlle,True,154,152,2020-08-08,2023-12-29
1,a budka,Mme A Budka,Mme,True,205,197,2024-01-13,2026-05-24
2,a fabre,Mme A Fabre,Mme,True,58,58,2015-11-02,2026-05-07
3,a fabre,A Fabre,,False,5037,4032,2015-02-25,2026-05-24
4,a wattel,Mlle A Wattel,Mlle,True,262,255,2018-03-13,2023-12-28
...,...,...,...,...,...,...,...,...
101,victoria head,Mme Victoria Head,Mme,True,306,295,2024-01-13,2026-05-22
102,w szczesniak,Mme W Szczesniak,Mme,True,20,20,2025-02-07,2026-05-19
103,w szczesniak,W Szczesniak,,False,1,1,2025-01-16,2025-01-16
104,y vollmer,Mlle Y Vollmer,Mlle,True,377,344,2015-04-15,2023-12-30


### Test chronology of strict trainer-title candidates

The strict transformation produced 53 candidate groups from 10,708 populated trainer labels.

This is substantially smaller than the jockey candidate population and includes visible systematic patterns, particularly French `Mlle` and `Mme` transitions.

Chronology may distinguish relatively safe source-label transitions from ambiguous relationships:

* non-overlapping `Mlle` then `Mme` periods may indicate a source convention change;
* overlap between title variants weakens that interpretation;
* titled and untitled variants may reflect different feeds rather than a real identity transition;
* chronology alone still cannot prove that two labels identify the same real-world trainer.

This stage classifies each candidate group by title structure and active-period relationship. It does not merge any labels.

In [32]:
# Classify strict trainer candidate groups by title structure and chronology.
#
# This remains candidate evidence only. No trainer identities are created and
# no raw labels are merged.

trainer_candidate_chronology_rows = []

for comparison_key, group_rows in (
    trainer_strict_candidate_labels.groupby(
        "strict_comparison_key"
    )
):
    group_rows = (
        group_rows.sort_values(
            [
                "first_date",
                "last_date",
                "raw_trainer_label",
            ]
        )
        .reset_index(drop=True)
    )

    assert len(group_rows) == 2

    left = group_rows.iloc[0]
    right = group_rows.iloc[1]

    left_title = left["leading_title"]
    right_title = right["leading_title"]

    if left["title_removed"] and right["title_removed"]:
        title_structure = "two_titled_labels"
    elif left["title_removed"] or right["title_removed"]:
        title_structure = "titled_and_untitled"
    else:
        title_structure = "two_untitled_labels"

    periods_overlap = (
        max(
            left["first_date"],
            right["first_date"],
        )
        <= min(
            left["last_date"],
            right["last_date"],
        )
    )

    if periods_overlap:
        chronology_status = "active_period_overlap"
        transition_gap_days = None
    else:
        chronology_status = "separate_active_periods"

        transition_gap_days = (
            pd.to_datetime(right["first_date"])
            - pd.to_datetime(left["last_date"])
        ).days

    if (
        left_title.casefold() == "mlle"
        and right_title.casefold() == "mme"
    ):
        title_sequence = "mlle_then_mme"
    elif (
        left_title.casefold() == "mme"
        and right_title.casefold() == "mlle"
    ):
        title_sequence = "mme_then_mlle"
    elif left_title == "" and right_title != "":
        title_sequence = "untitled_then_titled"
    elif left_title != "" and right_title == "":
        title_sequence = "titled_then_untitled"
    else:
        title_sequence = "other_title_sequence"

    trainer_candidate_chronology_rows.append(
        {
            "strict_comparison_key": comparison_key,
            "left_raw_trainer_label": (
                left["raw_trainer_label"]
            ),
            "right_raw_trainer_label": (
                right["raw_trainer_label"]
            ),
            "left_title": left_title,
            "right_title": right_title,
            "left_first_date": left["first_date"],
            "left_last_date": left["last_date"],
            "right_first_date": right["first_date"],
            "right_last_date": right["last_date"],
            "left_runner_rows": int(
                left["runner_rows"]
            ),
            "right_runner_rows": int(
                right["runner_rows"]
            ),
            "title_structure": title_structure,
            "title_sequence": title_sequence,
            "chronology_status": chronology_status,
            "transition_gap_days": transition_gap_days,
        }
    )

trainer_candidate_chronology = pd.DataFrame(
    trainer_candidate_chronology_rows
)

assert len(trainer_candidate_chronology) == 53
assert trainer_candidate_chronology[
    "strict_comparison_key"
].is_unique

trainer_candidate_chronology_summary = (
    trainer_candidate_chronology.groupby(
        [
            "title_structure",
            "title_sequence",
            "chronology_status",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        candidate_groups=(
            "strict_comparison_key",
            "size",
        ),
        combined_runner_rows=(
            "left_runner_rows",
            "sum",
        ),
    )
    .sort_values(
        [
            "candidate_groups",
            "title_structure",
            "title_sequence",
            "chronology_status",
        ],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

trainer_candidate_chronology_summary[
    "combined_runner_rows"
] = (
    trainer_candidate_chronology_summary[
        "combined_runner_rows"
    ]
    + trainer_candidate_chronology.groupby(
        [
            "title_structure",
            "title_sequence",
            "chronology_status",
        ],
        dropna=False,
    )["right_runner_rows"]
    .sum()
    .to_numpy()
)

display(trainer_candidate_chronology_summary)

display(
    trainer_candidate_chronology.sort_values(
        [
            "chronology_status",
            "title_sequence",
            "transition_gap_days",
            "strict_comparison_key",
        ],
        ascending=[True, True, True, True],
    )
)

,title_structure,title_sequence,chronology_status,candidate_groups,combined_runner_rows
0,two_titled_labels,mlle_then_mme,separate_active_periods,38,4825
1,titled_and_untitled,titled_then_untitled,separate_active_periods,5,434
2,titled_and_untitled,untitled_then_titled,active_period_overlap,4,10352
3,titled_and_untitled,untitled_then_titled,separate_active_periods,2,23
4,two_titled_labels,other_title_sequence,separate_active_periods,2,319
5,titled_and_untitled,titled_then_untitled,active_period_overlap,1,2585
6,two_titled_labels,mlle_then_mme,active_period_overlap,1,231


,strict_comparison_key,left_raw_trainer_label,right_raw_trainer_label,left_title,right_title,left_first_date,left_last_date,right_first_date,right_last_date,left_runner_rows,right_runner_rows,title_structure,title_sequence,chronology_status,transition_gap_days
29,l pontoir,Mlle L Pontoir,Mme L Pontoir,Mlle,Mme,2021-03-10,2024-01-13,2024-01-05,2026-05-26,218,311,two_titled_labels,mlle_then_mme,active_period_overlap,NaN
23,jane williams,Mrs Jane Williams,Jane Williams,Mrs,,2015-01-07,2022-10-23,2022-08-08,2026-05-06,308,548,titled_and_untitled,titled_then_untitled,active_period_overlap,NaN
1,a fabre,A Fabre,Mme A Fabre,,Mme,2015-02-25,2026-05-24,2015-11-02,2026-05-07,5037,58,titled_and_untitled,untitled_then_titled,active_period_overlap,NaN
22,j-f bernard,J-F Bernard,Mme J-F Bernard,,Mme,2015-03-17,2019-11-26,2016-03-10,2023-02-10,4,19,titled_and_untitled,untitled_then_titled,active_period_overlap,NaN
25,john dawson,John Dawson,Mrs John Dawson,,Mrs,2017-05-06,2026-05-25,2024-05-11,2026-05-19,39,4,titled_and_untitled,untitled_then_titled,active_period_overlap,NaN
26,jonjo oneill,Jonjo ONeill,Mrs Jonjo ONeill,,Mrs,2015-01-01,2024-04-30,2020-02-05,2024-05-03,5185,6,titled_and_untitled,untitled_then_titled,active_period_overlap,NaN
45,stephanie nigge,Mlle Stephanie Nigge,Mme Stephanie Nigge,Mlle,Mme,2020-02-20,2023-12-30,2024-01-05,2026-05-12,298,158,two_titled_labels,mlle_then_mme,separate_active_periods,6.0
52,y vollmer,Mlle Y Vollmer,Mme Y Vollmer,Mlle,Mme,2015-04-15,2023-12-30,2024-01-06,2026-05-21,377,197,two_titled_labels,mlle_then_mme,separate_active_periods,7.0
2,a wattel,Mlle A Wattel,Mme A Wattel,Mlle,Mme,2018-03-13,2023-12-28,2024-01-05,2026-05-24,262,149,two_titled_labels,mlle_then_mme,separate_active_periods,8.0
48,v dissaux,Mlle V Dissaux,Mme V Dissaux,Mlle,Mme,2015-01-03,2023-12-26,2024-01-05,2026-05-15,588,159,two_titled_labels,mlle_then_mme,separate_active_periods,10.0


### Test whether the French title transition is source-wide

Thirty-eight strict trainer candidate groups show non-overlapping `Mlle` followed by `Mme` activity, with many transitions clustered near January 2024.

This pattern may represent a systematic source-presentation change rather than 38 independent real-world identity events.

The next test profiles all populated trainer labels beginning with `Mlle` or `Mme` by month. It asks whether:

* `Mlle` usage falls sharply at the end of 2023;
* `Mme` usage rises correspondingly from the beginning of 2024;
* the transition appears across many distinct trainer labels;
* the small number of overlapping dates can be explained as feed latency or mixed source representations.

A source-wide convention change would justify treating strict `Mlle` and `Mme` variants as strong alias candidates, while still preserving the immutable raw labels.

It would not prove that every identical post-title name worldwide represents the same real-world trainer.

In [34]:
# Profile all populated trainer labels beginning with Mlle or Mme by month.
#
# This tests whether the candidate transitions reflect a systematic source
# convention change rather than separate trainer-by-trainer identity events.

trainer_title_source_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        date,
        trainer AS raw_trainer_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND trainer IS NOT NULL
      AND trainer <> ''
      AND (
          trainer LIKE 'Mlle %'
          OR trainer LIKE 'Mme %'
      )
    """,
    connection,
)

assert not trainer_title_source_rows.empty

trainer_title_source_rows["date"] = pd.to_datetime(
    trainer_title_source_rows["date"],
    errors="raise",
)

trainer_title_source_rows["month"] = (
    trainer_title_source_rows["date"]
    .dt.to_period("M")
    .astype(str)
)

trainer_title_source_rows["leading_title"] = (
    trainer_title_source_rows[
        "raw_trainer_label"
    ]
    .str.extract(
        r"^(Mlle|Mme)\s+",
        expand=False,
    )
)

trainer_title_source_rows[
    "post_title_name"
] = (
    trainer_title_source_rows[
        "raw_trainer_label"
    ]
    .str.replace(
        r"^(?:Mlle|Mme)\s+",
        "",
        regex=True,
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
    .str.casefold()
)

assert trainer_title_source_rows[
    "leading_title"
].notna().all()

trainer_title_monthly_profile = (
    trainer_title_source_rows.groupby(
        [
            "month",
            "leading_title",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_labels=(
            "raw_trainer_label",
            "nunique",
        ),
        distinct_post_title_names=(
            "post_title_name",
            "nunique",
        ),
    )
    .sort_values(
        [
            "month",
            "leading_title",
        ]
    )
    .reset_index(drop=True)
)

trainer_title_year_profile = (
    trainer_title_source_rows.assign(
        year=trainer_title_source_rows[
            "date"
        ].dt.year
    )
    .groupby(
        [
            "year",
            "leading_title",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_labels=(
            "raw_trainer_label",
            "nunique",
        ),
        distinct_post_title_names=(
            "post_title_name",
            "nunique",
        ),
    )
    .sort_values(
        [
            "year",
            "leading_title",
        ]
    )
    .reset_index(drop=True)
)

transition_window_profile = (
    trainer_title_monthly_profile.loc[
        trainer_title_monthly_profile[
            "month"
        ].between(
            "2023-07",
            "2024-06",
        )
    ]
    .copy()
    .reset_index(drop=True)
)

title_boundary_summary = pd.DataFrame(
    [
        {
            "last_mlle_date": (
                trainer_title_source_rows.loc[
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mlle"),
                    "date",
                ].max().date()
            ),
            "first_mme_date": (
                trainer_title_source_rows.loc[
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mme"),
                    "date",
                ].min().date()
            ),
            "mlle_rows_from_2024": int(
                (
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mlle")
                    & trainer_title_source_rows[
                        "date"
                    ].ge("2024-01-01")
                ).sum()
            ),
            "mme_rows_before_2024": int(
                (
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mme")
                    & trainer_title_source_rows[
                        "date"
                    ].lt("2024-01-01")
                ).sum()
            ),
            "distinct_mlle_names": int(
                trainer_title_source_rows.loc[
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mlle"),
                    "post_title_name",
                ].nunique()
            ),
            "distinct_mme_names": int(
                trainer_title_source_rows.loc[
                    trainer_title_source_rows[
                        "leading_title"
                    ].eq("Mme"),
                    "post_title_name",
                ].nunique()
            ),
        }
    ]
)

display(title_boundary_summary)

display(transition_window_profile)

display(trainer_title_year_profile)

,last_mlle_date,first_mme_date,mlle_rows_from_2024,mme_rows_before_2024,distinct_mlle_names,distinct_mme_names
0,2026-05-12,2015-01-03,41,8618,100,192


,month,leading_title,runner_rows,distinct_labels,distinct_post_title_names
0,2023-07,Mlle,53,22,22
1,2023-07,Mme,46,21,21
2,2023-08,Mlle,70,20,20
3,2023-08,Mme,48,18,18
4,2023-09,Mlle,97,23,23
5,2023-09,Mme,71,23,23
6,2023-10,Mlle,89,23,23
7,2023-10,Mme,96,29,29
8,2023-11,Mlle,103,26,26
9,2023-11,Mme,89,29,29


,year,leading_title,runner_rows,distinct_labels,distinct_post_title_names
0,2015,Mlle,662,38,38
1,2015,Mme,1174,50,50
2,2016,Mlle,645,41,41
3,2016,Mme,1095,53,53
4,2017,Mlle,477,45,45
5,2017,Mme,1074,53,53
6,2018,Mlle,575,47,47
7,2018,Mme,1088,53,53
8,2019,Mlle,674,43,43
9,2019,Mme,1173,47,47


### Trainer-title conclusion and bounded mapping rule

The trainer data offers a narrower usable rule than the jockey data.

`Mme` did not globally replace `Mlle`: both titles appear throughout the source period, and a small `Mlle` residue continues after 2024.

However, the source shows a strong presentation discontinuity around January 2024:

* `Mlle` falls from 935 runner rows in 2023 to 30 in 2024;
* `Mme` rises from 789 runner rows in 2023 to 1,939 in 2024;
* 38 exact post-title trainer names move from `Mlle` to `Mme` with no active-period overlap;
* most high-volume transitions occur close to the beginning of 2024.

For those 38 exact-name, non-overlapping `Mlle → Mme` pairs, the evidence supports a source-label transition rule.

This rule establishes source-presentation equivalence only. It does not independently verify legal names, marital status, licence identity or the identity of similarly named trainers outside the exact governed pair.

The database may therefore:

1. preserve both immutable raw labels;
2. assign both labels to one provisional trainer identity;
3. record the mapping method as `exact_mlle_to_mme_source_transition`;
4. record confidence as high for source-label equivalence;
5. retain the remaining 15 candidate groups as unresolved.

The overlapping `Mlle L Pontoir` and `Mme L Pontoir` pair is excluded from the automatic transition rule.

In [35]:
# Apply the bounded trainer-title decision.
#
# Only exact-name, non-overlapping Mlle -> Mme pairs are accepted as
# source-label transitions. This does not assert independently verified
# real-world identity.

trainer_title_decisions = (
    trainer_candidate_chronology.copy()
)

trainer_title_decisions[
    "identity_relationship"
] = "unresolved"

trainer_title_decisions[
    "decision_basis"
] = (
    "strict title-removal candidate only"
)

trainer_title_decisions[
    "confidence"
] = "low"

trainer_title_decisions[
    "database_action"
] = "preserve_raw_unresolved"

confirmed_transition_mask = (
    trainer_title_decisions[
        "title_structure"
    ].eq("two_titled_labels")
    & trainer_title_decisions[
        "title_sequence"
    ].eq("mlle_then_mme")
    & trainer_title_decisions[
        "chronology_status"
    ].eq("separate_active_periods")
)

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "identity_relationship",
] = "same_provisional_trainer"

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "decision_basis",
] = (
    "exact post-title name with non-overlapping Mlle-to-Mme "
    "source transition, supported by the source-wide January 2024 "
    "presentation discontinuity"
)

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "confidence",
] = "high_source_label_equivalence"

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "database_action",
] = (
    "map_both_labels_to_same_provisional_trainer_identity"
)

trainer_title_decisions[
    "mapping_method"
] = ""

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "mapping_method",
] = "exact_mlle_to_mme_source_transition"

trainer_title_decisions[
    "review_scope_status"
] = "deferred_until_material_use"

trainer_title_decisions.loc[
    confirmed_transition_mask,
    "review_scope_status",
] = "completed"

assert len(trainer_title_decisions) == 53

assert trainer_title_decisions[
    "strict_comparison_key"
].is_unique

assert confirmed_transition_mask.sum() == 38

assert trainer_title_decisions[
    "identity_relationship"
].eq("same_provisional_trainer").sum() == 38

assert trainer_title_decisions[
    "identity_relationship"
].eq("unresolved").sum() == 15

assert not trainer_title_decisions.loc[
    trainer_title_decisions[
        "strict_comparison_key"
    ].eq("l pontoir"),
    "identity_relationship",
].eq("same_provisional_trainer").any()

TRAINER_IDENTITY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "trainer_identity"
)

TRAINER_IDENTITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINER_TITLE_DECISIONS_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_strict_title_decisions.csv"
)

trainer_title_decisions.to_csv(
    TRAINER_TITLE_DECISIONS_PATH,
    index=False,
)

trainer_title_decisions = pd.read_csv(
    TRAINER_TITLE_DECISIONS_PATH,
    keep_default_na=False,
)

assert len(trainer_title_decisions) == 53
assert trainer_title_decisions[
    "strict_comparison_key"
].is_unique

trainer_title_decision_summary = pd.DataFrame(
    [
        {
            "candidate_groups": len(
                trainer_title_decisions
            ),
            "confirmed_source_transitions": int(
                trainer_title_decisions[
                    "identity_relationship"
                ].eq(
                    "same_provisional_trainer"
                ).sum()
            ),
            "preserved_unresolved": int(
                trainer_title_decisions[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "mapped_runner_rows": int(
                trainer_title_decisions.loc[
                    trainer_title_decisions[
                        "identity_relationship"
                    ].eq(
                        "same_provisional_trainer"
                    ),
                    [
                        "left_runner_rows",
                        "right_runner_rows",
                    ],
                ].sum().sum()
            ),
            "mapping_method": (
                "exact_mlle_to_mme_source_transition"
            ),
        }
    ]
)

display(trainer_title_decision_summary)

display(
    trainer_title_decisions[
        [
            "strict_comparison_key",
            "left_raw_trainer_label",
            "right_raw_trainer_label",
            "chronology_status",
            "transition_gap_days",
            "identity_relationship",
            "confidence",
            "mapping_method",
            "database_action",
        ]
    ]
)

,candidate_groups,confirmed_source_transitions,preserved_unresolved,mapped_runner_rows,mapping_method
0,53,38,15,6554,exact_mlle_to_mme_source_transition


,strict_comparison_key,left_raw_trainer_label,right_raw_trainer_label,chronology_status,transition_gap_days,identity_relationship,confidence,mapping_method,database_action
0,a budka,Mlle A Budka,Mme A Budka,separate_active_periods,15.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
1,a fabre,A Fabre,Mme A Fabre,active_period_overlap,,unresolved,low,,preserve_raw_unresolved
2,a wattel,Mlle A Wattel,Mme A Wattel,separate_active_periods,8.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
3,a-s crombez,Mlle A-S Crombez,Mme A-S Crombez,separate_active_periods,14.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
4,a-s pacault,Mlle A-S Pacault,Mme A-S Pacault,separate_active_periods,68.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
5,annabelle sowray,Miss Annabelle Sowray,Annabelle Sowray,separate_active_periods,1522.0,unresolved,low,,preserve_raw_unresolved
6,belinda clarke,Mrs Belinda Clarke,Belinda Clarke,separate_active_periods,604.0,unresolved,low,,preserve_raw_unresolved
7,c chenu,Mlle C Chenu,Mme C Chenu,separate_active_periods,93.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
8,c gryson,Mlle C Gryson,Mme C Gryson,separate_active_periods,49.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...
9,c herisson de beauvoir,Mlle C Herisson De Beauvoir,Mme C Herisson De Beauvoir,separate_active_periods,344.0,same_provisional_trainer,high_source_label_equivalence,exact_mlle_to_mme_source_transition,map_both_labels_to_same_provisional_trainer_id...


In [36]:
# Narrow the Mlle -> Mme rule to pairs that genuinely align with the
# observed source-presentation change around the beginning of 2024.
#
# Exact matching names outside this transition window remain candidates only.

trainer_title_decisions = trainer_candidate_chronology.copy()

trainer_title_decisions["left_last_date"] = pd.to_datetime(
    trainer_title_decisions["left_last_date"],
    errors="raise",
)

trainer_title_decisions["right_first_date"] = pd.to_datetime(
    trainer_title_decisions["right_first_date"],
    errors="raise",
)

trainer_title_decisions[
    "identity_relationship"
] = "unresolved"

trainer_title_decisions[
    "decision_basis"
] = "strict title-removal candidate only"

trainer_title_decisions["confidence"] = "low"

trainer_title_decisions[
    "mapping_method"
] = ""

trainer_title_decisions[
    "database_action"
] = "preserve_raw_unresolved"

# The source-wide discontinuity begins in January 2024.
#
# Require the Mlle label to remain active during the second half of 2023 and
# the Mme label to begin during the first half of 2024. This captures the
# actual feed transition rather than treating every historical exact-name
# recurrence as the same person.
source_transition_mask = (
    trainer_title_decisions[
        "title_structure"
    ].eq("two_titled_labels")
    & trainer_title_decisions[
        "title_sequence"
    ].eq("mlle_then_mme")
    & trainer_title_decisions[
        "chronology_status"
    ].eq("separate_active_periods")
    & trainer_title_decisions[
        "left_last_date"
    ].between(
        "2023-07-01",
        "2023-12-31",
    )
    & trainer_title_decisions[
        "right_first_date"
    ].between(
        "2024-01-01",
        "2024-06-30",
    )
)

trainer_title_decisions.loc[
    source_transition_mask,
    "identity_relationship",
] = "same_provisional_trainer"

trainer_title_decisions.loc[
    source_transition_mask,
    "decision_basis",
] = (
    "exact post-title name with a non-overlapping Mlle-to-Mme "
    "transition aligned to the source-wide January 2024 discontinuity"
)

trainer_title_decisions.loc[
    source_transition_mask,
    "confidence",
] = "high_source_label_equivalence"

trainer_title_decisions.loc[
    source_transition_mask,
    "mapping_method",
] = "bounded_2024_mlle_to_mme_source_transition"

trainer_title_decisions.loc[
    source_transition_mask,
    "database_action",
] = (
    "map_both_labels_to_same_provisional_trainer_identity"
)

assert len(trainer_title_decisions) == 53
assert trainer_title_decisions[
    "strict_comparison_key"
].is_unique

# The overlapping L Pontoir labels must remain unresolved.
assert not trainer_title_decisions.loc[
    trainer_title_decisions[
        "strict_comparison_key"
    ].eq("l pontoir"),
    "identity_relationship",
].eq("same_provisional_trainer").any()

trainer_title_bounded_summary = pd.DataFrame(
    [
        {
            "candidate_groups": len(
                trainer_title_decisions
            ),
            "bounded_2024_source_transitions": int(
                source_transition_mask.sum()
            ),
            "preserved_unresolved": int(
                (~source_transition_mask).sum()
            ),
            "mapped_runner_rows": int(
                trainer_title_decisions.loc[
                    source_transition_mask,
                    [
                        "left_runner_rows",
                        "right_runner_rows",
                    ],
                ].sum().sum()
            ),
        }
    ]
)

display(trainer_title_bounded_summary)

display(
    trainer_title_decisions.loc[
        source_transition_mask,
        [
            "strict_comparison_key",
            "left_raw_trainer_label",
            "right_raw_trainer_label",
            "left_last_date",
            "right_first_date",
            "transition_gap_days",
            "identity_relationship",
            "confidence",
        ],
    ].sort_values(
        [
            "right_first_date",
            "strict_comparison_key",
        ]
    )
)

,candidate_groups,bounded_2024_source_transitions,preserved_unresolved,mapped_runner_rows
0,53,26,27,6350


,strict_comparison_key,left_raw_trainer_label,right_raw_trainer_label,left_last_date,right_first_date,transition_gap_days,identity_relationship,confidence
2,a wattel,Mlle A Wattel,Mme A Wattel,2023-12-28,2024-01-05,8.0,same_provisional_trainer,high_source_label_equivalence
45,stephanie nigge,Mlle Stephanie Nigge,Mme Stephanie Nigge,2023-12-30,2024-01-05,6.0,same_provisional_trainer,high_source_label_equivalence
48,v dissaux,Mlle V Dissaux,Mme V Dissaux,2023-12-26,2024-01-05,10.0,same_provisional_trainer,high_source_label_equivalence
17,g gadbled,Mlle G Gadbled,Mme G Gadbled,2023-12-26,2024-01-06,11.0,same_provisional_trainer,high_source_label_equivalence
24,jessica dupont-fahn,Mlle Jessica Dupont-Fahn,Mme Jessica Dupont-Fahn,2023-12-26,2024-01-06,11.0,same_provisional_trainer,high_source_label_equivalence
46,stephanie penot,Mlle Stephanie Penot,Mme Stephanie Penot,2023-12-22,2024-01-06,15.0,same_provisional_trainer,high_source_label_equivalence
52,y vollmer,Mlle Y Vollmer,Mme Y Vollmer,2023-12-30,2024-01-06,7.0,same_provisional_trainer,high_source_label_equivalence
12,celine lequien,Mlle Celine Lequien,Mme Celine Lequien,2023-11-30,2024-01-12,43.0,same_provisional_trainer,high_source_label_equivalence
21,j soudan,Mlle J Soudan,Mme J Soudan,2023-12-30,2024-01-12,13.0,same_provisional_trainer,high_source_label_equivalence
31,laura hebrard de veyrinas,Mlle Laura Hebrard De Veyrinas,Mme Laura Hebrard De Veyrinas,2023-12-29,2024-01-12,14.0,same_provisional_trainer,high_source_label_equivalence


### Trainer identity stopping point

The trainer investigation produced a narrower usable result than the jockey investigation.

The immutable source contains 10,708 distinct populated trainer labels. Strict removal of one recognised leading title generated 53 candidate groups involving 106 raw labels.

A source-wide discontinuity occurs around January 2024:

* `Mlle` usage falls from 935 runner rows in 2023 to 30 in 2024;
* `Mme` usage rises from 789 runner rows in 2023 to 1,939 in 2024;
* multiple exact post-title names change from `Mlle` to `Mme` at approximately the same time.

A bounded rule therefore accepts 26 exact-name, non-overlapping `Mlle → Mme` transitions whose earlier label remained active in the second half of 2023 and whose later label began in the first half of 2024.

These mappings cover 6,350 runner rows.

The rule establishes high-confidence source-label equivalence, not independently verified legal or licensing identity. Both immutable raw labels must remain preserved and the relationship must be recorded as provisional.

The remaining 27 candidate groups are unresolved. They include:

* overlapping titled and untitled labels;
* the overlapping `Mlle L Pontoir` and `Mme L Pontoir` pair;
* English and German title variants;
* exact-name recurrences separated by periods too distant from the observed 2024 source transition.

No general title-stripping rule is authorised.

The practical database consequence is:

1. preserve every raw trainer label;
2. map the 26 governed pairs to shared provisional trainer identities;
3. record the method as `bounded_2024_mlle_to_mme_source_transition`;
4. retain all other title-derived relationships as unresolved candidates;
5. revisit unresolved labels only when they materially affect a specific analysis, article or participant.

In [37]:
# Persist and reload the bounded trainer title decisions.
#
# The file records:
# - 26 accepted provisional Mlle -> Mme source-label transitions;
# - 27 unresolved title-derived candidate groups;
# - the evidence basis and permitted database action for each group.

TRAINER_IDENTITY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "trainer_identity"
)

TRAINER_IDENTITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TRAINER_TITLE_DECISIONS_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_strict_title_decisions.csv"
)

trainer_title_decisions_to_write = (
    trainer_title_decisions.copy()
)

for date_column in (
    "left_first_date",
    "left_last_date",
    "right_first_date",
    "right_last_date",
):
    trainer_title_decisions_to_write[
        date_column
    ] = pd.to_datetime(
        trainer_title_decisions_to_write[
            date_column
        ],
        errors="raise",
    ).dt.strftime("%Y-%m-%d")

trainer_title_decisions_to_write.to_csv(
    TRAINER_TITLE_DECISIONS_PATH,
    index=False,
)

trainer_title_decisions_reloaded = pd.read_csv(
    TRAINER_TITLE_DECISIONS_PATH,
    keep_default_na=False,
)

assert len(trainer_title_decisions_reloaded) == 53

assert trainer_title_decisions_reloaded[
    "strict_comparison_key"
].is_unique

assert (
    trainer_title_decisions_reloaded[
        "identity_relationship"
    ].eq("same_provisional_trainer").sum()
    == 26
)

assert (
    trainer_title_decisions_reloaded[
        "identity_relationship"
    ].eq("unresolved").sum()
    == 27
)

assert (
    trainer_title_decisions_reloaded.loc[
        trainer_title_decisions_reloaded[
            "identity_relationship"
        ].eq("same_provisional_trainer"),
        "mapping_method",
    ].eq(
        "bounded_2024_mlle_to_mme_source_transition"
    ).all()
)

assert (
    trainer_title_decisions_reloaded.loc[
        trainer_title_decisions_reloaded[
            "identity_relationship"
        ].eq("unresolved"),
        "database_action",
    ].eq("preserve_raw_unresolved").all()
)

assert not trainer_title_decisions_reloaded.loc[
    trainer_title_decisions_reloaded[
        "strict_comparison_key"
    ].eq("l pontoir"),
    "identity_relationship",
].eq("same_provisional_trainer").any()

trainer_title_persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                TRAINER_TITLE_DECISIONS_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_candidate_groups": len(
                trainer_title_decisions_reloaded
            ),
            "persisted_source_transitions": int(
                trainer_title_decisions_reloaded[
                    "identity_relationship"
                ].eq(
                    "same_provisional_trainer"
                ).sum()
            ),
            "persisted_unresolved": int(
                trainer_title_decisions_reloaded[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "reload_validation": "passed",
        }
    ]
)

display(trainer_title_persistence_summary)

,output_path,persisted_candidate_groups,persisted_source_transitions,persisted_unresolved,reload_validation
0,data/processed/trainer_identity/trainer_strict...,53,26,27,passed


In [38]:
# Create one provisional trainer identity per accepted Mlle -> Mme transition.
#
# Each identity has exactly two governed raw labels. Unresolved candidate
# groups are deliberately excluded from this mapping.

accepted_trainer_transitions = (
    trainer_title_decisions_reloaded.loc[
        trainer_title_decisions_reloaded[
            "identity_relationship"
        ].eq("same_provisional_trainer")
    ]
    .copy()
    .sort_values("strict_comparison_key")
    .reset_index(drop=True)
)

assert len(accepted_trainer_transitions) == 26

trainer_identity_mapping_rows = []

for identity_number, transition in enumerate(
    accepted_trainer_transitions.itertuples(
        index=False
    ),
    start=1,
):
    provisional_trainer_id = (
        f"TRAINER-PROVISIONAL-{identity_number:04d}"
    )

    for raw_label, label_role in (
        (
            transition.left_raw_trainer_label,
            "earlier_mlle_label",
        ),
        (
            transition.right_raw_trainer_label,
            "later_mme_label",
        ),
    ):
        trainer_identity_mapping_rows.append(
            {
                "provisional_trainer_id": (
                    provisional_trainer_id
                ),
                "raw_trainer_label": raw_label,
                "label_role": label_role,
                "strict_comparison_key": (
                    transition.strict_comparison_key
                ),
                "identity_status": (
                    "provisional_source_label_identity"
                ),
                "mapping_method": (
                    "bounded_2024_mlle_to_mme_"
                    "source_transition"
                ),
                "confidence": (
                    "high_source_label_equivalence"
                ),
                "database_action": (
                    "map_raw_label_to_"
                    "provisional_trainer_identity"
                ),
            }
        )

trainer_provisional_identity_mapping = pd.DataFrame(
    trainer_identity_mapping_rows
)

assert len(trainer_provisional_identity_mapping) == 52

assert trainer_provisional_identity_mapping[
    "raw_trainer_label"
].is_unique

assert (
    trainer_provisional_identity_mapping.groupby(
        "provisional_trainer_id"
    )["raw_trainer_label"]
    .nunique()
    .eq(2)
    .all()
)

assert (
    trainer_provisional_identity_mapping[
        "provisional_trainer_id"
    ].nunique()
    == 26
)

TRAINER_PROVISIONAL_MAPPING_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_provisional_identity_mapping.csv"
)

trainer_provisional_identity_mapping.to_csv(
    TRAINER_PROVISIONAL_MAPPING_PATH,
    index=False,
)

trainer_provisional_identity_mapping_reloaded = (
    pd.read_csv(
        TRAINER_PROVISIONAL_MAPPING_PATH,
        keep_default_na=False,
    )
)

assert (
    trainer_provisional_identity_mapping_reloaded.equals(
        trainer_provisional_identity_mapping
    )
)

trainer_provisional_mapping_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                TRAINER_PROVISIONAL_MAPPING_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "provisional_trainer_identities": int(
                trainer_provisional_identity_mapping_reloaded[
                    "provisional_trainer_id"
                ].nunique()
            ),
            "mapped_raw_labels": len(
                trainer_provisional_identity_mapping_reloaded
            ),
            "labels_per_identity": 2,
            "reload_validation": "passed",
        }
    ]
)

display(trainer_provisional_mapping_summary)

display(
    trainer_provisional_identity_mapping_reloaded.head(
        10
    )
)

,output_path,provisional_trainer_identities,mapped_raw_labels,labels_per_identity,reload_validation
0,data/processed/trainer_identity/trainer_provis...,26,52,2,passed


,provisional_trainer_id,raw_trainer_label,label_role,strict_comparison_key,identity_status,mapping_method,confidence,database_action
0,TRAINER-PROVISIONAL-0001,Mlle A Budka,earlier_mlle_label,a budka,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
1,TRAINER-PROVISIONAL-0001,Mme A Budka,later_mme_label,a budka,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
2,TRAINER-PROVISIONAL-0002,Mlle A Wattel,earlier_mlle_label,a wattel,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
3,TRAINER-PROVISIONAL-0002,Mme A Wattel,later_mme_label,a wattel,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
4,TRAINER-PROVISIONAL-0003,Mlle A-S Crombez,earlier_mlle_label,a-s crombez,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
5,TRAINER-PROVISIONAL-0003,Mme A-S Crombez,later_mme_label,a-s crombez,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
6,TRAINER-PROVISIONAL-0004,Mlle A-S Pacault,earlier_mlle_label,a-s pacault,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
7,TRAINER-PROVISIONAL-0004,Mme A-S Pacault,later_mme_label,a-s pacault,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
8,TRAINER-PROVISIONAL-0005,Mlle C Gryson,earlier_mlle_label,c gryson,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity
9,TRAINER-PROVISIONAL-0005,Mme C Gryson,later_mme_label,c gryson,provisional_source_label_identity,bounded_2024_mlle_to_mme_source_transition,high_source_label_equivalence,map_raw_label_to_provisional_trainer_identity


In [39]:
# Validate the provisional trainer mapping against the immutable source rows.
#
# This confirms only coverage and join behaviour. It does not add any further
# trainer identities or reinterpret unresolved labels.

mapped_trainer_labels = (
    trainer_provisional_identity_mapping_reloaded[
        "raw_trainer_label"
    ].tolist()
)

mapped_trainer_placeholders = ", ".join(
    ["?"] * len(mapped_trainer_labels)
)

mapped_trainer_source_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        race_id AS source_race_id,
        date,
        course,
        off,
        horse,
        trainer AS raw_trainer_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND trainer IN (
          {mapped_trainer_placeholders}
      )
    """,
    connection,
    params=mapped_trainer_labels,
)

assert not mapped_trainer_source_rows.empty

mapped_trainer_source_rows = (
    mapped_trainer_source_rows.merge(
        trainer_provisional_identity_mapping_reloaded[
            [
                "provisional_trainer_id",
                "raw_trainer_label",
                "label_role",
                "mapping_method",
                "confidence",
            ]
        ],
        on="raw_trainer_label",
        how="left",
        validate="many_to_one",
    )
)

assert mapped_trainer_source_rows[
    "provisional_trainer_id"
].notna().all()

assert mapped_trainer_source_rows[
    "source_rowid"
].is_unique

assert len(mapped_trainer_source_rows) == 6350

assert (
    mapped_trainer_source_rows[
        "provisional_trainer_id"
    ].nunique()
    == 26
)

assert (
    mapped_trainer_source_rows[
        "raw_trainer_label"
    ].nunique()
    == 52
)

trainer_mapping_coverage = (
    mapped_trainer_source_rows.groupby(
        [
            "provisional_trainer_id",
            "label_role",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "source_race_id",
            "nunique",
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "provisional_trainer_id",
            "label_role",
        ]
    )
    .reset_index(drop=True)
)

assert len(trainer_mapping_coverage) == 52

assert (
    trainer_mapping_coverage.groupby(
        "provisional_trainer_id"
    )["label_role"]
    .nunique()
    .eq(2)
    .all()
)

trainer_mapping_validation_summary = pd.DataFrame(
    [
        {
            "mapped_source_rows": len(
                mapped_trainer_source_rows
            ),
            "mapped_raw_labels": int(
                mapped_trainer_source_rows[
                    "raw_trainer_label"
                ].nunique()
            ),
            "provisional_trainer_identities": int(
                mapped_trainer_source_rows[
                    "provisional_trainer_id"
                ].nunique()
            ),
            "duplicate_source_rows": int(
                mapped_trainer_source_rows[
                    "source_rowid"
                ].duplicated().sum()
            ),
            "unmatched_mapping_rows": int(
                mapped_trainer_source_rows[
                    "provisional_trainer_id"
                ].isna().sum()
            ),
            "validation_status": "passed",
        }
    ]
)

display(trainer_mapping_validation_summary)

display(trainer_mapping_coverage.head(12))

,mapped_source_rows,mapped_raw_labels,provisional_trainer_identities,duplicate_source_rows,unmatched_mapping_rows,validation_status
0,6350,52,26,0,0,passed


,provisional_trainer_id,label_role,runner_rows,provisional_races,first_date,last_date
0,TRAINER-PROVISIONAL-0001,earlier_mlle_label,154,152,2020-08-08,2023-12-29
1,TRAINER-PROVISIONAL-0001,later_mme_label,205,197,2024-01-13,2026-05-24
2,TRAINER-PROVISIONAL-0002,earlier_mlle_label,262,255,2018-03-13,2023-12-28
3,TRAINER-PROVISIONAL-0002,later_mme_label,149,139,2024-01-05,2026-05-24
4,TRAINER-PROVISIONAL-0003,earlier_mlle_label,318,306,2019-07-03,2023-12-30
5,TRAINER-PROVISIONAL-0003,later_mme_label,89,88,2024-01-13,2025-11-21
6,TRAINER-PROVISIONAL-0004,earlier_mlle_label,355,329,2015-03-29,2023-12-02
7,TRAINER-PROVISIONAL-0004,later_mme_label,201,183,2024-02-08,2026-05-26
8,TRAINER-PROVISIONAL-0005,earlier_mlle_label,51,51,2015-10-19,2023-12-29
9,TRAINER-PROVISIONAL-0005,later_mme_label,5,5,2024-02-16,2024-08-10


In [40]:
# Persist and reload the source-row coverage generated by the governed
# provisional trainer mapping.

TRAINER_MAPPING_COVERAGE_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_provisional_identity_coverage.csv"
)

trainer_mapping_coverage_to_write = (
    trainer_mapping_coverage.copy()
)

for date_column in (
    "first_date",
    "last_date",
):
    trainer_mapping_coverage_to_write[
        date_column
    ] = pd.to_datetime(
        trainer_mapping_coverage_to_write[
            date_column
        ],
        errors="raise",
    ).dt.strftime("%Y-%m-%d")

trainer_mapping_coverage_to_write.to_csv(
    TRAINER_MAPPING_COVERAGE_PATH,
    index=False,
)

trainer_mapping_coverage_reloaded = pd.read_csv(
    TRAINER_MAPPING_COVERAGE_PATH,
    keep_default_na=False,
)

assert len(trainer_mapping_coverage_reloaded) == 52

assert (
    trainer_mapping_coverage_reloaded[
        "provisional_trainer_id"
    ].nunique()
    == 26
)

assert (
    trainer_mapping_coverage_reloaded[
        "runner_rows"
    ].sum()
    == 6350
)

assert (
    trainer_mapping_coverage_reloaded.groupby(
        "provisional_trainer_id"
    )["label_role"]
    .nunique()
    .eq(2)
    .all()
)

assert (
    trainer_mapping_coverage_reloaded[
        "runner_rows"
    ].gt(0).all()
)

assert (
    trainer_mapping_coverage_reloaded[
        "provisional_races"
    ].gt(0).all()
)

trainer_mapping_coverage_persistence_summary = (
    pd.DataFrame(
        [
            {
                "output_path": str(
                    TRAINER_MAPPING_COVERAGE_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
                "persisted_coverage_rows": len(
                    trainer_mapping_coverage_reloaded
                ),
                "provisional_trainer_identities": int(
                    trainer_mapping_coverage_reloaded[
                        "provisional_trainer_id"
                    ].nunique()
                ),
                "total_mapped_runner_rows": int(
                    trainer_mapping_coverage_reloaded[
                        "runner_rows"
                    ].sum()
                ),
                "reload_validation": "passed",
            }
        ]
    )
)

display(
    trainer_mapping_coverage_persistence_summary
)

,output_path,persisted_coverage_rows,provisional_trainer_identities,total_mapped_runner_rows,reload_validation
0,data/processed/trainer_identity/trainer_provis...,52,26,6350,passed


In [41]:
# Persist the unresolved trainer identity candidates.
#
# These groups remain candidates only. They must not be joined to shared
# provisional trainer identities without additional evidence.

trainer_unresolved_candidates = (
    trainer_title_decisions_reloaded.loc[
        trainer_title_decisions_reloaded[
            "identity_relationship"
        ].eq("unresolved")
    ]
    .copy()
    .sort_values("strict_comparison_key")
    .reset_index(drop=True)
)

assert len(trainer_unresolved_candidates) == 27

assert trainer_unresolved_candidates[
    "strict_comparison_key"
].is_unique

assert trainer_unresolved_candidates[
    "database_action"
].eq("preserve_raw_unresolved").all()

assert trainer_unresolved_candidates[
    "mapping_method"
].eq("").all()

assert trainer_unresolved_candidates[
    "confidence"
].eq("low").all()

# Confirm that no unresolved raw label appears in the accepted mapping.
accepted_raw_trainer_labels = set(
    trainer_provisional_identity_mapping_reloaded[
        "raw_trainer_label"
    ]
)

unresolved_raw_trainer_labels = set(
    trainer_unresolved_candidates[
        "left_raw_trainer_label"
    ]
).union(
    trainer_unresolved_candidates[
        "right_raw_trainer_label"
    ]
)

assert accepted_raw_trainer_labels.isdisjoint(
    unresolved_raw_trainer_labels
)

TRAINER_UNRESOLVED_CANDIDATES_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_unresolved_identity_candidates.csv"
)

trainer_unresolved_candidates.to_csv(
    TRAINER_UNRESOLVED_CANDIDATES_PATH,
    index=False,
)

trainer_unresolved_candidates_reloaded = (
    pd.read_csv(
        TRAINER_UNRESOLVED_CANDIDATES_PATH,
        keep_default_na=False,
    )
)

assert len(
    trainer_unresolved_candidates_reloaded
) == 27

assert trainer_unresolved_candidates_reloaded[
    "strict_comparison_key"
].is_unique

assert trainer_unresolved_candidates_reloaded[
    "identity_relationship"
].eq("unresolved").all()

assert trainer_unresolved_candidates_reloaded[
    "database_action"
].eq("preserve_raw_unresolved").all()

trainer_unresolved_persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                TRAINER_UNRESOLVED_CANDIDATES_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_unresolved_groups": len(
                trainer_unresolved_candidates_reloaded
            ),
            "unresolved_raw_labels": len(
                set(
                    trainer_unresolved_candidates_reloaded[
                        "left_raw_trainer_label"
                    ]
                ).union(
                    trainer_unresolved_candidates_reloaded[
                        "right_raw_trainer_label"
                    ]
                )
            ),
            "overlap_with_accepted_mapping": len(
                accepted_raw_trainer_labels.intersection(
                    unresolved_raw_trainer_labels
                )
            ),
            "reload_validation": "passed",
        }
    ]
)

display(trainer_unresolved_persistence_summary)

display(
    trainer_unresolved_candidates_reloaded[
        [
            "strict_comparison_key",
            "left_raw_trainer_label",
            "right_raw_trainer_label",
            "chronology_status",
            "decision_basis",
            "database_action",
        ]
    ]
)

,output_path,persisted_unresolved_groups,unresolved_raw_labels,overlap_with_accepted_mapping,reload_validation
0,data/processed/trainer_identity/trainer_unreso...,27,54,0,passed


,strict_comparison_key,left_raw_trainer_label,right_raw_trainer_label,chronology_status,decision_basis,database_action
0,a fabre,A Fabre,Mme A Fabre,active_period_overlap,strict title-removal candidate only,preserve_raw_unresolved
1,annabelle sowray,Miss Annabelle Sowray,Annabelle Sowray,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
2,belinda clarke,Mrs Belinda Clarke,Belinda Clarke,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
3,c chenu,Mlle C Chenu,Mme C Chenu,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
4,c herisson de beauvoir,Mlle C Herisson De Beauvoir,Mme C Herisson De Beauvoir,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
5,c walton,Mrs C Walton,Miss C Walton,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
6,delphine domminger,Mlle Delphine Domminger,Mme Delphine Domminger,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
7,emma oliver,Ms Emma Oliver,Mrs Emma Oliver,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
8,g meijer,Mlle G Meijer,Mme G Meijer,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved
9,georgina nicholls,Ms Georgina Nicholls,Georgina Nicholls,separate_active_periods,strict title-removal candidate only,preserve_raw_unresolved


In [42]:
# Create and persist the final trainer-identity governance summary.
#
# This provides a compact audit record of what was mapped, what remains
# unresolved, and what the governed outputs cover.

trainer_identity_governance_summary = pd.DataFrame(
    [
        {
            "metric": "distinct_populated_raw_trainer_labels",
            "value": 10708,
            "status": "profiled",
            "governance_meaning": (
                "Immutable populated trainer labels observed in the source"
            ),
        },
        {
            "metric": "raw_blank_trainer_rows",
            "value": 9,
            "status": "reconciled_with_notebook_20",
            "governance_meaning": (
                "Four externally supplemented and five preserved unresolved"
            ),
        },
        {
            "metric": "strict_title_candidate_groups",
            "value": len(trainer_title_decisions_reloaded),
            "status": "reviewed",
            "governance_meaning": (
                "Candidates generated by removing one recognised leading title"
            ),
        },
        {
            "metric": "accepted_provisional_trainer_identities",
            "value": int(
                trainer_provisional_identity_mapping_reloaded[
                    "provisional_trainer_id"
                ].nunique()
            ),
            "status": "governed",
            "governance_meaning": (
                "Bounded 2024 Mlle-to-Mme source-label transitions"
            ),
        },
        {
            "metric": "accepted_raw_trainer_labels",
            "value": len(
                trainer_provisional_identity_mapping_reloaded
            ),
            "status": "mapped",
            "governance_meaning": (
                "Two immutable raw labels per provisional trainer identity"
            ),
        },
        {
            "metric": "mapped_source_runner_rows",
            "value": int(
                trainer_mapping_coverage_reloaded[
                    "runner_rows"
                ].sum()
            ),
            "status": "validated",
            "governance_meaning": (
                "Source rows covered by the accepted provisional mappings"
            ),
        },
        {
            "metric": "unresolved_candidate_groups",
            "value": len(
                trainer_unresolved_candidates_reloaded
            ),
            "status": "preserved_unresolved",
            "governance_meaning": (
                "No shared identity may be inferred without further evidence"
            ),
        },
        {
            "metric": "unresolved_raw_trainer_labels",
            "value": len(
                set(
                    trainer_unresolved_candidates_reloaded[
                        "left_raw_trainer_label"
                    ]
                ).union(
                    trainer_unresolved_candidates_reloaded[
                        "right_raw_trainer_label"
                    ]
                )
            ),
            "status": "preserved_unresolved",
            "governance_meaning": (
                "Raw labels excluded from the accepted provisional mapping"
            ),
        },
    ]
)

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("strict_title_candidate_groups"),
    "value",
].item() == 53

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("accepted_provisional_trainer_identities"),
    "value",
].item() == 26

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("accepted_raw_trainer_labels"),
    "value",
].item() == 52

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("mapped_source_runner_rows"),
    "value",
].item() == 6350

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("unresolved_candidate_groups"),
    "value",
].item() == 27

assert trainer_identity_governance_summary.loc[
    trainer_identity_governance_summary[
        "metric"
    ].eq("unresolved_raw_trainer_labels"),
    "value",
].item() == 54

TRAINER_GOVERNANCE_SUMMARY_PATH = (
    TRAINER_IDENTITY_OUTPUT_DIR
    / "trainer_identity_governance_summary.csv"
)

trainer_identity_governance_summary.to_csv(
    TRAINER_GOVERNANCE_SUMMARY_PATH,
    index=False,
)

trainer_identity_governance_summary_reloaded = pd.read_csv(
    TRAINER_GOVERNANCE_SUMMARY_PATH,
    keep_default_na=False,
)

assert trainer_identity_governance_summary_reloaded.equals(
    trainer_identity_governance_summary
)

display(trainer_identity_governance_summary_reloaded)

,metric,value,status,governance_meaning
0,distinct_populated_raw_trainer_labels,10708,profiled,Immutable populated trainer labels observed in...
1,raw_blank_trainer_rows,9,reconciled_with_notebook_20,Four externally supplemented and five preserve...
2,strict_title_candidate_groups,53,reviewed,Candidates generated by removing one recognise...
3,accepted_provisional_trainer_identities,26,governed,Bounded 2024 Mlle-to-Mme source-label transitions
4,accepted_raw_trainer_labels,52,mapped,Two immutable raw labels per provisional train...
5,mapped_source_runner_rows,6350,validated,Source rows covered by the accepted provisiona...
6,unresolved_candidate_groups,27,preserved_unresolved,No shared identity may be inferred without fur...
7,unresolved_raw_trainer_labels,54,preserved_unresolved,Raw labels excluded from the accepted provisio...


### Trainer identity conclusion

The trainer field is materially more usable than the jockey field, but it still does not support unrestricted name normalisation.

The source contains:

* 10,708 distinct populated raw trainer labels;
* nine blank trainer rows already governed through Notebook 20;
* 53 strict title-derived candidate groups;
* 106 raw labels involved in those candidate groups.

The investigation identified one narrow, defensible source-level rule.

A marked presentation change occurred around January 2024, when `Mlle` usage collapsed and `Mme` usage increased sharply. Twenty-six exact-name, non-overlapping `Mlle → Mme` pairs align with that transition window.

Those 26 pairs:

* create 26 provisional trainer identities;
* govern 52 immutable raw labels;
* cover 6,350 source runner rows;
* contain no duplicate source-row joins;
* contain no unmatched mapping rows.

The accepted relationship is **source-label equivalence**, not independently verified legal or licensing identity.

The remaining 27 candidate groups, containing 54 raw labels, remain unresolved. They include overlapping labels, English and German title variants, and `Mlle/Mme` recurrences outside the bounded 2024 transition window.

Therefore:

1. raw trainer labels remain immutable;
2. only the 26 governed pairs may share provisional trainer identities;
3. the mapping method is `bounded_2024_mlle_to_mme_source_transition`;
4. all other title-derived candidates remain explicitly unresolved;
5. no general trainer title-stripping or automatic identity-merging rule is authorised;
6. unresolved cases should be revisited only when they materially affect a specific article, analysis or participant.

This is the analytical stopping point for trainer identity in Notebook 22.

## Owner identity

The owner field can represent individuals, partnerships, syndicates, companies, clubs and other organisations.

This makes owner identity structurally different from jockey and trainer identity. A shared surname, punctuation change or removed title cannot be assumed to identify the same ownership entity.

The first stage therefore profiles the immutable raw owner labels before testing any bounded comparison rules.

In [43]:
# Profile the immutable raw owner field.

owner_source_profile = pd.read_sql_query(
    f"""
    SELECT
        COUNT(*) AS governed_runner_rows,
        SUM(
            CASE
                WHEN owner IS NULL THEN 1
                ELSE 0
            END
        ) AS sql_null_owner_rows,
        SUM(
            CASE
                WHEN owner IS NOT NULL
                 AND TRIM(owner) = ''
                THEN 1
                ELSE 0
            END
        ) AS empty_string_owner_rows,
        SUM(
            CASE
                WHEN owner IS NOT NULL
                 AND TRIM(owner) <> ''
                THEN 1
                ELSE 0
            END
        ) AS populated_owner_rows,
        COUNT(
            DISTINCT CASE
                WHEN owner IS NOT NULL
                 AND TRIM(owner) <> ''
                THEN owner
            END
        ) AS distinct_populated_raw_owner_labels
    FROM data
    WHERE {DATA_ROW_PREDICATE}
    """,
    connection,
)

owner_label_frequency = pd.read_sql_query(
    f"""
    SELECT
        owner AS raw_owner_label,
        COUNT(*) AS runner_rows,
        COUNT(DISTINCT race_id) AS provisional_races,
        MIN(date) AS first_date,
        MAX(date) AS last_date
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND owner IS NOT NULL
      AND TRIM(owner) <> ''
    GROUP BY owner
    ORDER BY
        runner_rows DESC,
        raw_owner_label
    """,
    connection,
)

assert len(owner_source_profile) == 1

assert (
    owner_source_profile[
        "governed_runner_rows"
    ].item()
    == 1851285
)

assert (
    owner_source_profile[
        "populated_owner_rows"
    ].item()
    + owner_source_profile[
        "sql_null_owner_rows"
    ].item()
    + owner_source_profile[
        "empty_string_owner_rows"
    ].item()
    == owner_source_profile[
        "governed_runner_rows"
    ].item()
)

assert (
    len(owner_label_frequency)
    == owner_source_profile[
        "distinct_populated_raw_owner_labels"
    ].item()
)

assert owner_label_frequency[
    "raw_owner_label"
].is_unique

assert owner_label_frequency[
    "runner_rows"
].sum() == owner_source_profile[
    "populated_owner_rows"
].item()

display(owner_source_profile)

display(owner_label_frequency.head(30))

,governed_runner_rows,sql_null_owner_rows,empty_string_owner_rows,populated_owner_rows,distinct_populated_raw_owner_labels
0,1851285,0,35,1851250,98234


,raw_owner_label,runner_rows,provisional_races,first_date,last_date
0,John P Mcmanus,15384,11472,2015-01-01,2026-05-27
1,Godolphin,13416,10644,2015-01-01,2026-05-27
2,Gigginstown House Stud,6998,5054,2015-01-01,2026-05-27
3,Hamdan Al Maktoum,6867,5346,2015-01-04,2021-03-23
4,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,3934,2015-01-04,2026-05-27
5,Mrs J S Bolger,3779,3443,2015-01-02,2026-05-26
6,Simon Munir Isaac Souede,3410,3242,2015-01-01,2026-05-26
7,Cheveley Park Stud,3283,3132,2015-01-11,2026-05-23
8,H H Aga Khan,2970,2728,2015-01-09,2025-03-04
9,Wertheimer Frere,2797,2519,2015-01-12,2026-05-24


### Owner blanks

The owner field contains 35 empty-string rows and no SQL-null rows.

Before investigating populated owner labels, these blank rows must be reconciled with any earlier manual verification or supplementation work. A blank owner cannot be interpreted as “no owner”; it means that no usable owner label was retained in this source row.

In [44]:
# Inspect every governed source row with a blank owner label.

blank_owner_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        race_id AS source_race_id,
        date,
        course,
        off,
        race_name,
        horse,
        trainer,
        jockey,
        owner AS raw_owner_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND owner IS NOT NULL
      AND TRIM(owner) = ''
    ORDER BY
        date,
        course,
        off,
        source_rowid
    """,
    connection,
)

assert len(blank_owner_rows) == 35

assert blank_owner_rows[
    "source_rowid"
].is_unique

assert blank_owner_rows[
    "raw_owner_label"
].eq("").all()

blank_owner_summary = pd.DataFrame(
    [
        {
            "blank_owner_rows": len(
                blank_owner_rows
            ),
            "distinct_races": int(
                blank_owner_rows[
                    "source_race_id"
                ].nunique()
            ),
            "distinct_horses": int(
                blank_owner_rows[
                    "horse"
                ].nunique()
            ),
            "first_date": blank_owner_rows[
                "date"
            ].min(),
            "last_date": blank_owner_rows[
                "date"
            ].max(),
        }
    ]
)

display(blank_owner_summary)

display(blank_owner_rows)

,blank_owner_rows,distinct_races,distinct_horses,first_date,last_date
0,35,21,33,2015-01-03,2022-11-13


,source_rowid,source_race_id,date,course,off,race_name,horse,trainer,jockey,raw_owner_label
0,1443,617158,2015-01-03,Santa Anita (USA),11:30,Santa Ynez Stakes (Fillies) (Dirt),Rattataptap (USA),Jeff Bonde,Edwin A Maldonado,
1,33009,623280,2015-04-04,Caulfield (AUS),6:10,Le Pine Funerals Easter Cup Handicap) (2yo+) (...,Post DFrance (NZ),Stephen Brown,Patrick Moloney,
2,48891,631223,2015-05-03,San Siro (ITY),12:07,Premio Razza Ticino (Turf),Balami Fan (ITY),V Cangiano,Luca Maniezzi,
3,71791,630083,2015-06-18,Longchamp (FR),11:15,Prix de Bougival (Maiden) (Unraced 3yo Colts &...,Rappeur Des Mottes (FR),E Lellouche,Anthony Crastus,
4,203870,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Star White (FR),,Tony Piccone,
5,203991,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Colombia DEmra (FR),,Richard Juteau,
6,712612,730833,2019-05-13,Saint-Cloud (FR),4:55,Prix de la Vallee du Lot (Handicap) (4yo+) (Turf),Le Letty (FR),Mme M-C Chaalon,Alexandre Chesneau,
7,725030,732516,2019-06-03,Longchamp (FR),4:55,Prix de la Porte de la Seine (Handicap) (4yo+)...,Le Letty (FR),Mme M-C Chaalon,Louis-Philippe Beuzelin,
8,801287,744785,2019-10-31,Mombetsu (JPN),11:07,Hokkaido Nisai Yushun (Local (Dirt) (2yo),Abenin Dream (JPN),Hideki Kakugawa,Ryu Abe,
9,801290,744785,2019-10-31,Mombetsu (JPN),11:07,Hokkaido Nisai Yushun (Local (Dirt) (2yo),Chimera Verite (JPN),Kazuya Nakatake,Yuichi Fukunaga,


In [49]:
# Reconcile the 35 blank owner rows against the permanent governed
# manual-verification register used by Notebook 20.
#
# The register does not have a dedicated source_rowid column. Notebook 20
# preserves the physical source row identifier inside raw_source_value.

MANUAL_VERIFICATIONS_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "manual_verifications.csv"
)

assert MANUAL_VERIFICATIONS_PATH.exists()

manual_verifications = pd.read_csv(
    MANUAL_VERIFICATIONS_PATH,
    keep_default_na=False,
)

required_manual_verification_columns = {
    "verification_id",
    "source_field",
    "raw_source_value",
    "verified_value",
    "verification_status",
    "governing_notebook",
    "confidence",
    "notes",
    "database_action",
}

assert required_manual_verification_columns.issubset(
    manual_verifications.columns
)

notebook_20_owner_verifications = (
    manual_verifications.loc[
        manual_verifications[
            "governing_notebook"
        ].astype(str).eq("20")
        & manual_verifications[
            "source_field"
        ].eq("owner")
    ]
    .copy()
    .reset_index(drop=True)
)

notebook_20_owner_verifications[
    "source_rowid"
] = pd.to_numeric(
    notebook_20_owner_verifications[
        "raw_source_value"
    ].str.extract(
        r"source_rowid=(\d+)"
    )[0],
    errors="raise",
).astype(int)

assert notebook_20_owner_verifications[
    "verification_id"
].is_unique

assert notebook_20_owner_verifications[
    "source_rowid"
].is_unique

blank_owner_reconciliation = (
    blank_owner_rows.merge(
        notebook_20_owner_verifications[
            [
                "source_rowid",
                "verification_id",
                "verified_value",
                "verification_status",
                "confidence",
                "database_action",
                "notes",
            ]
        ],
        on="source_rowid",
        how="left",
        validate="one_to_one",
    )
)

# The register was loaded with keep_default_na=False, but unmatched merge
# values are still represented as NaN. Fill them before status checks.
blank_owner_reconciliation[
    [
        "verification_id",
        "verified_value",
        "verification_status",
        "confidence",
        "database_action",
        "notes",
    ]
] = blank_owner_reconciliation[
    [
        "verification_id",
        "verified_value",
        "verification_status",
        "confidence",
        "database_action",
        "notes",
    ]
].fillna("")

blank_owner_reconciliation[
    "existing_governance_status"
] = blank_owner_reconciliation[
    "verification_id"
].ne("").map(
    {
        True: "governed_by_notebook_20",
        False: "not_found_in_notebook_20_register",
    }
)

blank_owner_reconciliation_summary = pd.DataFrame(
    [
        {
            "raw_blank_owner_rows": len(
                blank_owner_rows
            ),
            "notebook_20_owner_verifications": len(
                notebook_20_owner_verifications
            ),
            "matched_blank_owner_rows": int(
                blank_owner_reconciliation[
                    "verification_id"
                ].ne("").sum()
            ),
            "confirmed_owner_supplementations": int(
                blank_owner_reconciliation[
                    "database_action"
                ].eq(
                    "source_supplementation"
                ).sum()
            ),
            "preserved_unresolved_owner_rows": int(
                blank_owner_reconciliation[
                    "database_action"
                ].eq(
                    "preserve_raw_unresolved"
                ).sum()
            ),
            "unmatched_blank_owner_rows": int(
                blank_owner_reconciliation[
                    "verification_id"
                ].eq("").sum()
            ),
        }
    ]
)

assert (
    blank_owner_reconciliation_summary[
        "matched_blank_owner_rows"
    ].item()
    + blank_owner_reconciliation_summary[
        "unmatched_blank_owner_rows"
    ].item()
    == len(blank_owner_rows)
)

display(blank_owner_reconciliation_summary)

display(
    blank_owner_reconciliation[
        [
            "source_rowid",
            "source_race_id",
            "date",
            "course",
            "horse",
            "raw_owner_label",
            "verification_id",
            "verified_value",
            "verification_status",
            "confidence",
            "database_action",
            "existing_governance_status",
        ]
    ]
)

,raw_blank_owner_rows,notebook_20_owner_verifications,matched_blank_owner_rows,confirmed_owner_supplementations,preserved_unresolved_owner_rows,unmatched_blank_owner_rows
0,35,35,35,22,13,0


,source_rowid,source_race_id,date,course,horse,raw_owner_label,verification_id,verified_value,verification_status,confidence,database_action,existing_governance_status
0,1443,617158,2015-01-03,Santa Anita (USA),Rattataptap (USA),,NB20-CONNECTION-0003,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
1,33009,623280,2015-04-04,Caulfield (AUS),Post DFrance (NZ),,NB20-CONNECTION-0004,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
2,48891,631223,2015-05-03,San Siro (ITY),Balami Fan (ITY),,NB20-CONNECTION-0005,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
3,71791,630083,2015-06-18,Longchamp (FR),Rappeur Des Mottes (FR),,NB20-CONNECTION-0006,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
4,203870,651027,2016-05-07,Maisons-Laffitte (FR),Star White (FR),,NB20-CONNECTION-0007,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
5,203991,651027,2016-05-07,Maisons-Laffitte (FR),Colombia DEmra (FR),,NB20-CONNECTION-0008,,unresolved,low,preserve_raw_unresolved,governed_by_notebook_20
6,712612,730833,2019-05-13,Saint-Cloud (FR),Le Letty (FR),,NB20-CONNECTION-0009,Edouard Desespringalle,confirmed,high,source_supplementation,governed_by_notebook_20
7,725030,732516,2019-06-03,Longchamp (FR),Le Letty (FR),,NB20-CONNECTION-0010,Edouard Desespringalle,confirmed,high,source_supplementation,governed_by_notebook_20
8,801287,744785,2019-10-31,Mombetsu (JPN),Abenin Dream (JPN),,NB20-CONNECTION-0011,Mastec Co Ltd,confirmed,high,source_supplementation,governed_by_notebook_20
9,801290,744785,2019-10-31,Mombetsu (JPN),Chimera Verite (JPN),,NB20-CONNECTION-0012,Makoto Kato,confirmed,high,source_supplementation,governed_by_notebook_20


### Owner blank-field reconciliation

All 35 empty-string owner rows were previously governed through Notebook 20.

The permanent manual-verification register contains one matching record for every blank source row:

* 22 rows have confirmed externally verified owners and are authorised for `source_supplementation`;
* 13 rows remain unresolved and are authorised only for `preserve_raw_unresolved`;
* no blank owner row remains outside the existing governance record.

Notebook 23 therefore inherits these decisions without repeating the external research.

The immutable raw owner value remains blank. Any verified owner must be added only through the governed supplementation layer with its Notebook 20 verification identifier, evidence method and confidence preserved.

### Owner-label structural profile

Owner labels may describe individuals, jointly named people, syndicates, partnerships, companies, studs, racing clubs and other organisations.

Simple token removal cannot safely resolve those entities. The next step therefore measures observable label features without treating them as definitive classifications.

These features are candidate indicators only. A word such as `Stud`, `Racing`, `Ltd`, `Syndicate` or `Partnership` may support later entity-type rules, while titles and joining words may indicate personal or shared ownership. None of those markers proves identity by itself.

In [50]:
# Profile observable structural features in populated raw owner labels.
#
# The categories are deliberately non-exclusive. A label may contain several
# markers, such as both "Racing" and "Ltd". This cell measures candidate
# structure only and does not yet assign one definitive entity type.

owner_structure_profile = owner_label_frequency.copy()

owner_structure_profile[
    "comparison_label"
] = (
    owner_structure_profile[
        "raw_owner_label"
    ]
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.casefold()
)

owner_structure_profile[
    "token_count"
] = owner_structure_profile[
    "comparison_label"
].str.split().str.len()

owner_structure_profile[
    "contains_person_title"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
    r"^(mr|mrs|miss|ms|dr|sir|lady|lord|sheikh|"
    r"hh|h h|hrh|h r h|mme|mlle|frau)\b",
    regex=True,
)

owner_structure_profile[
    "contains_company_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(ltd|limited|plc|inc|incorporated|llc|"
        r"corp|corporation|company|co|snc|sa|sas|"
        r"gmbh|pty)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_syndicate_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(syndicate|syndicates)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_partnership_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(partnership|partners)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_club_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(club|society|association)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_stud_or_farm_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(stud|farm|farms|stable|stables)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_racing_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(racing|racehorses)\b",
        regex=True,
    )

owner_structure_profile[
    "contains_joint_owner_separator"
] = owner_structure_profile[
    "raw_owner_label"
].str.contains(
        r"(&|,|/|\+|\band\b)",
        case=False,
        regex=True,
    )

owner_structure_profile[
    "contains_possessive_or_family_marker"
] = owner_structure_profile[
    "comparison_label"
].str.contains(
        r"\b(family|estate of|executors of|trust|trustees)\b",
        regex=True,
    )

owner_structure_metrics = [
    "contains_person_title",
    "contains_company_marker",
    "contains_syndicate_marker",
    "contains_partnership_marker",
    "contains_club_marker",
    "contains_stud_or_farm_marker",
    "contains_racing_marker",
    "contains_joint_owner_separator",
    "contains_possessive_or_family_marker",
]

owner_structure_summary_rows = []

for metric in owner_structure_metrics:
    metric_mask = owner_structure_profile[
        metric
    ]

    owner_structure_summary_rows.append(
        {
            "structural_marker": metric,
            "distinct_raw_labels": int(
                metric_mask.sum()
            ),
            "runner_rows": int(
                owner_structure_profile.loc[
                    metric_mask,
                    "runner_rows",
                ].sum()
            ),
            "share_of_distinct_labels_pct": round(
                100
                * metric_mask.mean(),
                2,
            ),
            "share_of_populated_runner_rows_pct": round(
                100
                * owner_structure_profile.loc[
                    metric_mask,
                    "runner_rows",
                ].sum()
                / owner_source_profile[
                    "populated_owner_rows"
                ].item(),
                2,
            ),
        }
    )

owner_structure_summary = (
    pd.DataFrame(
        owner_structure_summary_rows
    )
    .sort_values(
        [
            "distinct_raw_labels",
            "structural_marker",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

owner_token_count_summary = (
    owner_structure_profile.groupby(
        "token_count",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=(
            "raw_owner_label",
            "size",
        ),
        runner_rows=(
            "runner_rows",
            "sum",
        ),
    )
    .sort_values("token_count")
    .reset_index(drop=True)
)

assert len(owner_structure_profile) == 98234

assert owner_structure_profile[
    "raw_owner_label"
].is_unique

assert owner_structure_profile[
    "runner_rows"
].sum() == 1851250

display(owner_structure_summary)

display(owner_token_count_summary.head(15))

display(
    owner_structure_profile[
        [
            "raw_owner_label",
            "runner_rows",
            "token_count",
            *owner_structure_metrics,
        ]
    ].head(40)
)

/tmp/ipykernel_741168/1754194681.py:34: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(
/tmp/ipykernel_741168/1754194681.py:44: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(
/tmp/ipykernel_741168/1754194681.py:55: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(
/tmp/ipykernel_741168/1754194681.py:64: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(
/tmp/ipykernel_741168/1754194681.py:73: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  ].str.contains(
/tmp/ipykernel_741168/1754194681.py:82: UserW

,structural_marker,distinct_raw_labels,runner_rows,share_of_distinct_labels_pct,share_of_populated_runner_rows_pct
0,contains_racing_marker,12112,215673,12.33,11.65
1,contains_person_title,11540,265937,11.75,14.37
2,contains_company_marker,7267,132719,7.40,7.17
3,contains_stud_or_farm_marker,6239,77378,6.35,4.18
4,contains_syndicate_marker,5806,96207,5.91,5.20
5,contains_partnership_marker,5099,85818,5.19,4.64
6,contains_joint_owner_separator,2710,41874,2.76,2.26
7,contains_club_marker,1363,43914,1.39,2.37
8,contains_possessive_or_family_marker,646,9904,0.66,0.53


,token_count,distinct_raw_labels,runner_rows
0,1,993,30232
1,2,20186,513438
2,3,22691,625719
3,4,15841,328218
4,5,10127,143198
5,6,9240,89618
6,7,7119,51890
7,8,6103,35075
8,9,3390,16521
9,10,1529,9028


,raw_owner_label,runner_rows,token_count,contains_person_title,contains_company_marker,contains_syndicate_marker,contains_partnership_marker,contains_club_marker,contains_stud_or_farm_marker,contains_racing_marker,contains_joint_owner_separator,contains_possessive_or_family_marker
0,John P Mcmanus,15384,3,False,False,False,False,False,False,False,False,False
1,Godolphin,13416,1,False,False,False,False,False,False,False,False,False
2,Gigginstown House Stud,6998,3,False,False,False,False,False,True,False,False,False
3,Hamdan Al Maktoum,6867,3,False,False,False,False,False,False,False,False,False
4,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,6,True,False,False,False,False,False,False,False,False
5,Mrs J S Bolger,3779,4,True,False,False,False,False,False,False,False,False
6,Simon Munir Isaac Souede,3410,4,False,False,False,False,False,False,False,False,False
7,Cheveley Park Stud,3283,3,False,False,False,False,False,True,False,False,False
8,H H Aga Khan,2970,4,True,False,False,False,False,False,False,False,False
9,Wertheimer Frere,2797,2,False,False,False,False,False,False,False,False,False


In [51]:
# Test whether owner labels contain exactly the same words in a different
# order or with punctuation differences.
#
# This is candidate generation only. Matching token multisets do not prove
# that two raw labels represent the same ownership entity.

owner_token_candidates = owner_label_frequency.copy()

owner_token_candidates[
    "tokenised_comparison_label"
] = (
    owner_token_candidates[
        "raw_owner_label"
    ]
    .str.casefold()
    .str.replace(
        r"[^a-z0-9]+",
        " ",
        regex=True,
    )
    .str.strip()
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)

owner_token_candidates[
    "comparison_tokens"
] = owner_token_candidates[
    "tokenised_comparison_label"
].str.split()

owner_token_candidates[
    "token_multiset_key"
] = owner_token_candidates[
    "comparison_tokens"
].apply(
    lambda tokens: " | ".join(
        sorted(tokens)
    )
)

owner_token_candidates[
    "token_count"
] = owner_token_candidates[
    "comparison_tokens"
].str.len()

owner_token_group_profile = (
    owner_token_candidates.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .agg(
        raw_label_count=(
            "raw_owner_label",
            "nunique",
        ),
        combined_runner_rows=(
            "runner_rows",
            "sum",
        ),
        token_count=(
            "token_count",
            "first",
        ),
        first_date=(
            "first_date",
            "min",
        ),
        last_date=(
            "last_date",
            "max",
        ),
    )
)

owner_token_collision_groups = (
    owner_token_group_profile.loc[
        owner_token_group_profile[
            "raw_label_count"
        ].gt(1)
    ]
    .copy()
    .sort_values(
        [
            "combined_runner_rows",
            "raw_label_count",
            "token_multiset_key",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

owner_token_collision_labels = (
    owner_token_candidates.merge(
        owner_token_collision_groups[
            [
                "token_multiset_key",
                "raw_label_count",
                "combined_runner_rows",
            ]
        ],
        on="token_multiset_key",
        how="inner",
        validate="many_to_one",
    )
    .sort_values(
        [
            "combined_runner_rows",
            "token_multiset_key",
            "runner_rows",
            "raw_owner_label",
        ],
        ascending=[
            False,
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

owner_token_collision_summary = pd.DataFrame(
    [
        {
            "distinct_populated_owner_labels": len(
                owner_label_frequency
            ),
            "token_multiset_candidate_groups": len(
                owner_token_collision_groups
            ),
            "candidate_raw_labels": len(
                owner_token_collision_labels
            ),
            "candidate_runner_rows": int(
                owner_token_collision_labels[
                    "runner_rows"
                ].sum()
            ),
            "largest_candidate_group": (
                int(
                    owner_token_collision_groups[
                        "raw_label_count"
                    ].max()
                )
                if not owner_token_collision_groups.empty
                else 0
            ),
        }
    ]
)

assert owner_token_candidates[
    "raw_owner_label"
].is_unique

assert owner_token_candidates[
    "token_multiset_key"
].ne("").all()

assert owner_token_collision_labels[
    "raw_owner_label"
].is_unique

assert owner_token_collision_labels[
    "token_multiset_key"
].nunique() == len(
    owner_token_collision_groups
)

display(owner_token_collision_summary)

display(owner_token_collision_groups.head(30))

display(
    owner_token_collision_labels[
        [
            "token_multiset_key",
            "raw_owner_label",
            "runner_rows",
            "first_date",
            "last_date",
            "raw_label_count",
            "combined_runner_rows",
        ]
    ].head(100)
)

,distinct_populated_owner_labels,token_multiset_candidate_groups,candidate_raw_labels,candidate_runner_rows,largest_candidate_group
0,98234,936,1917,34194,6


,token_multiset_key,raw_label_count,combined_runner_rows,token_count,first_date,last_date
0,derrick | john | magnier | michael | mrs | smi...,6,5052,7,2015-01-23,2026-05-25
1,d | j | m | magnier | mrs | smith | tabor | we...,5,851,8,2020-08-08,2026-05-24
2,anoj | daniel | don | macauliffe,2,696,4,2015-06-12,2026-05-21
3,heather | michael | yarrow,2,538,3,2015-01-09,2026-05-15
4,dineen | hughes | kerr | martin | michael,2,418,5,2015-04-13,2026-05-25
5,julie | martin | phil,2,325,3,2018-02-03,2023-04-22
6,katsumi | yoshida,2,324,2,2015-01-04,2026-05-24
7,aguiar | amo | de | giselle | limited | racing,2,310,6,2022-03-27,2026-05-26
8,david | jones | lynne | lyons | mrs | sean | s...,3,306,7,2019-03-15,2026-05-16
9,andy | bell | fergus | lyons,2,276,4,2016-11-19,2026-05-12


,token_multiset_key,raw_owner_label,runner_rows,first_date,last_date,raw_label_count,combined_runner_rows
0,derrick | john | magnier | michael | mrs | smi...,Mrs John Magnier Michael Tabor Derrick Smith,1673,2015-03-29,2026-05-19,6,5052
1,derrick | john | magnier | michael | mrs | smi...,Michael Tabor Derrick Smith Mrs John Magnier,1622,2015-03-07,2026-05-24,6,5052
2,derrick | john | magnier | michael | mrs | smi...,Derrick Smith Mrs John Magnier Michael Tabor,1609,2015-03-15,2026-05-25,6,5052
3,derrick | john | magnier | michael | mrs | smi...,Michael Tabor Mrs John Magnier Derrick Smith,116,2015-01-23,2026-02-27,6,5052
4,derrick | john | magnier | michael | mrs | smi...,Derrick Smith Michael Tabor Mrs John Magnier,29,2015-02-28,2024-06-26,6,5052
...,...,...,...,...,...,...,...
95,carmel | stud,Carmel Stud,116,2015-03-29,2024-11-14,2,126
96,carmel | stud,Stud Carmel,10,2017-05-25,2025-10-11,2,126
97,fyffe | fyffe | james | scott,James Fyffe Scott Fyffe,98,2018-08-11,2026-03-28,2,121
98,fyffe | fyffe | james | scott,Scott Fyffe James Fyffe,23,2022-04-09,2024-08-23,2,121


In [52]:
# Separate exact token-sequence variants from genuine token-order variants.
#
# Sequence variants differ only through punctuation, case or spacing.
# Reordered variants contain exactly the same token multiset but place one or
# more tokens in a different order.
#
# Both remain candidate relationships at this stage.

owner_token_collision_labels[
    "token_sequence_key"
] = owner_token_collision_labels[
    "comparison_tokens"
].apply(
    lambda tokens: " | ".join(tokens)
)

owner_token_group_structure = (
    owner_token_collision_labels.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .agg(
        raw_label_count=(
            "raw_owner_label",
            "nunique",
        ),
        token_sequence_count=(
            "token_sequence_key",
            "nunique",
        ),
        combined_runner_rows=(
            "runner_rows",
            "sum",
        ),
        first_date=(
            "first_date",
            "min",
        ),
        last_date=(
            "last_date",
            "max",
        ),
    )
)

owner_token_group_structure[
    "candidate_structure"
] = owner_token_group_structure[
    "token_sequence_count"
].eq(1).map(
    {
        True: "same_token_sequence",
        False: "reordered_token_sequence",
    }
)

owner_token_structure_summary = (
    owner_token_group_structure.groupby(
        "candidate_structure",
        as_index=False,
    )
    .agg(
        candidate_groups=(
            "token_multiset_key",
            "size",
        ),
        candidate_raw_labels=(
            "raw_label_count",
            "sum",
        ),
        candidate_runner_rows=(
            "combined_runner_rows",
            "sum",
        ),
        largest_group=(
            "raw_label_count",
            "max",
        ),
    )
    .sort_values(
        "candidate_structure"
    )
    .reset_index(drop=True)
)

owner_token_collision_labels = (
    owner_token_collision_labels.merge(
        owner_token_group_structure[
            [
                "token_multiset_key",
                "token_sequence_count",
                "candidate_structure",
            ]
        ],
        on="token_multiset_key",
        how="left",
        validate="many_to_one",
    )
)

assert len(owner_token_group_structure) == 936

assert (
    owner_token_group_structure[
        "candidate_structure"
    ]
    .isin(
        {
            "same_token_sequence",
            "reordered_token_sequence",
        }
    )
    .all()
)

assert owner_token_collision_labels[
    "candidate_structure"
].notna().all()

display(owner_token_structure_summary)

display(
    owner_token_group_structure.loc[
        owner_token_group_structure[
            "candidate_structure"
        ].eq("same_token_sequence")
    ]
    .sort_values(
        [
            "combined_runner_rows",
            "raw_label_count",
        ],
        ascending=False,
    )
    .head(30)
)

display(
    owner_token_collision_labels.loc[
        owner_token_collision_labels[
            "candidate_structure"
        ].eq("same_token_sequence"),
        [
            "token_multiset_key",
            "raw_owner_label",
            "runner_rows",
            "first_date",
            "last_date",
            "raw_label_count",
        ],
    ].head(100)
)

display(
    owner_token_group_structure.loc[
        owner_token_group_structure[
            "candidate_structure"
        ].eq("reordered_token_sequence")
    ]
    .sort_values(
        [
            "combined_runner_rows",
            "raw_label_count",
        ],
        ascending=False,
    )
    .head(30)
)

,candidate_structure,candidate_groups,candidate_raw_labels,candidate_runner_rows,largest_group
0,reordered_token_sequence,936,1917,34194,6


,token_multiset_key,raw_label_count,token_sequence_count,combined_runner_rows,first_date,last_date,candidate_structure


,token_multiset_key,raw_owner_label,runner_rows,first_date,last_date,raw_label_count


,token_multiset_key,raw_label_count,token_sequence_count,combined_runner_rows,first_date,last_date,candidate_structure
706,derrick | john | magnier | michael | mrs | smi...,6,6,5052,2015-01-23,2026-05-25,reordered_token_sequence
670,d | j | m | magnier | mrs | smith | tabor | we...,5,5,851,2020-08-08,2026-05-24,reordered_token_sequence
305,anoj | daniel | don | macauliffe,2,2,696,2015-06-12,2026-05-21,reordered_token_sequence
822,heather | michael | yarrow,2,2,538,2015-01-09,2026-05-15,reordered_token_sequence
724,dineen | hughes | kerr | martin | michael,2,2,418,2015-04-13,2026-05-25,reordered_token_sequence
891,julie | martin | phil,2,2,325,2018-02-03,2023-04-22,reordered_token_sequence
894,katsumi | yoshida,2,2,324,2015-01-04,2026-05-24,reordered_token_sequence
111,aguiar | amo | de | giselle | limited | racing,2,2,310,2022-03-27,2026-05-26,reordered_token_sequence
693,david | jones | lynne | lyons | mrs | sean | s...,3,3,306,2019-03-15,2026-05-16,reordered_token_sequence
297,andy | bell | fergus | lyons,2,2,276,2016-11-19,2026-05-12,reordered_token_sequence


In [53]:
# Test whether reordered owner-label variants occur within the same race.
#
# Same-race use of two or more raw labels with the exact same token multiset
# is strong source-internal evidence that token order is presentation noise.
#
# This still establishes ownership-composition equivalence, not verified
# legal identity for every named person or organisation inside the label.

owner_candidate_source_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        race_id AS source_race_id,
        date,
        course,
        off,
        horse,
        owner AS raw_owner_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND owner IS NOT NULL
      AND TRIM(owner) <> ''
    """,
    connection,
)

owner_candidate_source_rows = (
    owner_candidate_source_rows.merge(
        owner_token_collision_labels[
            [
                "raw_owner_label",
                "token_multiset_key",
                "candidate_structure",
            ]
        ],
        on="raw_owner_label",
        how="inner",
        validate="many_to_one",
    )
)

assert owner_candidate_source_rows[
    "source_rowid"
].is_unique

owner_same_race_variant_profile = (
    owner_candidate_source_rows.groupby(
        [
            "date",
            "course",
            "off",
            "token_multiset_key",
        ],
        as_index=False,
    )
    .agg(
        raw_label_count=(
            "raw_owner_label",
            "nunique",
        ),
        runner_rows=(
            "source_rowid",
            "size",
        ),
        raw_labels=(
            "raw_owner_label",
            lambda values: " || ".join(
                sorted(set(values))
            ),
        ),
        horses=(
            "horse",
            lambda values: " || ".join(
                sorted(set(values))
            ),
        ),
    )
)

owner_same_race_variant_evidence = (
    owner_same_race_variant_profile.loc[
        owner_same_race_variant_profile[
            "raw_label_count"
        ].gt(1)
    ]
    .copy()
    .sort_values(
        [
            "runner_rows",
            "date",
            "course",
            "off",
        ],
        ascending=[
            False,
            True,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

same_race_supported_keys = set(
    owner_same_race_variant_evidence[
        "token_multiset_key"
    ]
)

owner_token_group_evidence = (
    owner_token_group_structure.copy()
)

owner_token_group_evidence[
    "has_same_race_variant_evidence"
] = (
    owner_token_group_evidence[
        "token_multiset_key"
    ].isin(
        same_race_supported_keys
    )
)

owner_token_group_evidence[
    "evidence_status"
] = owner_token_group_evidence[
    "has_same_race_variant_evidence"
].map(
    {
        True: "same_race_source_presentation_evidence",
        False: "candidate_without_same_race_evidence",
    }
)

owner_same_race_evidence_summary = pd.DataFrame(
    [
        {
            "token_multiset_candidate_groups": len(
                owner_token_group_evidence
            ),
            "groups_with_same_race_variant_evidence": int(
                owner_token_group_evidence[
                    "has_same_race_variant_evidence"
                ].sum()
            ),
            "groups_without_same_race_variant_evidence": int(
                (
                    ~owner_token_group_evidence[
                        "has_same_race_variant_evidence"
                    ]
                ).sum()
            ),
            "same_race_evidence_instances": len(
                owner_same_race_variant_evidence
            ),
            "source_rows_in_evidence_instances": int(
                owner_same_race_variant_evidence[
                    "runner_rows"
                ].sum()
            ),
        }
    ]
)

assert len(owner_token_group_evidence) == 936

assert (
    owner_token_group_evidence[
        "groups_with_same_race_variant_evidence"
    ].sum()
    if "groups_with_same_race_variant_evidence"
    in owner_token_group_evidence.columns
    else True
)

display(owner_same_race_evidence_summary)

display(
    owner_same_race_variant_evidence[
        [
            "date",
            "course",
            "off",
            "token_multiset_key",
            "raw_label_count",
            "runner_rows",
            "raw_labels",
            "horses",
        ]
    ].head(50)
)

display(
    owner_token_group_evidence[
        [
            "token_multiset_key",
            "raw_label_count",
            "combined_runner_rows",
            "has_same_race_variant_evidence",
            "evidence_status",
        ]
    ]
    .sort_values(
        [
            "has_same_race_variant_evidence",
            "combined_runner_rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(50)
)

,token_multiset_candidate_groups,groups_with_same_race_variant_evidence,groups_without_same_race_variant_evidence,same_race_evidence_instances,source_rows_in_evidence_instances
0,936,41,895,897,2196


,date,course,off,token_multiset_key,raw_label_count,runner_rows,raw_labels,horses
0,2018-10-20,Ascot,2:40,derrick | john | magnier | michael | mrs | smi...,3,6,Derrick Smith Mrs John Magnier Michael Tabor |...,Broadway (IRE) || Bye Bye Baby (IRE) || Flatte...
1,2019-06-01,Epsom,4:30,derrick | john | magnier | michael | mrs | smi...,3,6,Derrick Smith Mrs John Magnier Michael Tabor |...,Anthony Van Dyck (IRE) || Broome (IRE) || Japa...
2,2020-06-13,Curragh (IRE),5:10,derrick | john | magnier | michael | mrs | smi...,3,6,Derrick Smith Mrs John Magnier Michael Tabor |...,Evening Primrose (IRE) || Grenadine (IRE) || K...
3,2018-10-07,Longchamp (FR),3:05,derrick | john | magnier | michael | mrs | smi...,2,5,Derrick Smith Mrs John Magnier Michael Tabor |...,Capri (IRE) || Hunting Horn (IRE) || Kew Garde...
4,2019-04-06,Leopardstown (IRE),2:45,derrick | john | magnier | michael | mrs | smi...,2,5,Michael Tabor Derrick Smith Mrs John Magnier |...,Fire Fly (IRE) || Happen (USA) || I Remember Y...
5,2019-06-29,Curragh (IRE),5:20,derrick | john | magnier | michael | mrs | smi...,3,5,Derrick Smith Mrs John Magnier Michael Tabor |...,Anthony Van Dyck (IRE) || Broome (IRE) || Il P...
6,2020-06-12,Curragh (IRE),6:40,derrick | john | magnier | michael | mrs | smi...,3,5,Derrick Smith Mrs John Magnier Michael Tabor |...,Armory (IRE) || Blue Soldier (USA) || Lope Y F...
7,2020-06-27,Curragh (IRE),7:15,derrick | john | magnier | michael | mrs | smi...,2,5,Michael Tabor Derrick Smith Mrs John Magnier |...,Arthurs Kingdom (IRE) || Dawn Patrol (IRE) || ...
8,2020-07-04,Epsom,4:55,derrick | john | magnier | michael | mrs | smi...,3,5,Derrick Smith Mrs John Magnier Michael Tabor |...,Amhran Na Bhfiann (IRE) || Mogul (GB) || Mythi...
9,2021-09-12,Curragh (IRE),4:40,derrick | john | magnier | michael | mrs | smi...,2,5,Derrick Smith Mrs John Magnier Michael Tabor |...,Amhran Na Bhfiann (IRE) || Carlisle Bay (IRE) ...


,token_multiset_key,raw_label_count,combined_runner_rows,has_same_race_variant_evidence,evidence_status
706,derrick | john | magnier | michael | mrs | smi...,6,5052,True,same_race_source_presentation_evidence
670,d | j | m | magnier | mrs | smith | tabor | we...,5,851,True,same_race_source_presentation_evidence
822,heather | michael | yarrow,2,538,True,same_race_source_presentation_evidence
891,julie | martin | phil,2,325,True,same_race_source_presentation_evidence
111,aguiar | amo | de | giselle | limited | racing,2,310,True,same_race_source_presentation_evidence
908,ltd | ollie | ownaracehorse | pears,2,245,True,same_race_source_presentation_evidence
505,burkes | syders,2,241,True,same_race_source_presentation_evidence
710,detre | detre | ecurie | jacques | patrice | s...,5,182,True,same_race_source_presentation_evidence
634,cooke | millen,2,172,True,same_race_source_presentation_evidence
631,con | mrs | okeeffe | osullivan | tadhg,2,170,True,same_race_source_presentation_evidence


### Bounded owner token-order rule

Exact token-multiset comparison generated 936 candidate groups containing owner labels with the same words in different orders.

All 936 groups represent genuine order changes rather than punctuation-only differences.

A same-race test found direct source-internal evidence for 41 groups. Within those groups, two or more differently ordered labels occur in the same reconstructed race while retaining exactly the same token multiset.

This demonstrates that, for those 41 groups, token order is source-presentation variation rather than a meaningful difference in the named ownership composition.

The rule is deliberately bounded:

1. preserve every immutable raw owner label;
2. accept only token-multiset groups with same-race variant evidence;
3. map those labels to one provisional ownership-composition identity;
4. record the method as `same_race_exact_owner_token_multiset`;
5. treat the result as composition equivalence, not verified legal identity of every named person or organisation;
6. preserve the remaining 895 token-multiset groups as unresolved candidates.

No general owner token-sorting or automatic word-order normalisation rule is authorised.

In [55]:
# Apply the bounded same-race exact-token-multiset owner rule.
#
# Only candidate groups with direct same-race evidence are accepted as
# provisional ownership-composition identities.

owner_token_group_decisions = (
    owner_token_group_evidence.copy()
    .sort_values("token_multiset_key")
    .reset_index(drop=True)
)

owner_token_group_decisions[
    "identity_relationship"
] = "unresolved"

owner_token_group_decisions[
    "decision_basis"
] = (
    "exact owner token multiset candidate without sufficient "
    "same-race source evidence"
)

owner_token_group_decisions[
    "confidence"
] = "low"

owner_token_group_decisions[
    "mapping_method"
] = ""

owner_token_group_decisions[
    "database_action"
] = "preserve_raw_unresolved"

accepted_owner_group_mask = (
    owner_token_group_decisions[
        "has_same_race_variant_evidence"
    ]
)

owner_token_group_decisions.loc[
    accepted_owner_group_mask,
    "identity_relationship",
] = "same_provisional_ownership_composition"

owner_token_group_decisions.loc[
    accepted_owner_group_mask,
    "decision_basis",
] = (
    "multiple raw owner labels with the same exact token multiset "
    "occur within the same reconstructed race"
)

owner_token_group_decisions.loc[
    accepted_owner_group_mask,
    "confidence",
] = "high_source_label_equivalence"

owner_token_group_decisions.loc[
    accepted_owner_group_mask,
    "mapping_method",
] = "same_race_exact_owner_token_multiset"

owner_token_group_decisions.loc[
    accepted_owner_group_mask,
    "database_action",
] = (
    "map_labels_to_same_provisional_ownership_composition"
)

assert len(owner_token_group_decisions) == 936

assert owner_token_group_decisions[
    "token_multiset_key"
].is_unique

assert (
    owner_token_group_decisions[
        "identity_relationship"
    ].eq(
        "same_provisional_ownership_composition"
    ).sum()
    == 41
)

assert (
    owner_token_group_decisions[
        "identity_relationship"
    ].eq("unresolved").sum()
    == 895
)

owner_token_decision_summary = pd.DataFrame(
    [
        {
            "candidate_groups": len(
                owner_token_group_decisions
            ),
            "accepted_composition_groups": int(
                owner_token_group_decisions[
                    "identity_relationship"
                ].eq(
                    "same_provisional_ownership_composition"
                ).sum()
            ),
            "preserved_unresolved_groups": int(
                owner_token_group_decisions[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "accepted_candidate_runner_rows": int(
                owner_token_group_decisions.loc[
                    accepted_owner_group_mask,
                    "combined_runner_rows",
                ].sum()
            ),
            "mapping_method": (
                "same_race_exact_owner_token_multiset"
            ),
        }
    ]
)

display(owner_token_decision_summary)

display(
    owner_token_group_decisions.loc[
        accepted_owner_group_mask,
        [
            "token_multiset_key",
            "raw_label_count",
            "combined_runner_rows",
            "identity_relationship",
            "confidence",
            "mapping_method",
            "database_action",
        ],
    ].sort_values(
        "combined_runner_rows",
        ascending=False,
    )
)

,candidate_groups,accepted_composition_groups,preserved_unresolved_groups,accepted_candidate_runner_rows,mapping_method
0,936,41,895,9788,same_race_exact_owner_token_multiset


,token_multiset_key,raw_label_count,combined_runner_rows,identity_relationship,confidence,mapping_method,database_action
706,derrick | john | magnier | michael | mrs | smi...,6,5052,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
670,d | j | m | magnier | mrs | smith | tabor | we...,5,851,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
822,heather | michael | yarrow,2,538,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
891,julie | martin | phil,2,325,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
111,aguiar | amo | de | giselle | limited | racing,2,310,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
908,ltd | ollie | ownaracehorse | pears,2,245,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
505,burkes | syders,2,241,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
710,detre | detre | ecurie | jacques | patrice | s...,5,182,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
634,cooke | millen,2,172,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...
631,con | mrs | okeeffe | osullivan | tadhg,2,170,same_provisional_ownership_composition,high_source_label_equivalence,same_race_exact_owner_token_multiset,map_labels_to_same_provisional_ownership_compo...


In [56]:
# Persist and reload the bounded owner token-order decisions.

OWNER_IDENTITY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "owner_identity"
)

OWNER_IDENTITY_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OWNER_TOKEN_DECISIONS_PATH = (
    OWNER_IDENTITY_OUTPUT_DIR
    / "owner_token_multiset_decisions.csv"
)

owner_token_group_decisions.to_csv(
    OWNER_TOKEN_DECISIONS_PATH,
    index=False,
)

owner_token_group_decisions_reloaded = pd.read_csv(
    OWNER_TOKEN_DECISIONS_PATH,
    keep_default_na=False,
)

assert len(
    owner_token_group_decisions_reloaded
) == 936

assert owner_token_group_decisions_reloaded[
    "token_multiset_key"
].is_unique

assert (
    owner_token_group_decisions_reloaded[
        "identity_relationship"
    ].eq(
        "same_provisional_ownership_composition"
    ).sum()
    == 41
)

assert (
    owner_token_group_decisions_reloaded[
        "identity_relationship"
    ].eq("unresolved").sum()
    == 895
)

assert (
    owner_token_group_decisions_reloaded.loc[
        owner_token_group_decisions_reloaded[
            "identity_relationship"
        ].eq(
            "same_provisional_ownership_composition"
        ),
        "mapping_method",
    ].eq(
        "same_race_exact_owner_token_multiset"
    ).all()
)

assert (
    owner_token_group_decisions_reloaded.loc[
        owner_token_group_decisions_reloaded[
            "identity_relationship"
        ].eq("unresolved"),
        "database_action",
    ].eq("preserve_raw_unresolved").all()
)

owner_token_decision_persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                OWNER_TOKEN_DECISIONS_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_candidate_groups": len(
                owner_token_group_decisions_reloaded
            ),
            "persisted_accepted_groups": int(
                owner_token_group_decisions_reloaded[
                    "identity_relationship"
                ].eq(
                    "same_provisional_ownership_composition"
                ).sum()
            ),
            "persisted_unresolved_groups": int(
                owner_token_group_decisions_reloaded[
                    "identity_relationship"
                ].eq("unresolved").sum()
            ),
            "reload_validation": "passed",
        }
    ]
)

display(
    owner_token_decision_persistence_summary
)

,output_path,persisted_candidate_groups,persisted_accepted_groups,persisted_unresolved_groups,reload_validation
0,data/processed/owner_identity/owner_token_mult...,936,41,895,passed


In [57]:
# Create one provisional ownership-composition identity for each accepted
# exact-token-multiset group with same-race source evidence.
#
# Unresolved candidate groups are excluded deliberately.

accepted_owner_groups = (
    owner_token_group_decisions_reloaded.loc[
        owner_token_group_decisions_reloaded[
            "identity_relationship"
        ].eq(
            "same_provisional_ownership_composition"
        )
    ]
    .copy()
    .sort_values("token_multiset_key")
    .reset_index(drop=True)
)

assert len(accepted_owner_groups) == 41

accepted_owner_labels = (
    owner_token_collision_labels.merge(
        accepted_owner_groups[
            [
                "token_multiset_key",
                "mapping_method",
                "confidence",
            ]
        ],
        on="token_multiset_key",
        how="inner",
        validate="many_to_one",
    )
    .copy()
    .sort_values(
        [
            "token_multiset_key",
            "raw_owner_label",
        ]
    )
    .reset_index(drop=True)
)

owner_group_id_lookup = {
    token_multiset_key: (
        f"OWNER-COMPOSITION-PROVISIONAL-{index:04d}"
    )
    for index, token_multiset_key in enumerate(
        accepted_owner_groups[
            "token_multiset_key"
        ],
        start=1,
    )
}

accepted_owner_labels[
    "provisional_owner_composition_id"
] = accepted_owner_labels[
    "token_multiset_key"
].map(owner_group_id_lookup)

accepted_owner_labels[
    "identity_status"
] = (
    "provisional_ownership_composition"
)

accepted_owner_labels[
    "database_action"
] = (
    "map_raw_label_to_provisional_ownership_composition"
)

owner_provisional_composition_mapping = (
    accepted_owner_labels[
        [
            "provisional_owner_composition_id",
            "raw_owner_label",
            "token_multiset_key",
            "identity_status",
            "mapping_method",
            "confidence",
            "database_action",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

assert owner_provisional_composition_mapping[
    "raw_owner_label"
].is_unique

assert (
    owner_provisional_composition_mapping[
        "provisional_owner_composition_id"
    ].nunique()
    == 41
)

assert (
    owner_provisional_composition_mapping.groupby(
        "provisional_owner_composition_id"
    )["raw_owner_label"]
    .nunique()
    .ge(2)
    .all()
)

expected_mapped_owner_labels = int(
    accepted_owner_groups[
        "raw_label_count"
    ].sum()
)

assert (
    len(owner_provisional_composition_mapping)
    == expected_mapped_owner_labels
)

OWNER_PROVISIONAL_MAPPING_PATH = (
    OWNER_IDENTITY_OUTPUT_DIR
    / "owner_provisional_composition_mapping.csv"
)

owner_provisional_composition_mapping.to_csv(
    OWNER_PROVISIONAL_MAPPING_PATH,
    index=False,
)

owner_provisional_composition_mapping_reloaded = (
    pd.read_csv(
        OWNER_PROVISIONAL_MAPPING_PATH,
        keep_default_na=False,
    )
)

assert (
    owner_provisional_composition_mapping_reloaded.equals(
        owner_provisional_composition_mapping
    )
)

owner_provisional_mapping_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                OWNER_PROVISIONAL_MAPPING_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "provisional_composition_identities": int(
                owner_provisional_composition_mapping_reloaded[
                    "provisional_owner_composition_id"
                ].nunique()
            ),
            "mapped_raw_owner_labels": len(
                owner_provisional_composition_mapping_reloaded
            ),
            "minimum_labels_per_identity": int(
                owner_provisional_composition_mapping_reloaded.groupby(
                    "provisional_owner_composition_id"
                )["raw_owner_label"]
                .nunique()
                .min()
            ),
            "maximum_labels_per_identity": int(
                owner_provisional_composition_mapping_reloaded.groupby(
                    "provisional_owner_composition_id"
                )["raw_owner_label"]
                .nunique()
                .max()
            ),
            "reload_validation": "passed",
        }
    ]
)

display(owner_provisional_mapping_summary)

display(
    owner_provisional_composition_mapping_reloaded.head(
        20
    )
)

,output_path,provisional_composition_identities,mapped_raw_owner_labels,minimum_labels_per_identity,maximum_labels_per_identity,reload_validation
0,data/processed/owner_identity/owner_provisiona...,41,95,2,6,passed


,provisional_owner_composition_id,raw_owner_label,token_multiset_key,identity_status,mapping_method,confidence,database_action
0,OWNER-COMPOSITION-PROVISIONAL-0001,B A Bracher M R Bracher E A Budlender Et Al,a | a | al | b | bracher | bracher | budlender...,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
1,OWNER-COMPOSITION-PROVISIONAL-0001,E A Budlender B A Bracher M R Bracher Et Al,a | a | al | b | bracher | bracher | budlender...,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
2,OWNER-COMPOSITION-PROVISIONAL-0002,A G Beatty P Mcbride Mrs Peter Casey Mrs Paula...,a | beatty | casey | casey | g | mcbride | mrs...,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
3,OWNER-COMPOSITION-PROVISIONAL-0002,Mrs Peter Casey P Mcbride A G Beatty Mrs Paula...,a | beatty | casey | casey | g | mcbride | mrs...,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
4,OWNER-COMPOSITION-PROVISIONAL-0003,T Lavalle A Bertuccio J White,a | bertuccio | j | lavalle | t | white,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
5,OWNER-COMPOSITION-PROVISIONAL-0003,T Lavalle J White A Bertuccio,a | bertuccio | j | lavalle | t | white,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
6,OWNER-COMPOSITION-PROVISIONAL-0004,M G Poletti Ms A M Turner,a | g | m | m | ms | poletti | turner,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
7,OWNER-COMPOSITION-PROVISIONAL-0004,Ms A M Turner M G Poletti,a | g | m | m | ms | poletti | turner,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
8,OWNER-COMPOSITION-PROVISIONAL-0005,Amo Racing Limited Giselle De Aguiar,aguiar | amo | de | giselle | limited | racing,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...
9,OWNER-COMPOSITION-PROVISIONAL-0005,Giselle De Aguiar Amo Racing Limited,aguiar | amo | de | giselle | limited | racing,provisional_ownership_composition,same_race_exact_owner_token_multiset,high_source_label_equivalence,map_raw_label_to_provisional_ownership_composi...


In [58]:
# Validate the provisional owner-composition mapping against the immutable
# source rows.
#
# This confirms source coverage and join cardinality only. It does not infer
# individual legal ownership identities inside a compressed owner label.

mapped_owner_labels = (
    owner_provisional_composition_mapping_reloaded[
        "raw_owner_label"
    ].tolist()
)

mapped_owner_placeholders = ", ".join(
    ["?"] * len(mapped_owner_labels)
)

mapped_owner_source_rows = pd.read_sql_query(
    f"""
    SELECT
        rowid AS source_rowid,
        race_id AS source_race_id,
        date,
        course,
        off,
        horse,
        owner AS raw_owner_label
    FROM data
    WHERE {DATA_ROW_PREDICATE}
      AND owner IN (
          {mapped_owner_placeholders}
      )
    """,
    connection,
    params=mapped_owner_labels,
)

assert not mapped_owner_source_rows.empty

mapped_owner_source_rows = (
    mapped_owner_source_rows.merge(
        owner_provisional_composition_mapping_reloaded[
            [
                "provisional_owner_composition_id",
                "raw_owner_label",
                "token_multiset_key",
                "mapping_method",
                "confidence",
            ]
        ],
        on="raw_owner_label",
        how="left",
        validate="many_to_one",
    )
)

assert mapped_owner_source_rows[
    "provisional_owner_composition_id"
].notna().all()

assert mapped_owner_source_rows[
    "source_rowid"
].is_unique

assert len(mapped_owner_source_rows) == 9788

assert (
    mapped_owner_source_rows[
        "provisional_owner_composition_id"
    ].nunique()
    == 41
)

assert (
    mapped_owner_source_rows[
        "raw_owner_label"
    ].nunique()
    == 95
)

owner_mapping_coverage = (
    mapped_owner_source_rows.groupby(
        [
            "provisional_owner_composition_id",
            "raw_owner_label",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=(
            "source_race_id",
            "nunique",
        ),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .sort_values(
        [
            "provisional_owner_composition_id",
            "runner_rows",
            "raw_owner_label",
        ],
        ascending=[
            True,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

assert len(owner_mapping_coverage) == 95

assert (
    owner_mapping_coverage.groupby(
        "provisional_owner_composition_id"
    )["raw_owner_label"]
    .nunique()
    .ge(2)
    .all()
)

owner_mapping_validation_summary = pd.DataFrame(
    [
        {
            "mapped_source_rows": len(
                mapped_owner_source_rows
            ),
            "mapped_raw_owner_labels": int(
                mapped_owner_source_rows[
                    "raw_owner_label"
                ].nunique()
            ),
            "provisional_composition_identities": int(
                mapped_owner_source_rows[
                    "provisional_owner_composition_id"
                ].nunique()
            ),
            "duplicate_source_rows": int(
                mapped_owner_source_rows[
                    "source_rowid"
                ].duplicated().sum()
            ),
            "unmatched_mapping_rows": int(
                mapped_owner_source_rows[
                    "provisional_owner_composition_id"
                ].isna().sum()
            ),
            "validation_status": "passed",
        }
    ]
)

display(owner_mapping_validation_summary)

display(owner_mapping_coverage.head(20))

,mapped_source_rows,mapped_raw_owner_labels,provisional_composition_identities,duplicate_source_rows,unmatched_mapping_rows,validation_status
0,9788,95,41,0,0,passed


,provisional_owner_composition_id,raw_owner_label,runner_rows,provisional_races,first_date,last_date
0,OWNER-COMPOSITION-PROVISIONAL-0001,E A Budlender B A Bracher M R Bracher Et Al,5,5,2019-02-02,2020-03-22
1,OWNER-COMPOSITION-PROVISIONAL-0001,B A Bracher M R Bracher E A Budlender Et Al,1,1,2020-03-22,2020-03-22
2,OWNER-COMPOSITION-PROVISIONAL-0002,A G Beatty P Mcbride Mrs Peter Casey Mrs Paula...,43,43,2020-11-07,2026-01-12
3,OWNER-COMPOSITION-PROVISIONAL-0002,Mrs Peter Casey P Mcbride A G Beatty Mrs Paula...,22,22,2021-04-05,2024-04-26
4,OWNER-COMPOSITION-PROVISIONAL-0003,T Lavalle J White A Bertuccio,5,5,2021-02-06,2023-11-18
5,OWNER-COMPOSITION-PROVISIONAL-0003,T Lavalle A Bertuccio J White,2,2,2023-11-04,2023-11-18
6,OWNER-COMPOSITION-PROVISIONAL-0004,M G Poletti Ms A M Turner,21,19,2015-03-05,2018-04-21
7,OWNER-COMPOSITION-PROVISIONAL-0004,Ms A M Turner M G Poletti,1,1,2015-04-02,2015-04-02
8,OWNER-COMPOSITION-PROVISIONAL-0005,Amo Racing Limited Giselle De Aguiar,302,283,2022-03-27,2026-05-26
9,OWNER-COMPOSITION-PROVISIONAL-0005,Giselle De Aguiar Amo Racing Limited,8,7,2025-02-21,2026-05-03


In [59]:
# Persist and reload the governed owner-composition source coverage.

OWNER_MAPPING_COVERAGE_PATH = (
    OWNER_IDENTITY_OUTPUT_DIR
    / "owner_provisional_composition_coverage.csv"
)

owner_mapping_coverage_to_write = (
    owner_mapping_coverage.copy()
)

for date_column in (
    "first_date",
    "last_date",
):
    owner_mapping_coverage_to_write[
        date_column
    ] = pd.to_datetime(
        owner_mapping_coverage_to_write[
            date_column
        ],
        errors="raise",
    ).dt.strftime("%Y-%m-%d")

owner_mapping_coverage_to_write.to_csv(
    OWNER_MAPPING_COVERAGE_PATH,
    index=False,
)

owner_mapping_coverage_reloaded = pd.read_csv(
    OWNER_MAPPING_COVERAGE_PATH,
    keep_default_na=False,
)

assert len(
    owner_mapping_coverage_reloaded
) == 95

assert (
    owner_mapping_coverage_reloaded[
        "provisional_owner_composition_id"
    ].nunique()
    == 41
)

assert (
    owner_mapping_coverage_reloaded[
        "runner_rows"
    ].sum()
    == 9788
)

assert (
    owner_mapping_coverage_reloaded.groupby(
        "provisional_owner_composition_id"
    )["raw_owner_label"]
    .nunique()
    .ge(2)
    .all()
)

assert owner_mapping_coverage_reloaded[
    "runner_rows"
].gt(0).all()

assert owner_mapping_coverage_reloaded[
    "provisional_races"
].gt(0).all()

owner_mapping_coverage_persistence_summary = (
    pd.DataFrame(
        [
            {
                "output_path": str(
                    OWNER_MAPPING_COVERAGE_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
                "persisted_coverage_rows": len(
                    owner_mapping_coverage_reloaded
                ),
                "provisional_composition_identities": int(
                    owner_mapping_coverage_reloaded[
                        "provisional_owner_composition_id"
                    ].nunique()
                ),
                "total_mapped_runner_rows": int(
                    owner_mapping_coverage_reloaded[
                        "runner_rows"
                    ].sum()
                ),
                "reload_validation": "passed",
            }
        ]
    )
)

display(
    owner_mapping_coverage_persistence_summary
)

,output_path,persisted_coverage_rows,provisional_composition_identities,total_mapped_runner_rows,reload_validation
0,data/processed/owner_identity/owner_provisiona...,95,41,9788,passed


In [60]:
# Persist unresolved owner token-order candidates.
#
# These groups have identical token multisets but no direct same-race variant
# evidence. They remain candidates only and must not share a provisional
# composition identity without further evidence.

owner_unresolved_token_candidates = (
    owner_token_group_decisions_reloaded.loc[
        owner_token_group_decisions_reloaded[
            "identity_relationship"
        ].eq("unresolved")
    ]
    .copy()
    .sort_values("token_multiset_key")
    .reset_index(drop=True)
)

assert len(owner_unresolved_token_candidates) == 895

assert owner_unresolved_token_candidates[
    "token_multiset_key"
].is_unique

assert owner_unresolved_token_candidates[
    "database_action"
].eq("preserve_raw_unresolved").all()

assert owner_unresolved_token_candidates[
    "mapping_method"
].eq("").all()

accepted_owner_token_keys = set(
    owner_token_group_decisions_reloaded.loc[
        owner_token_group_decisions_reloaded[
            "identity_relationship"
        ].eq(
            "same_provisional_ownership_composition"
        ),
        "token_multiset_key",
    ]
)

unresolved_owner_token_keys = set(
    owner_unresolved_token_candidates[
        "token_multiset_key"
    ]
)

assert accepted_owner_token_keys.isdisjoint(
    unresolved_owner_token_keys
)

OWNER_UNRESOLVED_TOKEN_CANDIDATES_PATH = (
    OWNER_IDENTITY_OUTPUT_DIR
    / "owner_unresolved_token_multiset_candidates.csv"
)

owner_unresolved_token_candidates.to_csv(
    OWNER_UNRESOLVED_TOKEN_CANDIDATES_PATH,
    index=False,
)

owner_unresolved_token_candidates_reloaded = (
    pd.read_csv(
        OWNER_UNRESOLVED_TOKEN_CANDIDATES_PATH,
        keep_default_na=False,
    )
)

assert len(
    owner_unresolved_token_candidates_reloaded
) == 895

assert owner_unresolved_token_candidates_reloaded[
    "token_multiset_key"
].is_unique

assert owner_unresolved_token_candidates_reloaded[
    "identity_relationship"
].eq("unresolved").all()

assert owner_unresolved_token_candidates_reloaded[
    "database_action"
].eq("preserve_raw_unresolved").all()

owner_unresolved_token_persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                OWNER_UNRESOLVED_TOKEN_CANDIDATES_PATH.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_unresolved_groups": len(
                owner_unresolved_token_candidates_reloaded
            ),
            "candidate_raw_labels": int(
                owner_unresolved_token_candidates_reloaded[
                    "raw_label_count"
                ].sum()
            ),
            "candidate_runner_rows": int(
                owner_unresolved_token_candidates_reloaded[
                    "combined_runner_rows"
                ].sum()
            ),
            "overlap_with_accepted_groups": len(
                accepted_owner_token_keys.intersection(
                    unresolved_owner_token_keys
                )
            ),
            "reload_validation": "passed",
        }
    ]
)

display(
    owner_unresolved_token_persistence_summary
)

display(
    owner_unresolved_token_candidates_reloaded[
        [
            "token_multiset_key",
            "raw_label_count",
            "combined_runner_rows",
            "decision_basis",
            "database_action",
        ]
    ]
    .sort_values(
        "combined_runner_rows",
        ascending=False,
    )
    .head(30)
)

,output_path,persisted_unresolved_groups,candidate_raw_labels,candidate_runner_rows,overlap_with_accepted_groups,reload_validation
0,data/processed/owner_identity/owner_unresolved...,895,1822,24406,0,passed


,token_multiset_key,raw_label_count,combined_runner_rows,decision_basis,database_action
294,anoj | daniel | don | macauliffe,2,696,exact owner token multiset candidate without s...,preserve_raw_unresolved
691,dineen | hughes | kerr | martin | michael,2,418,exact owner token multiset candidate without s...,preserve_raw_unresolved
855,katsumi | yoshida,2,324,exact owner token multiset candidate without s...,preserve_raw_unresolved
664,david | jones | lynne | lyons | mrs | sean | s...,3,306,exact owner token multiset candidate without s...,preserve_raw_unresolved
286,andy | bell | fergus | lyons,2,276,exact owner token multiset candidate without s...,preserve_raw_unresolved
230,alan | ann | partnership | potts,2,273,exact owner token multiset candidate without s...,preserve_raw_unresolved
238,albon | chris | mark | stedman,2,256,exact owner token multiset candidate without s...,preserve_raw_unresolved
791,hing | yue | yun,2,241,exact owner token multiset candidate without s...,preserve_raw_unresolved
140,al | bin | maktoum | maktoum | mohammed | sheikh,2,239,exact owner token multiset candidate without s...,preserve_raw_unresolved
573,chi | kin | larry | yung,2,211,exact owner token multiset candidate without s...,preserve_raw_unresolved


In [61]:
# Persist the owner-identity governance summary.

owner_identity_governance_summary = pd.DataFrame(
    [
        {
            "metric": "governed_runner_rows",
            "value": 1851285,
            "status": "confirmed",
        },
        {
            "metric": "distinct_populated_raw_owner_labels",
            "value": 98234,
            "status": "confirmed",
        },
        {
            "metric": "raw_blank_owner_rows",
            "value": 35,
            "status": "governed_by_notebook_20",
        },
        {
            "metric": "confirmed_owner_supplementations",
            "value": 22,
            "status": "inherited_from_notebook_20",
        },
        {
            "metric": "preserved_unresolved_blank_owner_rows",
            "value": 13,
            "status": "inherited_from_notebook_20",
        },
        {
            "metric": "token_multiset_candidate_groups",
            "value": 936,
            "status": "candidate_generation",
        },
        {
            "metric": "accepted_provisional_composition_groups",
            "value": 41,
            "status": "same_race_source_evidence",
        },
        {
            "metric": "accepted_raw_owner_labels",
            "value": 95,
            "status": "mapped",
        },
        {
            "metric": "accepted_mapped_runner_rows",
            "value": 9788,
            "status": "mapped",
        },
        {
            "metric": "unresolved_token_multiset_groups",
            "value": 895,
            "status": "preserve_raw_unresolved",
        },
        {
            "metric": "unresolved_candidate_raw_labels",
            "value": 1822,
            "status": "preserve_raw_unresolved",
        },
        {
            "metric": "unresolved_candidate_runner_rows",
            "value": 24406,
            "status": "preserve_raw_unresolved",
        },
    ]
)

assert owner_identity_governance_summary[
    "metric"
].is_unique

assert (
    owner_identity_governance_summary.set_index(
        "metric"
    ).loc[
        "accepted_provisional_composition_groups",
        "value",
    ]
    == 41
)

assert (
    owner_identity_governance_summary.set_index(
        "metric"
    ).loc[
        "unresolved_token_multiset_groups",
        "value",
    ]
    == 895
)

OWNER_IDENTITY_GOVERNANCE_SUMMARY_PATH = (
    OWNER_IDENTITY_OUTPUT_DIR
    / "owner_identity_governance_summary.csv"
)

owner_identity_governance_summary.to_csv(
    OWNER_IDENTITY_GOVERNANCE_SUMMARY_PATH,
    index=False,
)

owner_identity_governance_summary_reloaded = pd.read_csv(
    OWNER_IDENTITY_GOVERNANCE_SUMMARY_PATH,
    keep_default_na=False,
)

assert owner_identity_governance_summary_reloaded.equals(
    owner_identity_governance_summary
)

display(
    pd.DataFrame(
        [
            {
                "output_path": str(
                    OWNER_IDENTITY_GOVERNANCE_SUMMARY_PATH.relative_to(
                        PROJECT_ROOT
                    )
                ),
                "persisted_metrics": len(
                    owner_identity_governance_summary_reloaded
                ),
                "reload_validation": "passed",
            }
        ]
    )
)

display(owner_identity_governance_summary_reloaded)

,output_path,persisted_metrics,reload_validation
0,data/processed/owner_identity/owner_identity_g...,12,passed


,metric,value,status
0,governed_runner_rows,1851285,confirmed
1,distinct_populated_raw_owner_labels,98234,confirmed
2,raw_blank_owner_rows,35,governed_by_notebook_20
3,confirmed_owner_supplementations,22,inherited_from_notebook_20
4,preserved_unresolved_blank_owner_rows,13,inherited_from_notebook_20
5,token_multiset_candidate_groups,936,candidate_generation
6,accepted_provisional_composition_groups,41,same_race_source_evidence
7,accepted_raw_owner_labels,95,mapped
8,accepted_mapped_runner_rows,9788,mapped
9,unresolved_token_multiset_groups,895,preserve_raw_unresolved


### Owner identity conclusion and limitations

The governed source contains 1,851,285 runner rows and 98,234 distinct populated raw owner labels. A further 35 raw owner blanks were already governed through Notebook 20: 22 have evidence-backed supplementations and 13 remain unresolved.

Owner labels are structurally mixed. They can describe individuals, partnerships, syndicates, companies, studs, clubs and compressed groups of several named parties. Broad owner-name normalisation would therefore create unacceptable false equivalences.

Exact token-multiset comparison identified 936 groups in which two or more raw labels contained precisely the same tokens in different orders. These were treated as candidates only.

A bounded source-internal rule accepted 41 groups where differently ordered variants occurred within the same reconstructed race. This provides strong evidence that token order was presentation noise for those particular ownership compositions.

The accepted rule produced:

- 41 provisional ownership-composition identities;
- 95 mapped raw owner labels;
- 9,788 mapped runner rows;
- no duplicate source-row joins;
- no unmatched mapped labels.

The remaining 895 token-multiset groups, covering 1,822 raw labels and 24,406 runner rows, remain unresolved because they lacked direct same-race variant evidence.

These mappings establish high-confidence source-label equivalence for the complete named ownership composition. They do not independently verify the legal identity, ownership share or continuing membership of every person or organisation named within a compressed label.

No general token sorting, title removal, punctuation removal, partnership decomposition or owner-entity reconstruction is authorised. Immutable raw labels remain preserved, unresolved candidates remain explicit, and verified blank-field supplementations remain governed through their Notebook 20 verification records.

## Notebook conclusion

This notebook investigated whether participant labels could be converted into governed identity relationships without silently overwriting the immutable source.

### Jockeys

The source contained 7,917 distinct populated jockey labels.

Strict comparison after removing only recognised personal titles produced 212 candidate groups containing 426 labels and 216 candidate relationships.

Only one relationship was confirmed as the same jockey:

- `Mlle Marie Velon`
- `Mme Marie Velon`

One relationship was confirmed as different because both labels occurred in the same race:

- `Miss B ONeill`
- `Mr B ONeill`

The remaining 214 relationships remain unresolved. No general jockey-title removal or broad identity reconstruction is authorised.

### Trainers

The source contained 10,708 distinct populated trainer labels and nine blank trainer rows. Those blanks were already governed through Notebook 20, with four evidence-backed supplementations and five unresolved rows.

Strict title comparison produced 53 candidate groups. A bounded chronology rule accepted 26 `Mlle` to `Mme` transitions where:

- the post-title name matched exactly;
- the `Mlle` label ended between July and December 2023;
- the `Mme` label began between January and June 2024;
- the two labels had separate active periods.

Those decisions created:

- 26 provisional trainer identities;
- 52 mapped raw labels;
- 6,350 mapped runner rows;
- 27 preserved unresolved candidate groups.

The result represents high-confidence source-label equivalence, not independently verified legal or licensing identity. No general trainer-title removal is authorised.

### Owners

The source contained 98,234 distinct populated owner labels and 35 blank owner rows. Notebook 20 already governed all 35 blanks, producing 22 evidence-backed supplementations and preserving 13 unresolved rows.

Owner labels describe structurally different entities, including individuals, partnerships, syndicates, companies, studs, clubs and compressed groups of several named parties.

Exact token-multiset comparison produced 936 candidate groups whose labels contained the same words in different orders.

A bounded rule accepted only the 41 groups where differently ordered variants occurred within the same reconstructed race. These mappings created:

- 41 provisional ownership-composition identities;
- 95 mapped raw labels;
- 9,788 mapped runner rows;
- 895 preserved unresolved candidate groups covering 1,822 labels and 24,406 rows.

These mappings establish equivalence of the complete named ownership composition only. They do not verify individual legal identities, ownership shares or membership changes.

### Overall conclusion

Participant identity cannot be reconstructed safely through broad string normalisation.

The defensible database design is therefore to:

1. preserve every immutable raw participant label;
2. expose evidence-backed supplementations separately;
3. attach provisional identity identifiers only where a bounded rule has strong source-internal support;
4. retain the rule, confidence and provenance supporting each relationship;
5. preserve unresolved candidates explicitly rather than forcing a match;
6. prevent downstream analysis from treating candidate generation as proof.

The notebook has reached its analytical stopping point because each investigated participant field now has:

- a documented interpretation;
- a bounded acceptance rule;
- identified exceptions;
- persisted accepted mappings;
- persisted unresolved populations;
- source-wide coverage validation;
- explicit limitations.

Further participant identity expansion would require targeted external verification or a separately governed research programme rather than broader automatic normalisation.

## Notebook reproducibility classification

This notebook is classified as a **non-rerunnable archival construction record**.

Its purpose is to preserve the completed participant-identity investigation, including the exploratory sequence, inspected evidence, bounded decisions, unresolved populations and database consequences. It is not intended to serve as the durable production workflow.

The governed jockey, trainer and owner outputs created during the investigation have been persisted, reloaded and validated inside the notebook. Re-running the complete exploratory sequence would recreate permanent outputs and repeat candidate-generation work without improving the reliability of the final system.

Durable replacement reproducibility will therefore be provided through:

- reusable participant-identity logic under `src/inside_rails/`;
- governed persisted mapping and decision files;
- focused unit tests including failure behaviour;
- an independent source-wide validator;
- participant-identity integration documentation;
- explicit audit, report and closeout records.

The executed notebook outputs, raw-label evidence, lineage, acceptance rules and unresolved cases must remain preserved. No material cell is authorised as the production implementation merely because it appears in this notebook.